# Governança de Lead\-Time

# BAs responsáveis: @Lucas Alencar e @Denis Lobato

### Documentação: Neste link

In [1]:
# Célula 0 — Imports e configuração
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Paleta de Cores ===
COLORS = {
    "fluxo_pa":   "#6B46C1",
    "fluxo_tri":  "#EAB308",

    "status_ok":       "#16A34A",
    "status_atencao":  "#F97316",
    "status_critico":  "#DC2626",

    "neutro_escuro":   "#374151",
    "neutro_claro":    "#E5E7EB",
    "target_line":     "#9CA3AF",
}

TEMPLATE = "plotly_white"
TARGET_LEAD_TIME = 120

CONFIG = {
    "janela_meses": 12,
    "janela_meses_curta": 3,

    "min_ops_recomendacao": 5,
    "min_ops_grafico": 3,

    "percentil_recomendacao": 0.75,

    "threshold_desvio_atencao": 10,
    "threshold_desvio_critico": 30,

    "buckets_volume": [0, 200, 500, 1000, 2000, 5000, float("inf")],
    "labels_volume": ["<200", "200-499", "500-999", "1000-1999", "2000-4999", "5000+"],

    "top_n_fornecedores_grafico": 15,
    "top_n_fornecedores_heatmap": 20,
}

In [2]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_suppliers = _dntk.execute_sql(
  'SELECT \'(Todos)\' AS supplier_name\n\nUNION ALL\n\nSELECT DISTINCT supplier_name\nFROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history`\nWHERE supplier_name IS NOT NULL\n  AND dt_planned_entry_warehouse IS NOT NULL\n  AND production_order_type = \'committed\'\n  AND op_code LIKE \'OP%\'\n  AND (supplier_relationship_status IS NULL\n       OR supplier_relationship_status NOT IN (\'terminated\', \'discontinued\'))\nORDER BY supplier_name',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_suppliers

,supplier_name
0,(Todos)
1,ABBA
2,ART LIVRE
3,ARTIGO X
4,AZZURRA
5,BAE BRASIL
6,BLUTEXTIL
7,BY COTTON
8,CENTRAL DA MODELAGEM
9,CLARA BELLA


In [3]:
suppliers = df_suppliers['supplier_name'].dropna().astype(str).tolist()

In [4]:
supplier_filtro = '(Todos)'

In [5]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_products = _dntk.execute_sql(
  'SELECT \'(Todos)\' AS product_name\n\nUNION ALL\n\nSELECT DISTINCT product_name\nFROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history`\nWHERE product_name IS NOT NULL\n  AND dt_planned_entry_warehouse IS NOT NULL\n  AND production_order_type = \'committed\'\n  AND op_code LIKE \'OP%\'\n  AND (supplier_relationship_status IS NULL\n       OR supplier_relationship_status NOT IN (\'terminated\', \'discontinued\'))\n  AND (\n    {% if supplier_filtro == \'(Todos)\' %}\n      TRUE\n    {% else %}\n      supplier_name = {{ supplier_filtro }}\n    {% endif %}\n  )\nORDER BY product_name',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_products

,product_name
0,(Todos)
1,Action Top Feminino
2,Bermuda Kyoto Feminino
3,Blusa Manga Longa Comfy InLounge
4,Blusa de Alça FutureForm Feminino
...,...
195,Viseira Esportiva JoggIn
196,Wingsuit Feminino
197,Zipper Legging Feminino
198,Zipper T-Shirt NYIN Feminino


In [6]:
products = df_products['product_name'].dropna().astype(str).tolist()

In [7]:
product_filtro = '(Todos)'

In [8]:

from dateutil.parser import parse as _deepnote_parse
data_inicio = _deepnote_parse('2026-01-01T00:00:00.000Z').date()


In [9]:

from dateutil.parser import parse as _deepnote_parse
data_fim = _deepnote_parse('2026-06-30T00:00:00.000Z').date()


### Tabela de Capacidade

In [10]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_capacity = _dntk.execute_sql(
  'WITH\nfull_price AS (\n    SELECT *\n    FROM `insider-data-lake.integrated.muninn_products`\n),\nfabric_costs AS(\n    SELECT \n    mfs.id AS fabric_sku_id,\n    mfs.fabric_id,\n    mfs.knitting_factory_id,\n    mfs.sku AS fabric_sku,\n    mfs.invoice_fabric_name AS factory_fabric_name,\n    mfs.unit_price,\n    mfs.minimum_volume_per_order,\n    mfs.multiple_volume_per_order,\n    mf.name AS fabric_name,\n    mf.article_id,\n    ma.name AS article_name,\n    ma.unit AS article_unit,\n    mkf.supplier_id,\n    ms.alias AS knitting_factory_name,\n    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id\n    WHERE mfs.status IN (\'available\')\n    ),\nfabric_min_max_cost AS (\n    SELECT\n    fc.fabric_id,\n    fc.fabric_name,\n    MIN(fc.unit_price) AS min_fabric_cost,\n    MAX(fc.unit_price) AS max_fabric_cost,\n    COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,\n    ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,\n    FROM fabric_costs AS fc\n    GROUP BY fc.fabric_id,\n    fc.fabric_name\n),\narticle_sku AS (\nSELECT\n    mpsf.product_sku_id,\n    mps.sku,\n    mps.sku_name,\n    mps.product_id,\n    s.sku_state,\n    s.gender,\n    s.color,\n    s.size,\n    s.product_name,\n    mpsf.fabric_id,\n    mf.name AS fabric_name,\n    mpsf.consumption,\n    fc.min_fabric_cost AS min_fabric_unitary_cost,\n    fc.max_fabric_cost AS max_fabric_unitary_cost,\n    fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,\n    fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,\n    ma.unit AS article_unit,\n    ma.name AS article_name,\n    mf.article_id,\n    fc.number_knitting_factories,\n    fc.knitting_factories_names\nFROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\nLEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id\nLEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\nLEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\nLEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku\nLEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id\n),\nsku_fabric_costs AS (\n    SELECT \n        a_sku.sku, a_sku.sku_state, a_sku.product_id,\n        SUM(a_sku.min_fabric_cost) AS min_fabric_cost,\n        SUM(a_sku.max_fabric_cost) AS max_fabric_cost,\n        STRING_AGG(DISTINCT article_name, \',\' ORDER BY article_name) AS article_names,\n    FROM article_sku AS a_sku\n    GROUP BY a_sku.sku, a_sku.sku_state, a_sku.product_id\n),\narticle_freq AS (\n    SELECT product_id, article_names, COUNT(*) AS freq\n    FROM sku_fabric_costs GROUP BY product_id, article_names\n),\ntop_article AS (\n    SELECT product_id,\n        ARRAY_AGG(article_names ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS most_common_article_names\n    FROM article_freq GROUP BY product_id\n),\navg_fabric_cost_prod AS (\n    SELECT fc.product_id,\n        AVG(fc.max_fabric_cost) AS max_fabric_cost,\n        AVG(fc.min_fabric_cost) AS min_fabric_cost,\n        t.most_common_article_names AS article_names\n    FROM sku_fabric_costs AS fc\n    LEFT JOIN top_article AS t ON fc.product_id = t.product_id\n    GROUP BY fc.product_id, t.most_common_article_names\n),\ncosts AS (\n    SELECT\n        amp.product_id, p.product_name,\n        amp.apparel_manufacturer_id, amp.is_finished_product,\n        amp.manufacturer_cost as manufacture_cost,\n        fc.min_fabric_cost, fc.max_fabric_cost, fc.article_names,\n        CASE WHEN amp.is_finished_product = True THEN amp.manufacturer_cost\n             ELSE amp.manufacturer_cost + fc.max_fabric_cost END AS manufacturing_cost,\n    FROM `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp\n    LEFT JOIN avg_fabric_cost_prod AS fc ON fc.product_id = amp.product_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id\n    WHERE amp.status IN (\'available\',\'approved\',\'incubation\')\n),\nbase_intermediaria AS (\n    SELECT\n        ampup.apparel_manufacturer_production_unit_id,\n        am.supplier_id, amp.apparel_manufacturer_id,\n        s.alias, s.city, s.state, s.created_at AS date_supplier_creation,\n        p.product_id, p.product_name,\n        amp.is_finished_product, amp.order_minimum_volume, amp.lead_time,\n        ampu.apparel_manufacturer_cell_number,\n        MAX(ampup.weekly_maximum_productive_capacity) OVER (\n            PARTITION BY ampup.apparel_manufacturer_production_unit_id\n        ) AS max_capacity,\n        ampup.weekly_maximum_productive_capacity,\n        fp.full_price, amp.status AS status_cell,\n        4*ampup.weekly_maximum_productive_capacity AS monthly_capacity,\n        4*MAX(ampup.weekly_maximum_productive_capacity) OVER (\n            PARTITION BY ampup.apparel_manufacturer_production_unit_id\n        ) AS cell_max_monthly_capacity,\n        COUNT(DISTINCT am.supplier_id) OVER (PARTITION BY p.product_id) AS num_suppliers_per_product,\n    FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup\n    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp\n        ON amp.id = ampup.apparel_manufacturer_product_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am ON am.id = amp.apparel_manufacturer_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu\n        ON ampu.id = ampup.apparel_manufacturer_production_unit_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s ON s.id = am.supplier_id\n    LEFT JOIN full_price AS fp ON p.product_name = fp.product_name\n    WHERE amp.status IN (\'available\',\'approved\',\'incubation\')\n        AND ampup.weekly_maximum_productive_capacity > 0\n),\ncell_products AS (\n    SELECT apparel_manufacturer_production_unit_id,\n        COUNT(DISTINCT b.product_name) AS n_products_in_cell,\n        STRING_AGG(DISTINCT b.product_name, \', \') AS products_in_cell\n    FROM base_intermediaria AS b\n    GROUP BY apparel_manufacturer_production_unit_id\n),\nsku_data AS (\n    SELECT ps.sku, ps.product_sku_id, ps.sku_name, sku_d.sku_state, sku_d.product_name,\n        sku_d.family, sku_d.category, psf.fabric_id, a.name AS article_name\n    FROM `insider-data-lake.integrated.muninn_product_skus` AS ps\n    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus_fabrics` AS psf ON ps.product_sku_id = psf.product_sku_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS f ON psf.fabric_id = f.id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS a ON f.article_id = a.id\n    LEFT JOIN `insider-data-lake.integrated.skus` AS sku_d ON sku_d.sku = ps.sku\n    ORDER BY ps.product_sku_id, psf.fabric_id\n),\ndpi AS (\n    SELECT product_name, SUM(treated_generated_revenue) AS treated_generated_revenue\n    FROM `insider-data-lake.sop_silver.demand_prediction_input`\n    WHERE DATE(reference_date) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 3 MONTH)\n        AND DATE(reference_date) <  DATE_TRUNC(CURRENT_DATE(), MONTH)\n        AND product_name IS NOT NULL\n    GROUP BY product_name\n),\nrevenue_totals AS (\n    SELECT product_name, treated_generated_revenue,\n        SUM(treated_generated_revenue) OVER () AS total_treated_generated_revenue\n    FROM dpi\n),\ncum AS (\n    SELECT product_name, treated_generated_revenue, total_treated_generated_revenue,\n        SUM(treated_generated_revenue) OVER (\n            ORDER BY treated_generated_revenue DESC\n            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n        ) AS cum_treated_generated_revenue\n    FROM revenue_totals\n),\nabc_curve AS (\n    SELECT product_name, treated_generated_revenue,\n        SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) AS cum_share,\n        CASE\n            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.8 THEN \'A\'\n            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.95 THEN \'B\'\n            ELSE \'C\'\n        END AS tag_abc\n    FROM cum\n),\nfreqs AS (\n    SELECT product_name, sku_state, article_name, family, category, COUNT(*) AS freq\n    FROM sku_data GROUP BY product_name, sku_state, article_name, family, category\n),\nproduct_data AS (\n    SELECT f.product_name, p.product_id,\n        CASE WHEN SUM(CASE WHEN f.sku_state = \'ativo_perene\' THEN 1 ELSE 0 END) > 0\n             THEN \'ativo_perene\'\n             ELSE ARRAY_AGG(f.sku_state ORDER BY freq DESC LIMIT 1)[OFFSET(0)] END AS product_state,\n        ARRAY_TO_STRING(ARRAY_AGG(DISTINCT f.article_name), \', \') AS article_name,\n        ARRAY_AGG(f.family ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS family,\n        ARRAY_AGG(f.category ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS category,\n        abc.tag_abc\n    FROM freqs AS f\n    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON f.product_name = p.product_name\n    LEFT JOIN abc_curve AS abc ON abc.product_name = f.product_name\n    GROUP BY f.product_name, p.product_id, abc.tag_abc\n)\nSELECT\n    b.*,\n    c.manufacturing_cost, c.article_names,\n    SAFE_DIVIDE(b.full_price, c.manufacturing_cost) AS mark_up,\n    MIN(c.manufacturing_cost) OVER (PARTITION BY b.product_id, b.is_finished_product) AS min_manufacturing_cost,\n    cp.n_products_in_cell, cp.products_in_cell,\n    pd.tag_abc, pd.product_state,\nFROM base_intermediaria AS b\nLEFT JOIN cell_products AS cp ON b.apparel_manufacturer_production_unit_id = cp.apparel_manufacturer_production_unit_id\nLEFT JOIN costs AS c ON b.apparel_manufacturer_id = c.apparel_manufacturer_id\n    AND b.product_id = c.product_id AND b.is_finished_product = c.is_finished_product\nLEFT JOIN product_data AS pd ON b.product_id = pd.product_id\nWHERE pd.product_state NOT IN (\'desativado\')',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_capacity

,apparel_manufacturer_production_unit_id,supplier_id,apparel_manufacturer_id,alias,city,state,date_supplier_creation,product_id,product_name,is_finished_product,...,cell_max_monthly_capacity,num_suppliers_per_product,manufacturing_cost,article_names,mark_up,min_manufacturing_cost,n_products_in_cell,products_in_cell,tag_abc,product_state
0,38,7,37,BAE BRASIL,Londrina,PR,2025-02-06 13:31:01.894,226,NoHo Socks,True,...,24000,2,10.950000,Viscose,6.301370,10.950000,6,Meia Noho Socks INSIDER + Ziraldo | Expressões...,C,ativo_perene
1,132,45,61,N8,Blumenau,SC,2025-02-06 13:31:01.894,226,NoHo Socks,True,...,1000,2,13.650000,Viscose,5.054945,10.950000,1,NoHo Socks,C,ativo_perene
2,413,106,110,Italia Milano,Apucarana,PR,2025-09-03 13:48:42.142,6802,Pochete Slim Esportiva SprintIn,True,...,6000,1,51.550000,Supreme,2.890398,51.550000,1,Pochete Slim Esportiva SprintIn,C,ativo_perene
3,416,110,114,Santa Rita,Itu,SP,2025-09-12 16:21:17.950,6821,Regata Gola Quadrada IN-ACTION Seamless Feminino,True,...,1400,1,46.800000,None,4.038462,46.800000,1,Regata Gola Quadrada IN-ACTION Seamless Feminino,C,ativo_perene
4,457,124,125,RW COMERCIO DE CONFECÇÕES EIRELI,CAJAMAR,SP,2026-03-20 17:02:38.536,8859,Jaqueta Corta-Vento HikIN Masculino,True,...,20000,1,96.320000,None,3.934801,96.320000,2,"Jaqueta Corta-Vento HikIN Masculino, Jaqueta R...",C,ativo_em_lancamento
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,202,16,74,DDAL,Blumenau,SC,2025-02-06 13:31:01.894,256,Wingsuit Feminino,False,...,10000,3,111.528500,Boucle,4.474193,111.528500,1,Wingsuit Feminino,A,ativo_perene
237,415,110,114,Santa Rita,Itu,SP,2025-09-12 16:21:17.950,6820,Top Fitness IN-ACTION Seamless Feminino,True,...,1400,1,42.000000,None,3.785714,42.000000,1,Top Fitness IN-ACTION Seamless Feminino,C,ativo_perene
238,485,41,43,MAURA,São Paulo,SP,2025-02-06 13:31:01.894,9738,Calça Wide Leg com Bolso UltraBold Feminino,False,...,6400,1,113.271400,"Joplin Light,Vis UP",3.522513,113.271400,1,Calça Wide Leg com Bolso UltraBold Feminino,None,ativo_em_lancamento
239,59,17,42,FABIO,São Paulo,SP,2025-02-06 13:31:01.894,92,Tech T-shirt Gola V Masculino,False,...,10000,4,39.205168,Modal,4.310656,38.435168,10,"Tech T-shirt Gola V Masculino, Tech T-shirt Go...",A,ativo_perene


In [11]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_lead_time_cadastro_audit = _dntk.execute_sql(
  'WITH sku_status AS (\n    SELECT\n        p.product_id,\n        s.product_name,\n        STRING_AGG(DISTINCT s.sku, \', \' ORDER BY s.sku LIMIT 20) AS product_sku,\n        STRING_AGG(DISTINCT s.sku_state, \', \' ORDER BY s.sku_state LIMIT 20) AS product_status_values,\n        ARRAY_AGG(DISTINCT s.category IGNORE NULLS ORDER BY s.category LIMIT 1)[SAFE_OFFSET(0)] AS product_category,\n        CASE\n            WHEN COUNTIF(LOWER(COALESCE(s.sku_state, \'\')) IN (\n                \'ativo_perene\', \'ativo_em_lancamento\', \'ativo_capsula\',\n                \'personalizacao\', \'kit\', \'active\', \'ativo\', \'enabled\',\n                \'publicado\', \'disponivel\', \'disponível\', \'em linha\'\n            )) > 0 THEN ARRAY_AGG(\n                DISTINCT IF(\n                    LOWER(COALESCE(s.sku_state, \'\')) IN (\n                        \'ativo_perene\', \'ativo_em_lancamento\', \'ativo_capsula\',\n                        \'personalizacao\', \'kit\', \'active\', \'ativo\', \'enabled\',\n                        \'publicado\', \'disponivel\', \'disponível\', \'em linha\'\n                    ),\n                    s.sku_state,\n                    NULL\n                ) IGNORE NULLS LIMIT 1\n            )[SAFE_OFFSET(0)]\n            WHEN COUNTIF(LOWER(COALESCE(s.sku_state, \'\')) IN (\n                \'desativado\', \'inativo\', \'inactive\', \'disabled\', \'archived\',\n                \'descontinuado\', \'fora de linha\'\n            )) > 0 THEN ARRAY_AGG(\n                DISTINCT IF(\n                    LOWER(COALESCE(s.sku_state, \'\')) IN (\n                        \'desativado\', \'inativo\', \'inactive\', \'disabled\', \'archived\',\n                        \'descontinuado\', \'fora de linha\'\n                    ),\n                    s.sku_state,\n                    NULL\n                ) IGNORE NULLS LIMIT 1\n            )[SAFE_OFFSET(0)]\n            ELSE ARRAY_AGG(DISTINCT s.sku_state IGNORE NULLS ORDER BY s.sku_state LIMIT 1)[SAFE_OFFSET(0)]\n        END AS product_status_original\n    FROM `insider-data-lake.integrated.skus` AS s\n    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p\n        ON p.product_name = s.product_name\n    WHERE s.product_name IS NOT NULL\n    GROUP BY p.product_id, s.product_name\n),\nlead_time_cadastro_all AS (\n    SELECT DISTINCT\n        amp.id AS apparel_manufacturer_product_id,\n        p.product_id,\n        sku.product_sku,\n        p.product_name,\n        LOWER(TRIM(p.product_name)) AS product_name_key,\n        sku.product_category,\n        s.alias AS supplier_name,\n        s.alias AS supplier,\n        amp.is_finished_product AS is_finished_product_order,\n        CAST(amp.lead_time AS STRING) AS lead_time_cadastrado_raw,\n        SAFE_CAST(CAST(amp.lead_time AS STRING) AS FLOAT64) AS lead_time_teorico_base,\n        SAFE_CAST(CAST(amp.lead_time AS STRING) AS FLOAT64) AS lead_time_cadastrado,\n        amp.status,\n        sku.product_status_original,\n        sku.product_status_values\n    FROM `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp\n    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am\n        ON am.id = amp.apparel_manufacturer_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s\n        ON s.id = am.supplier_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p\n        ON p.product_id = amp.product_id\n    LEFT JOIN sku_status AS sku\n        ON sku.product_id = p.product_id\n        OR sku.product_name = p.product_name\n    WHERE s.alias IS NOT NULL\n      AND p.product_name IS NOT NULL\n)\nSELECT\n    apparel_manufacturer_product_id,\n    product_id,\n    product_sku,\n    product_name,\n    product_name_key,\n    product_category,\n    supplier_name,\n    supplier,\n    is_finished_product_order,\n    lead_time_teorico_base,\n    lead_time_cadastrado,\n    CASE\n        WHEN lead_time_cadastrado_raw IS NULL THEN \'nulo\'\n        WHEN TRIM(lead_time_cadastrado_raw) = \'\' THEN \'nulo\'\n        WHEN lead_time_teorico_base IS NULL THEN \'invalido\'\n        WHEN lead_time_teorico_base < 0 THEN \'negativo\'\n        WHEN lead_time_teorico_base = 0 THEN \'zerado\'\n        ELSE \'ok\'\n    END AS lead_time_status,\n    status,\n    COALESCE(product_status_original, product_status_values, status) AS product_status_original,\n    CASE\n        WHEN LOWER(COALESCE(product_status_original, \'\')) IN (\n            \'ativo_perene\', \'ativo_em_lancamento\', \'ativo_capsula\',\n            \'personalizacao\', \'kit\', \'active\', \'ativo\', \'enabled\',\n            \'publicado\', \'disponivel\', \'disponível\', \'em linha\'\n        ) THEN \'ativo\'\n        WHEN LOWER(COALESCE(product_status_original, \'\')) IN (\n            \'desativado\', \'inativo\', \'inactive\', \'disabled\', \'archived\',\n            \'descontinuado\', \'fora de linha\'\n        ) THEN \'inativo\'\n        ELSE \'status_indeterminado\'\n    END AS product_status_classificado,\n    LOWER(COALESCE(product_status_original, \'\')) IN (\n        \'ativo_perene\', \'ativo_em_lancamento\', \'ativo_capsula\',\n        \'personalizacao\', \'kit\', \'active\', \'ativo\', \'enabled\',\n        \'publicado\', \'disponivel\', \'disponível\', \'em linha\'\n    ) AS is_active_product,\n    CASE\n        WHEN lead_time_cadastrado_raw IS NULL THEN TRUE\n        WHEN TRIM(lead_time_cadastrado_raw) = \'\' THEN TRUE\n        WHEN lead_time_teorico_base IS NULL THEN TRUE\n        WHEN lead_time_teorico_base <= 0 THEN TRUE\n        ELSE FALSE\n    END AS is_invalid_lead_time,\n    (\n        LOWER(COALESCE(product_status_original, \'\')) IN (\n            \'ativo_perene\', \'ativo_em_lancamento\', \'ativo_capsula\',\n            \'personalizacao\', \'kit\', \'active\', \'ativo\', \'enabled\',\n            \'publicado\', \'disponivel\', \'disponível\', \'em linha\'\n        )\n        AND (\n            lead_time_cadastrado_raw IS NULL\n            OR TRIM(lead_time_cadastrado_raw) = \'\'\n            OR lead_time_teorico_base IS NULL\n            OR lead_time_teorico_base <= 0\n        )\n    ) AS is_critical_lead_time_issue,\n    CURRENT_TIMESTAMP() AS data_referencia\nFROM lead_time_cadastro_all',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_lead_time_cadastro_audit

,apparel_manufacturer_product_id,product_id,product_sku,product_name,product_name_key,product_category,supplier_name,supplier,is_finished_product_order,lead_time_teorico_base,lead_time_cadastrado,lead_time_status,status,product_status_original,product_status_classificado,is_active_product,is_invalid_lead_time,is_critical_lead_time_issue,data_referencia
0,53,1625,"102010040103, 102010040104, 102010040105, 1020...",Performance T-shirt Masculino,performance t-shirt masculino,T-shirt,GOAT,GOAT,False,30.0,30.0,ok,discontinued,desativado,inativo,False,False,False,2026-07-24 21:43:04.092916+00:00
1,61,70,"202010105204, 202010105205, 202010105206, 2020...",Tech T-shirt Gola V Feminino,tech t-shirt gola v feminino,T-shirt,GOAT,GOAT,False,42.0,42.0,ok,discontinued,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
2,58,317,"102010100103, 102010100104, 102010100105, 1020...",Tech T-shirt Gola U Masculino,tech t-shirt gola u masculino,T-shirt,GOAT,GOAT,False,30.0,30.0,ok,discontinued,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
3,60,92,"102010310103, 102010310104, 102010310105, 1020...",Tech T-shirt Gola V Masculino,tech t-shirt gola v masculino,T-shirt,GOAT,GOAT,False,40.0,40.0,ok,discontinued,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
4,59,260,"202010100103, 202010100104-s, 202010100105-s, ...",Tech T-shirt Gola U Feminino,tech t-shirt gola u feminino,T-shirt,GOAT,GOAT,False,35.0,35.0,ok,discontinued,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,715,62,"102010090103, 102010090104, 102010090105, 1020...",Daily T-shirt Masculino,daily t-shirt masculino,T-shirt,Casual Têxtil,Casual Têxtil,True,120.0,120.0,ok,incubation,ativo_capsula,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
732,716,247,"202010090103, 202010090104-s, 202010090105-s, ...",Daily T-shirt Feminino,daily t-shirt feminino,T-shirt,Casual Têxtil,Casual Têxtil,True,120.0,120.0,ok,incubation,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
733,721,288,"202130300104, 202130300105, 202130300106, 2021...",Techsture Vest Feminino,techsture vest feminino,Regata,FIO DE ARTE INDÚSTRIA DE CONFECCOES LTDA,FIO DE ARTE INDÚSTRIA DE CONFECCOES LTDA,False,50.0,50.0,ok,incubation,ativo_capsula,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00
734,720,256,"202130290104, 202130290105, 202130290106, 2021...",Wingsuit Feminino,wingsuit feminino,Casaco,FIO DE ARTE INDÚSTRIA DE CONFECCOES LTDA,FIO DE ARTE INDÚSTRIA DE CONFECCOES LTDA,False,50.0,50.0,ok,incubation,ativo_perene,ativo,True,False,False,2026-07-24 21:43:04.092916+00:00


In [12]:
# Celula 1.1b - Sumarizacao e base cadastral de lead time teorico
# Mantem df_capacity para capacidade vigente e cria df_lead_time_cadastro para joins de LT.

from pathlib import Path
from IPython.display import FileLink, display

ACTIVE_PRODUCT_STATUSES = {
    "ativo_perene", "ativo_em_lancamento", "ativo_capsula",
    "personalizacao", "kit", "active", "ativo", "enabled",
    "publicado", "disponivel", "disponível", "em linha",
}
INACTIVE_PRODUCT_STATUSES = {
    "desativado", "inativo", "inactive", "disabled", "archived",
    "descontinuado", "fora de linha",
}
INVALID_LEAD_TIME_STATUSES = {"zerado", "nulo", "negativo", "invalido", "ausente_pos_join"}


def _classificar_product_status(valor):
    status = str(valor).strip().lower() if pd.notna(valor) else ""
    if status in ACTIVE_PRODUCT_STATUSES:
        return "ativo"
    if status in INACTIVE_PRODUCT_STATUSES:
        return "inativo"
    return "status_indeterminado"


def _classificar_lead_time(valor):
    if pd.isna(valor):
        return "nulo"
    try:
        lead_time = float(valor)
    except (TypeError, ValueError):
        return "invalido"
    if lead_time < 0:
        return "negativo"
    if lead_time == 0:
        return "zerado"
    return "ok"

# A auditoria vem direto do cadastro fornecedor x produto, sem depender de capacidade produtiva.
df_lead_time_cadastro_audit = df_lead_time_cadastro_audit.copy()
df_lead_time_cadastro_audit["supplier_name"] = df_lead_time_cadastro_audit["supplier_name"].astype(str).str.strip()
df_lead_time_cadastro_audit["product_name"] = df_lead_time_cadastro_audit["product_name"].astype(str).str.strip()
df_lead_time_cadastro_audit["product_name_key"] = df_lead_time_cadastro_audit["product_name"].str.lower().str.strip()
df_lead_time_cadastro_audit["lead_time_teorico_base"] = pd.to_numeric(
    df_lead_time_cadastro_audit["lead_time_teorico_base"],
    errors="coerce",
)
df_lead_time_cadastro_audit["lead_time_cadastrado"] = df_lead_time_cadastro_audit["lead_time_teorico_base"]
df_lead_time_cadastro_audit["is_finished_product_order"] = df_lead_time_cadastro_audit[
    "is_finished_product_order"
].astype(bool)

if "lead_time_status" not in df_lead_time_cadastro_audit.columns:
    df_lead_time_cadastro_audit["lead_time_status"] = df_lead_time_cadastro_audit[
        "lead_time_teorico_base"
    ].apply(_classificar_lead_time)

if "product_status_classificado" not in df_lead_time_cadastro_audit.columns:
    df_lead_time_cadastro_audit["product_status_classificado"] = df_lead_time_cadastro_audit[
        "product_status_original"
    ].apply(_classificar_product_status)

# Base limpa para lookup: somente cadastro válido, uma linha por fornecedor x produto x fluxo.
_status_priority = {
    "available": 1,
    "approved": 2,
    "incubation": 3,
}

df_lead_time_cadastro = (
    df_lead_time_cadastro_audit[
        df_lead_time_cadastro_audit["supplier_name"].notna()
        & df_lead_time_cadastro_audit["product_name"].notna()
        & df_lead_time_cadastro_audit["lead_time_teorico_base"].notna()
        & (df_lead_time_cadastro_audit["lead_time_teorico_base"] > 0)
    ]
    .copy()
)
df_lead_time_cadastro["_status_priority"] = (
    df_lead_time_cadastro["status"].astype(str).str.lower().map(_status_priority).fillna(99)
)
df_lead_time_cadastro = (
    df_lead_time_cadastro
    .sort_values(
        [
            "supplier_name",
            "product_name_key",
            "is_finished_product_order",
            "_status_priority",
            "lead_time_teorico_base",
        ]
    )
    .drop_duplicates(
        subset=["supplier_name", "product_name_key", "is_finished_product_order"],
        keep="first",
    )
    .drop(columns="_status_priority")
)

invalid_lead_time_cadastro_rows = df_lead_time_cadastro_audit[
    df_lead_time_cadastro_audit["lead_time_status"].isin(INVALID_LEAD_TIME_STATUSES)
].copy()

# Compatibilidade com blocos finais antigos; nao representa limpeza de df_capacity.
invalid_capacity_rows = invalid_lead_time_cadastro_rows

summary_lt_cadastro = (
    df_lead_time_cadastro_audit
    .groupby(["product_status_classificado", "lead_time_status"], dropna=False)
    .agg(
        linhas=("product_id", "size"),
        produtos=("product_id", "nunique"),
        fornecedores=("supplier_name", "nunique"),
        criticos=("is_critical_lead_time_issue", "sum"),
    )
    .reset_index()
    .sort_values(["product_status_classificado", "lead_time_status"])
)

export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)
audit_csv_path = export_dir / "lead_time_cadastrado_auditoria.csv"
df_lead_time_cadastro_audit.to_csv(audit_csv_path, index=False, encoding="utf-8-sig")

print("Auditoria de lead time cadastrado carregada")
print(f"  Linhas auditadas: {len(df_lead_time_cadastro_audit):,}")
print(f"  Chaves validas fornecedor x produto x fluxo: {len(df_lead_time_cadastro):,}")
print(f"  Produtos com issue critica ativa: {int(df_lead_time_cadastro_audit['is_critical_lead_time_issue'].sum()):,}")
print(f"  Linhas invalidas no cadastro de LT: {len(invalid_lead_time_cadastro_rows):,}")
print(f"  df_capacity preservado para capacidade: {len(df_capacity):,} linhas")
display(summary_lt_cadastro)
display(FileLink(str(audit_csv_path), result_html_prefix="Baixar auditoria CSV: "))

Auditoria de lead time cadastrado carregada
  Linhas auditadas: 736
  Chaves validas fornecedor x produto x fluxo: 736
  Produtos com issue critica ativa: 0
  Linhas invalidas no cadastro de LT: 0
  df_capacity preservado para capacidade: 241 linhas


,product_status_classificado,lead_time_status,linhas,produtos,fornecedores,criticos
0,ativo,ok,552,146,84,0
1,inativo,ok,175,92,42,0
2,status_indeterminado,ok,9,9,3,0


/datasets/_deepnote_work/exports/lead_time_cadastrado_auditoria.csv

In [13]:
# Célula 1.5 — Validação dos filtros do Deepnote

supplier_filtro = supplier_filtro if 'supplier_filtro' in globals() else '(Todos)'
product_filtro  = product_filtro  if 'product_filtro'  in globals() else '(Todos)'
data_inicio     = data_inicio     if 'data_inicio'     in globals() else '2026-01-01'
data_fim        = data_fim        if 'data_fim'        in globals() else '2026-12-31'
data_inicio_str = str(data_inicio) if data_inicio else '2026-01-01'
data_fim_str    = str(data_fim)    if data_fim    else '2026-12-31'

# Se o fornecedor mudar, o produto previamente selecionado pode não existir
# mais na lista condicional. Nesse caso, volta para todos para evitar filtros
# combinados sem dados e erros em células abaixo.
_products_validos = set(products) if 'products' in globals() else {'(Todos)'}
if product_filtro != '(Todos)' and product_filtro not in _products_validos:
    print(
        f"Produto '{product_filtro}' não pertence ao fornecedor '{supplier_filtro}'. "
        "Filtro de produto redefinido para '(Todos)'."
    )
    product_filtro = '(Todos)'

print(
    f"Filtros ativos:"
    f"\n  fornecedor  = {supplier_filtro}"
    f"\n  produto     = {product_filtro}"
    f"\n  data_inicio = {data_inicio}"
    f"\n  data_fim    = {data_fim}"
)

Filtros ativos:
  fornecedor  = (Todos)
  produto     = (Todos)
  data_inicio = 2026-01-01
  data_fim    = 2026-06-30


### Tabela com as OPs

In [14]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_ops_raw = _dntk.execute_sql(
  'WITH CTE_OPS AS (\n  SELECT\n    h.op_code,\n    ANY_VALUE(h.current_production_stage)                   AS current_production_stage,\n    ANY_VALUE(COALESCE(h.is_finished_product_order, false)) AS is_finished_product_order,\n    STRING_AGG(DISTINCT h.product_name)                     AS product_names,\n    STRING_AGG(DISTINCT h.product_color)                    AS product_colors,\n    STRING_AGG(DISTINCT h.status_sku)                       AS sku_status,\n    ANY_VALUE(h.supplier_name)                              AS supplier_name,\n    ANY_VALUE(h.cycle_name)                                 AS cycle_name,\n    ANY_VALUE(h.production_order_type)                      AS production_order_type,\n    ANY_VALUE(h.supplier_relationship_status)               AS supplier_relationship_status,\n    MAX(h.planned_quantity_op)                              AS planned_quantity_op,\n    MAX(h.received_quantity_op)                             AS received_quantity_op,\n    MAX(h.dt_planned_production_start)                      AS dt_planned_production_start,\n    MAX(h.dt_planned_production_end)                        AS dt_planned_production_end,\n    MAX(h.dt_planned_entry_warehouse)                       AS dt_planned_entry_warehouse,\n    MAX(h.dt_reviewed_entry_warehouse)                      AS dt_reviewed_entry_warehouse,\n    MAX(h.dt_largest_entry_warehouse)                       AS dt_largest_entry_warehouse,\n    MIN(CAST(h.ingestion_date AS TIMESTAMP))                AS stamp_created_production_order,\n    MIN(CASE WHEN h.current_production_stage = \'order_request_validation\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_order_request_validation,\n    MIN(CASE WHEN h.current_production_stage = \'waiting_fabric_arrival\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_waiting_fabric_arrival,\n    MIN(CASE WHEN h.current_production_stage = \'fabric_validation_and_pre_cutting\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_fabric_validation_and_pre_cutting,\n    MIN(CASE WHEN h.current_production_stage = \'cut_fabric_and_sewing_process\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_cut_fabric_and_sewing_process,\n    MIN(CASE WHEN h.current_production_stage = \'quality_inspection\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_quality_inspection,\n    MIN(CASE WHEN h.current_production_stage = \'items_delivery_and_invoicing\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_items_delivery_and_invoicing,\n    MIN(CASE WHEN h.current_production_stage = \'finished\' THEN CAST(h.ingestion_date AS TIMESTAMP) END) AS stamp_stage_finished\n  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history` h\n  GROUP BY h.op_code\n)\nSELECT * EXCEPT (supplier_relationship_status)\nFROM CTE_OPS\nWHERE (\n    {% if supplier_filtro == \'(Todos)\' %}\n        TRUE\n    {% else %}\n        supplier_relationship_status IS NULL\n        OR supplier_relationship_status NOT IN (\'terminated\',\'discontinued\')\n    {% endif %}\n    )\n    AND (\n    {% if data_inicio_str == \'\' %}\n        TRUE\n    {% else %}\n        dt_planned_entry_warehouse >= CAST({{ data_inicio_str }} AS DATE)\n    {% endif %}\n    )\n    AND (\n    {% if data_fim_str == \'\' %}\n        TRUE\n    {% else %}\n        dt_planned_entry_warehouse < DATE_ADD(CAST({{ data_fim_str }} AS DATE), INTERVAL 1 MONTH)\n    {% endif %}\n    )\n    AND (\n    {% if supplier_filtro == \'(Todos)\' %}\n        TRUE\n    {% else %}\n        supplier_name = {{ supplier_filtro }}\n    {% endif %}\n    )\n    AND current_production_stage != \'canceled\'\n    AND NOT REGEXP_CONTAINS(LOWER(sku_status), r\'desativado\')',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_ops_raw

,op_code,current_production_stage,is_finished_product_order,product_names,product_colors,sku_status,supplier_name,cycle_name,production_order_type,planned_quantity_op,...,dt_reviewed_entry_warehouse,dt_largest_entry_warehouse,stamp_created_production_order,stamp_stage_order_request_validation,stamp_stage_waiting_fabric_arrival,stamp_stage_fabric_validation_and_pre_cutting,stamp_stage_cut_fabric_and_sewing_process,stamp_stage_quality_inspection,stamp_stage_items_delivery_and_invoicing,stamp_stage_finished
0,OPF114N10,order_request_validation,True,Regata Nadador IN-ACTION Seamless Feminino,Off White,"ativo_perene,ativo_em_lancamento",Santa Rita,LAN2209,committed,100,...,2026-03-13,2026-03-12,2025-09-25 00:00:00+00:00,2025-09-25 00:00:00+00:00,2025-12-05 00:00:00+00:00,2025-12-18 00:00:00+00:00,2026-01-22 00:00:00+00:00,2026-02-19 00:00:00+00:00,NaT,2026-03-14 00:00:00+00:00
1,OPF114N1,order_request_validation,True,Top Fitness IN-ACTION Seamless Feminino,Preto,"ativo_em_lancamento,ativo_perene",Santa Rita,LAN2209,committed,261,...,2026-02-20,2026-02-20,2025-09-27 00:00:00+00:00,2025-09-27 00:00:00+00:00,2025-12-05 00:00:00+00:00,2025-12-18 00:00:00+00:00,2026-01-07 00:00:00+00:00,2026-02-11 00:00:00+00:00,2026-02-14 00:00:00+00:00,2026-02-24 00:00:00+00:00
2,OPF33N429,fabric_delivery_and_validation,True,Spectrum Socks Mid 2.0,Branco,ativo_perene,MALHAS D'STEFANO,C012026B,committed,1112,...,2026-01-15,2026-01-13,2025-09-26 00:00:00+00:00,2025-09-26 00:00:00+00:00,2025-10-29 00:00:00+00:00,NaT,2025-12-17 00:00:00+00:00,2026-01-13 00:00:00+00:00,2026-01-19 00:00:00+00:00,2026-04-18 00:00:00+00:00
3,OPF37N1548,fabric_delivery_and_validation,True,The Perfect Top Feminino,Branco,ativo_perene,BAE BRASIL,C012026B,committed,6000,...,2026-02-04,2026-02-03,2025-09-27 00:00:00+00:00,2025-10-03 00:00:00+00:00,2025-10-28 00:00:00+00:00,2026-01-14 00:00:00+00:00,2026-01-15 00:00:00+00:00,2026-01-30 00:00:00+00:00,2026-01-31 00:00:00+00:00,2026-02-07 00:00:00+00:00
4,OPF37N1585,fabric_delivery_and_validation,True,Cueca Boxer Comfort Anti Suor Masculino,Preto,ativo_perene,BAE BRASIL,C012026B,committed,2465,...,2026-02-16,2025-12-26,2025-09-27 00:00:00+00:00,2025-10-03 00:00:00+00:00,2025-10-28 00:00:00+00:00,2025-12-02 00:00:00+00:00,2025-12-05 00:00:00+00:00,NaT,NaT,2026-01-06 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399,ORD_734_25_70_1_2_1,pending,True,Tech T-shirt Gola V Feminino,Preto,ativo_perene,ART LIVRE,C062026,committed,1691,...,2026-06-15,None,2026-02-03 00:00:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT
2400,OPF25N1618,waiting_fabric_arrival,True,Daily T-shirt Masculino,Mineral Orange,ativo_capsula,ART LIVRE,LAN2410,committed,3140,...,2026-05-26,2026-05-28,2025-10-25 00:00:00+00:00,2025-10-25 00:00:00+00:00,2025-10-29 00:00:00+00:00,2026-02-28 00:00:00+00:00,NaT,2026-05-13 00:00:00+00:00,2026-05-27 00:00:00+00:00,2026-05-29 00:00:00+00:00
2401,OPF74N215,fabric_validation_and_pre_cutting,False,Wingsuit Feminino,Gray Cloud,ativo_perene,DDAL,C022026,committed,753,...,2026-03-18,2026-03-23,2025-11-04 00:00:00+00:00,NaT,2025-11-04 00:00:00+00:00,2025-12-20 00:00:00+00:00,2026-02-02 00:00:00+00:00,2026-02-20 00:00:00+00:00,2026-03-14 00:00:00+00:00,2026-03-31 00:00:00+00:00
2402,OPF60N529,order_request_validation,True,Tube Dress Feminino,Preto,ativo_perene,DALOP,C052026,incubation,1013,...,2026-07-20,None,2026-01-20 00:00:00+00:00,2026-01-20 00:00:00+00:00,2026-03-13 00:00:00+00:00,2026-05-16 00:00:00+00:00,2026-05-27 00:00:00+00:00,2026-07-03 00:00:00+00:00,2026-07-07 00:00:00+00:00,NaT


### Tabela de Postergações

In [15]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_postponement = _dntk.execute_sql(
  'WITH ops_in_scope AS (\n  SELECT DISTINCT op_code\n  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history`\n  WHERE production_order_type = \'committed\'\n    AND (\n    {% if data_inicio_str == \'\' %}\n        TRUE\n    {% else %}\n        dt_planned_entry_warehouse >= CAST({{ data_inicio_str }} AS DATE)\n    {% endif %}\n    )\n    AND (\n    {% if data_fim_str == \'\' %}\n        TRUE\n    {% else %}\n        dt_planned_entry_warehouse < DATE_ADD(CAST({{ data_fim_str }} AS DATE), INTERVAL 1 MONTH)\n    {% endif %}\n    )\n    AND (\n    {% if supplier_filtro == \'(Todos)\' %}\n        TRUE\n    {% else %}\n        supplier_name = {{ supplier_filtro }}\n    {% endif %}\n    )\n    AND (\n    supplier_relationship_status IS NULL\n    OR supplier_relationship_status NOT IN (\'terminated\', \'discontinued\')\n    )\n),\n\ncommitted_history AS (\n  SELECT\n    h.op_code,\n    h.dt_planned_entry_warehouse,\n    h.dt_reviewed_entry_warehouse,\n    h.ingestion_date,\n    h.production_order_type\n  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history` h\n  INNER JOIN ops_in_scope s USING (op_code)\n  WHERE h.production_order_type = \'committed\'\n),\n\ndaily_dedup AS (\n  SELECT\n    op_code,\n    dt_planned_entry_warehouse,\n    dt_reviewed_entry_warehouse,\n    ingestion_date,\n    production_order_type,\n    ROW_NUMBER() OVER (\n      PARTITION BY op_code, DATE(ingestion_date)\n      ORDER BY ingestion_date ASC\n    ) AS rn\n  FROM committed_history\n),\n\ndeduped AS (\n  SELECT op_code, dt_planned_entry_warehouse, dt_reviewed_entry_warehouse, ingestion_date, production_order_type\n  FROM daily_dedup\n  WHERE rn = 1\n),\n\nwith_lag AS (\n  SELECT\n    op_code,\n    ingestion_date,\n    dt_planned_entry_warehouse,\n    dt_reviewed_entry_warehouse,\n    production_order_type,\n    LAG(dt_planned_entry_warehouse) OVER (\n      PARTITION BY op_code\n      ORDER BY ingestion_date\n    ) AS prev_planned,\n    LAG(dt_reviewed_entry_warehouse) OVER (\n      PARTITION BY op_code\n      ORDER BY ingestion_date\n    ) AS prev_reviewed,\n    LAG(production_order_type) OVER (\n      PARTITION BY op_code\n      ORDER BY ingestion_date\n    ) AS prev_production_order_type\n  FROM deduped\n),\n\noriginal_planned AS (\n  SELECT op_code, MIN(dt_planned_entry_warehouse) AS dt_planned_original\n  FROM with_lag\n  WHERE dt_reviewed_entry_warehouse IS NOT NULL\n  GROUP BY op_code\n),\n\npostponement_totals AS (\n  SELECT\n    op_code,\n    SUM(DATE_DIFF(dt_planned_entry_warehouse, prev_planned, DAY)) AS qt_dias_postergacao_intencional\n  FROM with_lag\n  WHERE production_order_type = \'committed\'\n    AND prev_production_order_type = \'committed\'\n    AND prev_planned IS NOT NULL\n    AND prev_reviewed IS NOT NULL\n    AND prev_planned = prev_reviewed\n    AND dt_planned_entry_warehouse != prev_planned\n    AND DATE_DIFF(dt_planned_entry_warehouse, prev_planned, DAY) > 0\n  GROUP BY op_code\n)\n\nSELECT\n  o.op_code,\n  o.dt_planned_original,\n  COALESCE(p.qt_dias_postergacao_intencional, 0)     AS qt_dias_postergacao_intencional,\n  COALESCE(p.qt_dias_postergacao_intencional, 0) > 0 AS flag_teve_postergacao\nFROM original_planned o\nLEFT JOIN postponement_totals p USING (op_code)',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_postponement

,op_code,dt_planned_original,qt_dias_postergacao_intencional,flag_teve_postergacao
0,0258b14a-04bb-4f0d-b458-408924b08116,2026-01-19,0,False
1,1f99ae29-579f-4d9a-9ee7-94dd792eda3b,2026-01-19,0,False
2,2eba76e3-02bb-4714-99cf-ce780803435d,2026-01-19,0,False
3,40b7417f-e017-4a29-ad95-a9a6641831ba,2026-01-19,0,False
4,618c8ef9-97dc-4f2a-9605-df201fdff586,2026-01-19,0,False
...,...,...,...,...
3362,ORD_680_37_338_37_2_1,2026-03-09,0,False
3363,ORD_680_25_126_18_3_1,2026-03-16,0,False
3364,ORD_680_118_301_1_2_1,2026-03-09,0,False
3365,ORD_680_1_215_1_4_1,2026-03-23,0,False


### Tabela de tempo de Matéria Prima\.

In [16]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_fabric_tempo = _dntk.execute_sql(
  '# Célula 2.1 — Tempo de produção da matéria-prima principal por produto (Tri)\n# Identifica o tecido principal de cada produto e busca o tempo de tingimento\n# + produção na malharia. Usado para corrigir o lead time teórico de triangulação.\n\nWITH fabric_costs AS (\n    SELECT\n        mfs.fabric_id,\n        mfs.knitting_factory_id,\n        mfs.unit_price,\n        mfs.minimum_volume_per_order,\n        mf.name AS fabric_name,\n        mf.article_id,\n        ma.name AS article_name,\n        ma.unit AS article_unit,\n        mkf.supplier_id,\n        ms.alias AS knitting_factory_name\n    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id\n),\nfabric_min_max_cost AS (\n    SELECT\n        fc.fabric_id,\n        fc.fabric_name,\n        MIN(fc.unit_price) AS min_fabric_cost,\n        MAX(fc.unit_price) AS max_fabric_cost,\n        MIN(fc.minimum_volume_per_order) AS minimum_volume_per_order,\n        COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,\n        ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names\n    FROM fabric_costs AS fc\n    GROUP BY fc.fabric_id, fc.fabric_name\n),\nskp_with_sales_l8m AS (\n    SELECT DISTINCT s.product_name\n    FROM `insider-data-lake.fpa.analytical_dre` d\n    LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)\n    WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)\n    AND d.order_status != \'Not authorized\'\n    AND d.quantity > 0\n    AND s.product_name IS NOT NULL\n),\nskp_status AS (\n    SELECT\n        s.product_name,\n        CASE\n            WHEN COUNTIF(s.sku_state = \'ativo_perene\') > 0         THEN \'ativo_perene\'\n            WHEN COUNTIF(s.sku_state = \'ativo_em_lancamento\') > 0  THEN \'ativo_em_lancamento\'\n            WHEN COUNTIF(s.sku_state = \'ativo_capsula\') > 0        THEN \'ativo_capsula\'\n            WHEN COUNTIF(s.sku_state = \'personalizacao\') > 0       THEN \'personalizacao\'\n            WHEN COUNTIF(s.sku_state = \'kit\') > 0                  THEN \'kit\'\n            ELSE \'desativado\'\n        END AS product_status\n    FROM `insider-data-lake.integrated.skus` s\n    INNER JOIN skp_with_sales_l8m l8m USING(product_name)\n    GROUP BY s.product_name\n),\nsku_fabrics AS (\n    SELECT\n        s.product_name,\n        mpsf.fabric_id,\n        mpsf.consumption,\n        fc.minimum_volume_per_order,\n        ma.unit AS article_unit,\n        ma.name AS article_name,\n        mf.article_id,\n        fc.number_knitting_factories,\n        fc.knitting_factories_names\n    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\n    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\n    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\n    LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku\n    LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id\n    INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name\n),\nproduct_article_agg AS (\n    SELECT\n        sf.product_name,\n        ss.product_status,\n        REGEXP_REPLACE(sf.article_name, r\'Modal (\\d+)\', \'Modal\') AS article_name,\n        sf.article_unit,\n        MIN(sf.minimum_volume_per_order) AS minimum_volume_per_order,\n        APPROX_QUANTILES(sf.consumption, 2)[OFFSET(1)] AS median_article_consumption\n    FROM sku_fabrics AS sf\n    INNER JOIN skp_status AS ss ON ss.product_name = sf.product_name\n    WHERE ss.product_status IN (\'ativo_perene\', \'ativo_em_lancamento\', \'desativado\')\n    AND LOWER(sf.product_name) NOT LIKE \'%ziraldo%\'\n    AND LOWER(sf.product_name) NOT LIKE \'% xp%\'\n    AND LOWER(sf.product_name) NOT LIKE \'%maluquinho%\'\n    AND LOWER(sf.product_name) NOT LIKE \'% b2b %\'\n    GROUP BY sf.product_name, ss.product_status, article_name, sf.article_unit\n),\nproduct_main_fabric AS (\n    SELECT\n        product_name,\n        product_status,\n        article_name AS tecido_principal,\n        article_unit,\n        ROUND(CAST(median_article_consumption AS FLOAT64), 4) AS consumo_mediano,\n        minimum_volume_per_order\n    FROM product_article_agg\n    QUALIFY ROW_NUMBER() OVER (\n        PARTITION BY product_name\n        ORDER BY median_article_consumption DESC\n    ) = 1\n),\nsupplier_map AS (\n    SELECT\n        art.name                        AS artigo,\n        akf.coloring_time               AS tempo_tingimento_dias,\n        akf.production_time             AS tempo_producao_dias\n    FROM `insider-lake-sensitive.prepared_br.prepared_muninn_articles_knitting_factories` akf\n    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_knitting_factories`          kf\n        ON akf.knitting_factory_id = kf.id\n    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_suppliers`                   sup\n        ON kf.supplier_id = sup.id\n    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_articles`                    art\n        ON akf.article_id = art.id\n),\narticle_final AS (\n    SELECT\n        REGEXP_REPLACE(artigo, r\'Modal (\\d+)\', \'Modal\') AS artigo,\n        MAX(tempo_tingimento_dias)                        AS tempo_tingimento,\n        MAX(tempo_producao_dias)                          AS tempo_producao_dias,\n        MAX(tempo_tingimento_dias) + MAX(tempo_producao_dias) AS tempo_total_dias\n    FROM supplier_map\n    GROUP BY artigo\n)\n\nSELECT\n    pmf.product_name,\n    pmf.product_status,\n    pmf.tecido_principal,\n    af.tempo_tingimento,\n    af.tempo_producao_dias,\n    af.tempo_total_dias\nFROM product_main_fabric pmf\nLEFT JOIN article_final af\n    ON af.artigo = pmf.tecido_principal\nORDER BY pmf.product_name',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_fabric_tempo

,product_name,product_status,tecido_principal,tempo_tingimento,tempo_producao_dias,tempo_total_dias
0,Action Top Feminino,desativado,Sportiva Pro,30.0,45.0,75.0
1,Air Blouse Feminino,desativado,String Stretch,30.0,45.0,75.0
2,Air Loop Top 2.0 Feminino,desativado,New String Stretch,30.0,45.0,75.0
3,Air Loop Top Feminino,desativado,String Stretch,30.0,45.0,75.0
4,Bermuda Kyoto Feminino,ativo_perene,Nylon WR 50+,150.0,150.0,300.0
...,...,...,...,...,...,...
161,Vestido Tube Dress Curto Feminino,desativado,Staff Special,30.0,45.0,75.0
162,Vestido Wingsuit Feminino,ativo_perene,Top Visco Comfort,30.0,30.0,60.0
163,Viseira Esportiva JoggIn,ativo_perene,Mac Power,NaN,NaN,NaN
164,Wingsuit Feminino,ativo_perene,Boucle,30.0,60.0,90.0


In [17]:
# Célula 3 — Pré-processamento: filtros, lead time realizado, etapas (6 fases) e mês de fechamento

import numpy as np


# --- 3.1 Filtros conforme metodologia do relatório original ---
df_ops = df_ops_raw.copy()

# Apenas production_order_type = 'committed'
df_ops = df_ops[df_ops["production_order_type"] == "committed"]

# Remover ciclos B2B e EPA
df_ops = df_ops[
    ~df_ops["cycle_name"].str.contains("B2B|EPA", na=False, case=False)
]

# Lead time: metodologia diferenciada por fluxo
# PA: a partir da reserva de MP (waiting_fabric_arrival), excluindo order_request_validation
# TRI: a partir da criação da OP (stamp_created_production_order), incluindo todas as etapas
df_ops["dt_largest_entry_warehouse"] = pd.to_datetime(
    df_ops["dt_largest_entry_warehouse"],
    utc=True,
)

# Cohort de chegada: exclui OPs com data de entrada futura (data planejada, não realizada)
hoje = pd.Timestamp.now(tz="UTC").normalize()
df_ops = df_ops[df_ops["dt_largest_entry_warehouse"] <= hoje]

df_ops["stamp_created_production_order"] = pd.to_datetime(
    df_ops["stamp_created_production_order"],
    utc=True,
)
df_ops["stamp_stage_waiting_fabric_arrival"] = pd.to_datetime(
    df_ops["stamp_stage_waiting_fabric_arrival"],
    utc=True,
)

inicio_lt = df_ops["stamp_stage_waiting_fabric_arrival"].where(
    df_ops["is_finished_product_order"],
    other=df_ops["stamp_created_production_order"],
)

df_ops["lead_time_realizado"] = (
    df_ops["dt_largest_entry_warehouse"] - inicio_lt
).dt.days

# Remover lead times nulos ou negativos
df_ops = df_ops[
    df_ops["lead_time_realizado"].notna()
    & (df_ops["lead_time_realizado"] > 0)
]

print(f"OPs após filtros: {len(df_ops):,}")

# Filtro de produto (pós-SQL: product_names vem de STRING_AGG, não filtrável no BQ diretamente)
if product_filtro != "(Todos)":
    df_ops = df_ops[df_ops["product_names"].str.contains(product_filtro, na=False, regex=False)]
    print(f"OPs após filtro de produto '{product_filtro}': {len(df_ops):,}")
print(f"  Triangulação: {(~df_ops['is_finished_product_order']).sum():,}")
print(f"  Produto acabado: {df_ops['is_finished_product_order'].sum():,}")


# --- 3.2 Etapas (6 fases) — mapeadas a partir dos production_stage existentes ---
stamp_cols = [
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
]

for col in stamp_cols:
    df_ops[col] = pd.to_datetime(df_ops[col], utc=True)


def _diff_days(end_col, start_col):
    """
    Dias entre dois timestamps, com clip(>=0) para não permitir negativos
    por inversão de stamps.
    """
    return (
        (df_ops[end_col] - df_ops[start_col]).dt.total_seconds() / 86400.0
    ).clip(lower=0)


df_ops["etapa_criacao_agd"] = _diff_days(
    "stamp_stage_order_request_validation",
    "stamp_created_production_order",
)
df_ops["etapa_agd_validmp"] = _diff_days(
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_order_request_validation",
)
df_ops["etapa_valid_corte"] = _diff_days(
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_waiting_fabric_arrival",
)
df_ops["etapa_valid_corte_exec"] = _diff_days(
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_fabric_validation_and_pre_cutting",
)
df_ops["etapa_costura"] = _diff_days(
    "stamp_stage_quality_inspection",
    "stamp_stage_cut_fabric_and_sewing_process",
)
df_ops["etapa_inspecao"] = _diff_days(
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_quality_inspection",
)
df_ops["etapa_fat_estoque"] = _diff_days(
    "dt_largest_entry_warehouse",
    "stamp_stage_items_delivery_and_invoicing",
)

# PA: etapa_criacao_agd não se aplica (LT começa em waiting_fabric_arrival)
df_ops.loc[df_ops["is_finished_product_order"], "etapa_criacao_agd"] = np.nan

ETAPAS_COLS = [
    "etapa_criacao_agd",
    "etapa_agd_validmp",
    "etapa_valid_corte",
    "etapa_valid_corte_exec",
    "etapa_costura",
    "etapa_inspecao",
    "etapa_fat_estoque",
]

ETAPAS_LABELS = [
    "Criação",
    "Agd da OP",
    "Aguardo MP",
    "Aguardo Corte",
    "Costura",
    "Inspeção",
    "Faturamento",
]


# --- 3.3 Mês de fechamento (para séries temporais e janelas móveis) ---
df_ops["mes_fechamento"] = (
    df_ops["dt_largest_entry_warehouse"]
    .dt.tz_convert(None)
    .dt.to_period("M")
    .dt.to_timestamp()
)


# --- 3.4 Enriquecer com lead time teórico cadastral ---
# Capacidade atual continua em df_capacity; o LT teórico vem do cadastro fornecedor x produto x fluxo.
if "df_lead_time_cadastro" not in globals():
    raise ValueError("df_lead_time_cadastro não encontrado. Execute a célula 1.1b antes da célula 3.")

df_ops["product_name_join"] = df_ops.get("product_name_join", df_ops["product_names"])
df_ops["product_name_key"] = df_ops["product_name_join"].astype(str).str.lower().str.strip()
df_ops["is_finished_product_order"] = df_ops["is_finished_product_order"].astype(bool)

cadastro_lt = (
    df_lead_time_cadastro[
        [
            "supplier_name",
            "product_name",
            "product_name_key",
            "is_finished_product_order",
            "lead_time_teorico_base",
            "status",
        ]
    ]
    .drop_duplicates(
        subset=["supplier_name", "product_name_key", "is_finished_product_order"],
        keep="first",
    )
    .rename(columns={"product_name": "product_name_cadastro", "status": "status_cadastro"})
)

cadastro_lt["is_finished_product_order"] = cadastro_lt["is_finished_product_order"].astype(bool)

df_ops = df_ops.merge(
    cadastro_lt,
    how="left",
    on=["supplier_name", "product_name_key", "is_finished_product_order"],
)

# Preserva tag_abc para leituras que dependem da curva ABC, sem usar df_capacity como fonte de LT.
if "tag_abc" in df_capacity.columns:
    capacity_tag = (
        df_capacity[["alias", "product_name", "is_finished_product", "tag_abc"]]
        .dropna(subset=["alias", "product_name"])
        .copy()
    )
    capacity_tag["supplier_name"] = capacity_tag["alias"].astype(str).str.strip()
    capacity_tag["product_name_key"] = capacity_tag["product_name"].astype(str).str.lower().str.strip()
    capacity_tag["is_finished_product_order"] = capacity_tag["is_finished_product"].astype(bool)
    capacity_tag = capacity_tag.drop_duplicates(
        subset=["supplier_name", "product_name_key", "is_finished_product_order"],
        keep="first",
    )[["supplier_name", "product_name_key", "is_finished_product_order", "tag_abc"]]

    df_ops = df_ops.merge(
        capacity_tag,
        how="left",
        on=["supplier_name", "product_name_key", "is_finished_product_order"],
    )


def _join_unique(series):
    values = [str(v) for v in series.dropna().unique()]
    return ", ".join(sorted(values)) if values else np.nan


audit_source = df_lead_time_cadastro_audit.copy()
audit_source["supplier_name"] = audit_source["supplier_name"].astype(str).str.strip()
audit_source["product_name_key"] = audit_source["product_name_key"].astype(str).str.lower().str.strip()
audit_source["is_finished_product_order"] = audit_source["is_finished_product_order"].astype(bool)
audit_source["lead_time_teorico_base"] = pd.to_numeric(
    audit_source["lead_time_teorico_base"],
    errors="coerce",
)

lead_time_audit_lookup = (
    audit_source
    .groupby(["supplier_name", "product_name_key", "is_finished_product_order"], dropna=False)
    .agg(
        product_name_cadastro=("product_name", "first"),
        lead_time_status=("lead_time_status", _join_unique),
        status_cadastro=("status", _join_unique),
        has_valid_lead_time=("lead_time_teorico_base", lambda s: bool((pd.to_numeric(s, errors="coerce") > 0).any())),
    )
    .reset_index()
)

missing_lt_base_mask = df_ops["lead_time_teorico_base"].isna()
df_lead_time_join_audit = df_ops.loc[
    missing_lt_base_mask,
    [
        "op_code",
        "supplier_name",
        "product_names",
        "product_name_join",
        "product_name_key",
        "is_finished_product_order",
        "planned_quantity_op",
        "dt_planned_entry_warehouse",
    ],
].copy()

if len(df_lead_time_join_audit) > 0:
    df_lead_time_join_audit = df_lead_time_join_audit.merge(
        lead_time_audit_lookup,
        how="left",
        on=["supplier_name", "product_name_key", "is_finished_product_order"],
    )

    def _motivo_lt(row):
        status = str(row.get("lead_time_status", ""))
        if pd.isna(row.get("lead_time_status")):
            return "sem cadastro fornecedor x produto x fluxo"
        if any(s in status for s in ["nulo", "zerado", "invalido", "negativo"]):
            return "lead time cadastrado nulo, zero ou invalido"
        if not bool(row.get("has_valid_lead_time", False)):
            return "lead time cadastrado sem valor positivo"
        return "produto com nome divergente ou fluxo divergente"

    df_lead_time_join_audit["motivo_provavel"] = df_lead_time_join_audit.apply(_motivo_lt, axis=1)
    df_lead_time_join_audit["lead_time_teorico"] = np.nan
    df_lead_time_join_audit["data_referencia"] = pd.Timestamp.now(tz="UTC")
else:
    df_lead_time_join_audit = pd.DataFrame(
        columns=[
            "op_code",
            "supplier_name",
            "product_names",
            "product_name_join",
            "product_name_key",
            "is_finished_product_order",
            "planned_quantity_op",
            "dt_planned_entry_warehouse",
            "product_name_cadastro",
            "lead_time_status",
            "status_cadastro",
            "has_valid_lead_time",
            "motivo_provavel",
            "lead_time_teorico",
            "data_referencia",
        ]
    )

if missing_lt_base_mask.any():
    print(
        "Atenção: OPs removidas das análises principais por ausência de LT teórico cadastral: "
        f"{int(missing_lt_base_mask.sum()):,}"
    )
    df_ops = df_ops[~missing_lt_base_mask].copy()


# --- 3.5 Ajuste Tri: somar tempo real de produção da malha ao lead time teórico ---
# Para triangulação, o prazo cadastrado não inclui o tempo de tingimento + produção
# da matéria-prima na malharia. df_fabric_tempo fornece esse valor por produto;
# se não houver mapeamento, aplica fallback de 60d e registra auditoria.
FALLBACK_TRI_DIAS = 60

fabric_tempo_map = (
    df_fabric_tempo[["product_name", "tempo_total_dias"]]
    .drop_duplicates(subset="product_name")
)
fabric_tempo_map["product_name_key"] = fabric_tempo_map["product_name"].astype(str).str.lower().str.strip()

df_ops = df_ops.merge(
    fabric_tempo_map[["product_name_key", "tempo_total_dias"]],
    on="product_name_key",
    how="left",
)

mask_tri = ~df_ops["is_finished_product_order"]
missing_fabric_tri_mask = mask_tri & df_ops["tempo_total_dias"].isna()

df_ops["lead_time_teorico"] = np.where(
    df_ops["is_finished_product_order"],
    df_ops["lead_time_teorico_base"],
    df_ops["lead_time_teorico_base"] + df_ops["tempo_total_dias"].fillna(FALLBACK_TRI_DIAS),
)

if missing_fabric_tri_mask.any():
    df_fabric_join_audit = df_ops.loc[
        missing_fabric_tri_mask,
        [
            "op_code",
            "supplier_name",
            "product_names",
            "product_name_join",
            "product_name_key",
            "is_finished_product_order",
            "planned_quantity_op",
            "dt_planned_entry_warehouse",
            "product_name_cadastro",
            "lead_time_teorico_base",
            "lead_time_teorico",
        ],
    ].copy()
    df_fabric_join_audit["lead_time_status"] = "ok_com_fallback_tri"
    df_fabric_join_audit["status_cadastro"] = df_fabric_join_audit.get("status_cadastro", np.nan)
    df_fabric_join_audit["has_valid_lead_time"] = True
    df_fabric_join_audit["motivo_provavel"] = "tecido nao encontrado; fallback 60d aplicado"
    df_fabric_join_audit["data_referencia"] = pd.Timestamp.now(tz="UTC")
    df_lead_time_join_audit = pd.concat(
        [df_lead_time_join_audit, df_fabric_join_audit],
        ignore_index=True,
        sort=False,
    )
else:
    df_fabric_join_audit = pd.DataFrame()

cobertura_tri = df_ops.loc[mask_tri, "tempo_total_dias"].notna().mean()
fallback_tri = int(missing_fabric_tri_mask.sum())

print(
    f"\n   Ajuste Tri — cobertura tempo_total_dias: {cobertura_tri:.1%} "
    f"| fallback {FALLBACK_TRI_DIAS}d aplicado em {fallback_tri} OPs"
)

if len(df_lead_time_join_audit) > 0:
    print("\nAuditoria de join lead time teórico:")
    display(
        df_lead_time_join_audit[
            [
                "op_code",
                "supplier_name",
                "product_names",
                "is_finished_product_order",
                "lead_time_status",
                "motivo_provavel",
            ]
        ].head(50)
    )

df_ops["desvio_lt"] = (
    df_ops["lead_time_realizado"] - df_ops["lead_time_teorico"]
)
df_ops["dentro_do_prazo"] = (
    df_ops["lead_time_realizado"] <= TARGET_LEAD_TIME
)


# --- 3.6 Buckets de volume (consome CONFIG quando definido; fallback para defaults) ---
_bins = (
    CONFIG["buckets_volume"]
    if "CONFIG" in globals()
    else [0, 200, 500, 1000, 2000, 5000, float("inf")]
)
_labels = (
    CONFIG["labels_volume"]
    if "CONFIG" in globals()
    else [
        "<200",
        "200-499",
        "500-999",
        "1000-1999",
        "2000-4999",
        "5000+",
    ]
)

df_ops["volume_bucket"] = pd.cut(
    df_ops["planned_quantity_op"],
    bins=_bins,
    labels=_labels,
    right=False,
)


# --- 3.7 Separar fluxos (re-derivado APÓS todas as colunas calculadas) ---
df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
df_pa = df_ops[df_ops["is_finished_product_order"]].copy()

print("\n✅ Pré-processamento completo.")
print(f"   df_tri (triangulação): {len(df_tri):,} OPs")
print(f"   df_pa  (produto acabado): {len(df_pa):,} OPs")
print(
    "   Cobertura lead time teórico: "
    f"{df_ops['lead_time_teorico'].notna().mean():.1%}"
)
print(
    "   Cobertura das 6 etapas (todas preenchidas): "
    f"{df_ops[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)
print(
    f"     ↳ PA:  {df_pa[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)
print(
    f"     ↳ Tri: {df_tri[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)

# --- 3.8 Enriquecer df_ops com dados de postergação intencional ---
df_ops_enriched = df_ops.merge(df_postponement, on="op_code", how="left")
df_ops_enriched["qt_dias_postergacao_intencional"] = (
    df_ops_enriched["qt_dias_postergacao_intencional"].fillna(0).astype(int)
)
df_ops_enriched["flag_teve_postergacao"] = (
    df_ops_enriched["flag_teve_postergacao"].fillna(False)
)

n_enr = df_ops_enriched["flag_teve_postergacao"].sum()
print(
    f"\n✅ df_ops_enriched: {len(df_ops_enriched):,} OPs "
    f"| {n_enr:,} com postergação ({n_enr/len(df_ops_enriched):.1%})"
)

OPs após filtros: 792
  Triangulação: 125
  Produto acabado: 667
Atenção: OPs removidas das análises principais por ausência de LT teórico cadastral: 36

   Ajuste Tri — cobertura tempo_total_dias: 100.0% | fallback 60d aplicado em 0 OPs

Auditoria de join lead time teórico:


,op_code,supplier_name,product_names,is_finished_product_order,lead_time_status,motivo_provavel
0,OPF118N47,ABBA,"Core T-shirt Masculino,Core T-Shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
1,OPF118N51,ABBA,"Core T-shirt Masculino,Core T-Shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
2,OPF92N82,LUTESTIL,"Undershirt Anti Suor Gola V Masculino,Undershi...",True,NaN,sem cadastro fornecedor x produto x fluxo
3,OPF111N17,"Ges Confecção, Comercio e Serviços de Serigra...","Core T-Shirt Masculino,Core T-shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
4,OPF111N18,"Ges Confecção, Comercio e Serviços de Serigra...","Core T-Shirt Masculino,Core T-shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
5,OPF111N38,"Ges Confecção, Comercio e Serviços de Serigra...","Core T-shirt Masculino,Core T-Shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
6,OPF22N889,GOAT,"Wingsuit Feminino,Techsture Vest Feminino",False,NaN,sem cadastro fornecedor x produto x fluxo
7,OPF42N731,FABIO,"Core T-Shirt Masculino,Core T-shirt Masculino",False,NaN,sem cadastro fornecedor x produto x fluxo
8,OPF111N14,"Ges Confecção, Comercio e Serviços de Serigra...","Core T-Shirt Masculino,Core T-shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo
9,OPF118N11,ABBA,"Core T-Shirt Masculino,Core T-shirt Masculino",True,NaN,sem cadastro fornecedor x produto x fluxo



✅ Pré-processamento completo.
   df_tri (triangulação): 120 OPs
   df_pa  (produto acabado): 636 OPs
   Cobertura lead time teórico: 100.0%
   Cobertura das 6 etapas (todas preenchidas): 7.0%
     ↳ PA:  0.0%
     ↳ Tri: 44.2%

✅ df_ops_enriched: 756 OPs | 164 com postergação (21.7%)


In [18]:
# Celula 3.0.1 - Auditoria do join de capacidade e filtro de LT teorico ausente
# Se o LT cadastrado ficou ausente apos o join, a OP sai dos KPIs principais e entra na auditoria.

missing_capacity_mask = df_ops["lead_time_teorico"].isna()

df_lead_time_join_audit = df_ops.loc[
    missing_capacity_mask,
    [
        "op_code",
        "supplier_name",
        "product_names",
        "is_finished_product_order",
        "planned_quantity_op",
        "dt_planned_entry_warehouse",
    ],
].copy()

df_lead_time_join_audit = df_lead_time_join_audit.rename(
    columns={
        "supplier_name": "supplier",
        "product_names": "product_name",
    }
)
df_lead_time_join_audit["lead_time_cadastrado"] = np.nan
df_lead_time_join_audit["lead_time_status"] = "ausente_pos_join"
df_lead_time_join_audit["product_status_classificado"] = "status_indeterminado"
df_lead_time_join_audit["is_active_product"] = False
df_lead_time_join_audit["is_invalid_lead_time"] = True
df_lead_time_join_audit["is_critical_lead_time_issue"] = False
df_lead_time_join_audit["data_referencia"] = pd.Timestamp.now(tz="UTC")

if missing_capacity_mask.any():
    n_missing_lt = int(missing_capacity_mask.sum())
    print(
        "Atencao: OPs removidas das analises principais por ausencia de LT teorico "
        f"utilizavel apos join: {n_missing_lt:,}"
    )

    df_ops = df_ops.loc[~missing_capacity_mask].copy()
    df_ops_enriched = df_ops_enriched[df_ops_enriched["lead_time_teorico"].notna()].copy()
    df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
    df_pa = df_ops[df_ops["is_finished_product_order"]].copy()
else:
    print("Join de capacidade OK: nenhuma OP sem LT teorico utilizavel.")

print(f"Linhas em df_lead_time_join_audit: {len(df_lead_time_join_audit):,}")

Join de capacidade OK: nenhuma OP sem LT teorico utilizavel.
Linhas em df_lead_time_join_audit: 0


In [19]:
# Bloco 6-MP — Decomposição de Lead Time por Matéria-prima (tecido principal)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Verificações básicas
missing = [v for v in ["df_ops", "df_mpp", "ETAPAS_COLS", "ETAPAS_LABELS", "CONFIG", "TARGET_LEAD_TIME", "TEMPLATE", "COLORS"] if v not in globals()]
if missing:
    print(f"Variáveis ausentes: {missing}. Execute as células anteriores.")
else:
    # Preparar base com tecido principal por OP (mapeia product_names -> tecido_principal)
    map_tecido = df_mpp[["product_name", "tecido_principal"]].drop_duplicates().rename(columns={"product_name": "product_names"})

    base = df_ops.merge(map_tecido, on="product_names", how="left")

    # Se não encontrou tecido, marca como 'Sem mapeamento'
    base["tecido_principal"] = base["tecido_principal"].fillna("Sem mapeamento")

    def montar_etapas_mp(df_subset, label_fluxo):
        if len(df_subset) == 0:
            return None
        agg = (
            df_subset.groupby("tecido_principal")
            .agg(**{col: (col, "median") for col in ETAPAS_COLS},
                 n_ops=("op_code", "count"),
                 lt_total=("lead_time_realizado", "median"))
            .reset_index()
        )
        # Mantém apenas grupos com volume mínimo, mas NÃO limita ao top-N
        agg = agg[agg["n_ops"] >= CONFIG["min_ops_grafico"]]
        if len(agg) == 0:
            return None
        # Soma das etapas exibidas
        agg["lt_soma_etapas"] = agg[ETAPAS_COLS].fillna(0).sum(axis=1)
        agg["fluxo"] = label_fluxo
        # Ordenar pela soma
        agg = agg.sort_values("lt_soma_etapas", ascending=True)
        return agg

    FLUXO_FILTRO_MP = "Ambos"  # "PA", "Tri" ou "Ambos"

    dfs_plot = []
    if FLUXO_FILTRO_MP in ("PA", "Ambos"):
        pa = montar_etapas_mp(base[base["is_finished_product_order"]], "PA")
        if pa is not None:
            dfs_plot.append(pa)
    if FLUXO_FILTRO_MP in ("Tri", "Ambos"):
        tri = montar_etapas_mp(base[~base["is_finished_product_order"]], "Tri")
        if tri is not None:
            dfs_plot.append(tri)

    if not dfs_plot:
        print("⚠ Sem dados suficientes para decompor por matéria-prima com os filtros atuais.")
    else:
        fig = make_subplots(
            rows=1,
            cols=len(dfs_plot),
            subplot_titles=[f"{d['fluxo'].iloc[0]} (total {len(d)} materiais)" for d in dfs_plot],
            shared_yaxes=False,
            horizontal_spacing=0.18,
        )

        ETAPAS_CORES = ["#DBEAFE", "#93C5FD", "#60A5FA", "#2563EB", "#3B82F6", "#1D4ED8", "#1E3A8A"]

        for idx, agg in enumerate(dfs_plot, start=1):
            max_total = agg["lt_soma_etapas"].max()
            for i, (col, label) in enumerate(zip(ETAPAS_COLS, ETAPAS_LABELS)):
                if agg[col].isna().all():
                    continue
                fig.add_trace(
                    go.Bar(
                        y=agg["tecido_principal"],
                        x=agg[col],
                        name=label,
                        orientation="h",
                        marker_color=ETAPAS_CORES[i],
                        text=[f"{v:.0f}d" if pd.notna(v) and v >= 10 else "" for v in agg[col]],
                        textposition="inside",
                        insidetextanchor="middle",
                        textfont=dict(size=10, color="#111827" if i < 3 else "white"),
                        hovertemplate=(f"<b>%{{y}}</b><br>{label}: %{{x:.0f}}d<extra></extra>"),
                        showlegend=(idx == 1),
                        legendgroup=label,
                    ),
                    row=1, col=idx,
                )

            # Rótulo total da barra
            for _, linha in agg.iterrows():
                total_barra = linha["lt_soma_etapas"]
                if pd.notna(total_barra):
                    fig.add_annotation(
                        x=total_barra,
                        y=linha["tecido_principal"],
                        text=f"<b>{total_barra:.0f}d</b>",
                        showarrow=False,
                        xanchor="left",
                        yanchor="middle",
                        xshift=8,
                        font=dict(size=12, color="#111827"),
                        row=1, col=idx,
                    )

            fig.add_vline(x=TARGET_LEAD_TIME, line_dash="dash", line_color=COLORS["target_line"], row=1, col=idx)

            fig.update_xaxes(
                title_text="Dias (mediana)",
                range=[0, max(max_total * 1.25, TARGET_LEAD_TIME * 1.10)],
                row=1, col=idx,
            )

        fig.update_layout(
            title="Decomposição de Lead Time por Etapa — por Matéria-prima (tecido principal)",
            barmode="stack",
            template=TEMPLATE,
            height=max(450, 30 * max(len(d) for d in dfs_plot) + 120),
            legend=dict(orientation="h", yanchor="bottom", y=-0.24, xanchor="center", x=0.5),
            margin=dict(l=40, r=100, t=80, b=130),
        )

        fig.show()

    # Também gerar uma tabela resumo para export/insumo adicional
    def tabela_resumo_mp(df_base):
        agg = (
            df_base.groupby(["tecido_principal", "is_finished_product_order"]) 
            .agg(
                n_ops=("op_code", "nunique"),
                lt_mediana=("lead_time_realizado", "median"),
                lt_p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
                lt_p90=("lead_time_realizado", lambda x: x.quantile(0.90)),
                pct_no_prazo=("dentro_do_prazo", "mean"),
            ).reset_index()
        )
        agg["fluxo"] = np.where(agg["is_finished_product_order"], "PA", "Tri")
        agg = agg.drop(columns=["is_finished_product_order"]) 
        agg["pct_no_prazo"] = (agg["pct_no_prazo"] * 100).round(1)
        num_cols = ["lt_mediana", "lt_p75", "lt_p90"]
        agg[num_cols] = agg[num_cols].round(1)
        return agg.sort_values(["fluxo", "lt_mediana"], ascending=[True, True])

    df_leadtime_por_mp = tabela_resumo_mp(base)
    df_leadtime_por_mp = df_leadtime_por_mp[df_leadtime_por_mp["n_ops"] >= CONFIG["min_ops_grafico"]]

    df_leadtime_por_mp

Variáveis ausentes: ['df_mpp']. Execute as células anteriores.


In [20]:
# Célula 3.1 — Lead Time Limpo: ajuste de postergação e propagação para todos os gráficos

# ── Passo 1: lead time bruto (original) e ajustado ───────────────────────
df_ops_enriched["lead_time_realizado_bruto"] = df_ops_enriched["lead_time_realizado"]
df_ops_enriched["lead_time_ajustado"] = (
    df_ops_enriched["lead_time_realizado_bruto"]
    - df_ops_enriched["qt_dias_postergacao_intencional"]
).clip(lower=0)

# Sobrescrever lead_time_realizado com o ajustado:
# todos os gráficos existentes passam a usar o lead time limpo automaticamente
df_ops_enriched["lead_time_realizado"] = df_ops_enriched["lead_time_ajustado"]

# Recalcular desvio e flag dentro_do_prazo com o lead time ajustado
df_ops_enriched["desvio_lt"] = (
    df_ops_enriched["lead_time_realizado"] - df_ops_enriched["lead_time_teorico"]
)
df_ops_enriched["dentro_do_prazo"] = (
    df_ops_enriched["lead_time_realizado"] <= TARGET_LEAD_TIME
)

# Propagar para df_ops, df_tri e df_pa (usados em todos os gráficos)
df_ops = df_ops_enriched.copy()
df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
df_pa  = df_ops[df_ops["is_finished_product_order"]].copy()

# ── Passo 2: mediana de referência por etapa (baseline limpa) ──────────────
# Calculada APENAS em OPs sem postergação para garantir referência não inflada
# Granularidade: supplier + produto + tipo de fluxo (PA vs Triangulação)
baseline   = df_ops_enriched[~df_ops_enriched["flag_teve_postergacao"]].copy()
group_cols = ["supplier_name", "product_names", "is_finished_product_order"]

medians_ref = (
    baseline
    .groupby(group_cols)[ETAPAS_COLS]
    .median()
    .reset_index()
    .rename(columns={c: f"med_{c}" for c in ETAPAS_COLS})
)

# ── Passo 3: identificar etapa inflada para OPs com postergação ────────────
postergadas = df_ops_enriched[df_ops_enriched["flag_teve_postergacao"]].copy()
postergadas = postergadas.merge(medians_ref, on=group_cols, how="left")

for col in ETAPAS_COLS:
    postergadas[f"dev_{col}"] = postergadas[col] - postergadas[f"med_{col}"]

dev_cols = [f"dev_{c}" for c in ETAPAS_COLS]

# Considera apenas desvios positivos: etapa que demorou MAIS que a mediana esperada
# Se nenhuma etapa teve desvio positivo (OP naturalmente rápida), etapa_inflada = None
dev_df_positivo = postergadas[dev_cols].where(postergadas[dev_cols] > 0)

postergadas["etapa_inflada"] = (
    dev_df_positivo
    .idxmax(axis=1, skipna=True)
    .where(dev_df_positivo.notna().any(axis=1))
    .str.replace("dev_", "", regex=False)
)

postergadas["duracao_etapa_inflada_original"] = postergadas.apply(
    lambda r: r[r["etapa_inflada"]] if pd.notna(r["etapa_inflada"]) else np.nan,
    axis=1,
)
postergadas["duracao_etapa_inflada_ajustada"] = (
    postergadas["duracao_etapa_inflada_original"]
    - postergadas["qt_dias_postergacao_intencional"]
).clip(lower=0)

# ── Passo 4: montar df_lead_time_clean ────────────────────────────────────
OUTPUT_COLS = [
    "op_code", "supplier_name", "product_names", "is_finished_product_order",
    "lead_time_realizado_bruto", "qt_dias_postergacao_intencional",
    "lead_time_ajustado", "flag_teve_postergacao",
]

sem_post = df_ops_enriched[~df_ops_enriched["flag_teve_postergacao"]][OUTPUT_COLS].copy()
sem_post["etapa_inflada"] = None
sem_post["duracao_etapa_inflada_original"] = np.nan
sem_post["duracao_etapa_inflada_ajustada"] = np.nan

com_post = postergadas[
    OUTPUT_COLS + [
        "etapa_inflada",
        "duracao_etapa_inflada_original",
        "duracao_etapa_inflada_ajustada",
    ]
].copy()

df_lead_time_clean = pd.concat([sem_post, com_post], ignore_index=True)
print(f"✅ df_lead_time_clean: {len(df_lead_time_clean):,} OPs")
print(f"   df_ops, df_tri e df_pa agora refletem lead time ajustado.")


✅ df_lead_time_clean: 756 OPs
   df_ops, df_tri e df_pa agora refletem lead time ajustado.
/tmp/ipykernel_363/865315500.py:56: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmax(axis=1, skipna=True)


# Criação de Relatórios

In [21]:
# Célula 3.1b — Gerador de CSVs: rankings de lead time

from pathlib import Path
from IPython.display import display, FileLink
import numpy as np


# Obs.: ao reexecutar com os mesmos filtros (fornecedor/produto/datas), os arquivos serão sobrescritos.

def _slug_filtro(valor):
    valor = str(valor or "todos").strip()
    if valor == "(Todos)":
        return "todos"
    return (
        valor.lower()
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(":", "-")
        .replace(",", "")
        .replace("(", "")
        .replace(")", "")
    )


def _to_iso_date_str(x):
    # Garante sufixos estáveis YYYY-MM-DD para datas de filtro
    if x is None:
        return ""
    try:
        import pandas as pd
        if isinstance(x, (pd.Timestamp,)):
            return x.strftime("%Y-%m-%d")
    except Exception:
        pass
    try:
        from datetime import date, datetime
        if isinstance(x, (date, datetime)):
            return x.strftime("%Y-%m-%d")
    except Exception:
        pass
    return str(x)


def gerar_csv_rankings_lead_time(df_base=None, pasta_saida="relatorios_csv"):
    """
    Gera dois CSVs respeitando os filtros já aplicados no início do Deepnote:
    1) ranking por fornecedor, com lead time médio considerando todos os produtos;
    2) ranking por produto separado por fornecedor.

    Obs.: ao reexecutar com os mesmos filtros (fornecedor/produto/datas), os arquivos serão sobrescritos.
    """
    if df_base is None:
        if "df_ops" not in globals():
            raise NameError("df_ops não encontrado. Execute as células de preparação até a Célula 3.1 antes de gerar os CSVs.")
        df_base = df_ops

    base = df_base.copy()
    base = base[
        base["supplier_name"].notna()
        & base["product_names"].notna()
        & base["lead_time_realizado"].notna()
    ].copy()

    if base.empty:
        print("Nenhuma OP disponível após os filtros atuais. CSVs não gerados.")
        return None, None

    base["fluxo"] = np.where(
        base["is_finished_product_order"],
        "Produto Acabado",
        "Triangulação",
    )

    ranking_fornecedor = (
        base
        .groupby("supplier_name", dropna=False)
        .agg(
            lead_time_medio_dias=("lead_time_realizado", "mean"),
            lead_time_mediano_dias=("lead_time_realizado", "median"),
            n_ops=("op_code", "nunique"),
            n_produtos=("product_names", "nunique"),
            pct_dentro_do_prazo=("dentro_do_prazo", "mean"),
        )
        .reset_index()
        .sort_values(["lead_time_medio_dias", "n_ops"], ascending=[True, False])
    )

    ranking_fornecedor.insert(0, "ranking", range(1, len(ranking_fornecedor) + 1))

    ranking_produto_fornecedor = (
        base
        .groupby(["supplier_name", "product_names", "fluxo"], dropna=False)
        .agg(
            lead_time_medio_dias=("lead_time_realizado", "mean"),
            lead_time_mediano_dias=("lead_time_realizado", "median"),
            n_ops=("op_code", "nunique"),
            volume_planejado_total=("planned_quantity_op", "sum"),
            pct_dentro_do_prazo=("dentro_do_prazo", "mean"),
        )
        .reset_index()
        .sort_values(
            ["product_names", "lead_time_medio_dias", "n_ops"],
            ascending=[True, True, False],
        )
    )

    ranking_produto_fornecedor["ranking_no_produto"] = (
        ranking_produto_fornecedor
        .groupby("product_names")["lead_time_medio_dias"]
        .rank(method="first", ascending=True)
        .astype(int)
    )

    ranking_produto_fornecedor = ranking_produto_fornecedor[
        [
            "product_names",
            "ranking_no_produto",
            "supplier_name",
            "fluxo",
            "lead_time_medio_dias",
            "lead_time_mediano_dias",
            "n_ops",
            "volume_planejado_total",
            "pct_dentro_do_prazo",
        ]
    ].sort_values(["product_names", "ranking_no_produto"])

    for df_csv in [ranking_fornecedor, ranking_produto_fornecedor]:
        for col in ["lead_time_medio_dias", "lead_time_mediano_dias", "pct_dentro_do_prazo"]:
            if col in df_csv.columns:
                df_csv[col] = df_csv[col].round(2)

    pasta = Path(pasta_saida)
    pasta.mkdir(parents=True, exist_ok=True)

    # Garantir datas estáveis no sufixo
    inicio_str = _to_iso_date_str(globals().get('data_inicio', ''))
    fim_str    = _to_iso_date_str(globals().get('data_fim', ''))

    filtro_suffix = (
        f"fornecedor-{_slug_filtro(globals().get('supplier_filtro', '(Todos)'))}"
        f"__produto-{_slug_filtro(globals().get('product_filtro', '(Todos)'))}"
        f"__inicio-{_slug_filtro(inicio_str)}"
        f"__fim-{_slug_filtro(fim_str)}"
    )

    arquivo_fornecedor = pasta / f"ranking_lead_time_por_fornecedor__{filtro_suffix}.csv"
    arquivo_produto = pasta / f"ranking_lead_time_por_produto_fornecedor__{filtro_suffix}.csv"

    # Mensagem clara sobre criação/sobrescrita
    existed_fornecedor = arquivo_fornecedor.exists()
    existed_produto = arquivo_produto.exists()

    if existed_fornecedor:
        print(f"Sobrescrevendo arquivo: {arquivo_fornecedor}")
    else:
        print(f"Criando arquivo: {arquivo_fornecedor}")

    if existed_produto:
        print(f"Sobrescrevendo arquivo: {arquivo_produto}")
    else:
        print(f"Criando arquivo: {arquivo_produto}")

    ranking_fornecedor.to_csv(arquivo_fornecedor, index=False, encoding="utf-8-sig")
    ranking_produto_fornecedor.to_csv(arquivo_produto, index=False, encoding="utf-8-sig")

    print("CSVs gerados com sucesso (comportamento determinístico por filtro):")
    print(f"- {arquivo_fornecedor} ({len(ranking_fornecedor):,} fornecedores)")
    print(f"- {arquivo_produto} ({len(ranking_produto_fornecedor):,} combinações produto/fornecedor)")

    # Não é necessário exibir DataFrames longos novamente; apenas os links
    display(FileLink(str(arquivo_fornecedor)))
    display(FileLink(str(arquivo_produto)))

    return ranking_fornecedor, ranking_produto_fornecedor


ranking_lead_time_fornecedor, ranking_lead_time_produto_fornecedor = gerar_csv_rankings_lead_time()

Sobrescrevendo arquivo: relatorios_csv/ranking_lead_time_por_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv
Sobrescrevendo arquivo: relatorios_csv/ranking_lead_time_por_produto_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv
CSVs gerados com sucesso (comportamento determinístico por filtro):
- relatorios_csv/ranking_lead_time_por_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv (34 fornecedores)
- relatorios_csv/ranking_lead_time_por_produto_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv (102 combinações produto/fornecedor)


/datasets/_deepnote_work/relatorios_csv/ranking_lead_time_por_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv

/datasets/_deepnote_work/relatorios_csv/ranking_lead_time_por_produto_fornecedor__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv

In [22]:
# Célula 3.1d — CSV decomposição do lead time por etapa (corrigida)

from pathlib import Path
from IPython.display import display, FileLink
import numpy as np
import pandas as pd


def gerar_csv_decomposicao_lead_time_por_etapa(df_base=None, pasta_saida="relatorios_csv"):
    # Obs.: ao reexecutar com os mesmos filtros, este arquivo será sobrescrito.
    if df_base is None:
        if "df_ops" not in globals():
            raise NameError("df_ops não encontrado. Execute as células até a Célula 3.1 antes deste bloco.")
        df_base = df_ops

    base = df_base.copy()

    etapas_cols = ETAPAS_COLS
    etapas_labels = dict(zip(ETAPAS_COLS, ETAPAS_LABELS))

    base["fluxo"] = np.where(
        base["is_finished_product_order"],
        "Produto Acabado",
        "Triangulação",
    )

    base["data"] = pd.to_datetime(
        base["dt_largest_entry_warehouse"],
        errors="coerce",
        utc=True,
    ).dt.date

    base["mes_fechamento"] = pd.to_datetime(
        base["dt_largest_entry_warehouse"],
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None).dt.to_period("M").dt.to_timestamp().dt.date

    base["total_etapas_dias"] = base[etapas_cols].sum(axis=1, min_count=1)

    base["diferenca_lt_vs_soma_etapas"] = (
        base["lead_time_realizado"] - base["total_etapas_dias"]
    )

    # idxmax com todas as colunas NaN em uma linha pode emitir FutureWarning; garantimos ao menos um valor usando fillna(-inf)
    base["etapa_gargalo_coluna"] = (
        base[etapas_cols]
        .fillna(float("-inf"))
        .idxmax(axis=1)
        .where(base[etapas_cols].notna().any(axis=1))
    )
    base["etapa_gargalo"] = base["etapa_gargalo_coluna"].map(etapas_labels)
    base["duracao_etapa_gargalo_dias"] = base.apply(
        lambda r: r[r["etapa_gargalo_coluna"]]
        if pd.notna(r["etapa_gargalo_coluna"])
        else np.nan,
        axis=1,
    )

    for col in etapas_cols:
        base[f"pct_{col}"] = np.where(
            base["total_etapas_dias"] > 0,
            base[col] / base["total_etapas_dias"],
            np.nan,
        )

    colunas_saida = [
        "op_code",
        "data",
        "mes_fechamento",
        "supplier_name",
        "product_names",
        "fluxo",
        "cycle_name",
        "planned_quantity_op",
        "received_quantity_op",
        "dt_planned_entry_warehouse",
        "dt_reviewed_entry_warehouse",
        "dt_largest_entry_warehouse",
        "lead_time_realizado_bruto",
        "qt_dias_postergacao_intencional",
        "lead_time_ajustado",
        "lead_time_realizado",
        "lead_time_teorico",
        "desvio_lt",
        "dentro_do_prazo",
        "flag_teve_postergacao",
        "total_etapas_dias",
        "diferenca_lt_vs_soma_etapas",
        "etapa_gargalo",
        "duracao_etapa_gargalo_dias",
    ]

    colunas_saida = [c for c in colunas_saida if c in base.columns]
    colunas_saida += etapas_cols
    colunas_saida += [f"pct_{c}" for c in etapas_cols]

    df_decomposicao = base[colunas_saida].copy()

    rename_etapas = {
        "etapa_criacao_agd": "dias_criacao_ate_agd",
        "etapa_agd_validmp": "dias_agd_ate_validacao_mp",
        "etapa_valid_corte": "dias_validacao_mp_ate_corte",
        "etapa_valid_corte_exec": "dias_validacao_corte_ate_execucao_corte",
        "etapa_costura": "dias_costura",
        "etapa_inspecao": "dias_inspecao",
        "etapa_fat_estoque": "dias_faturamento_ate_estoque",
        "pct_etapa_criacao_agd": "pct_criacao_ate_agd",
        "pct_etapa_agd_validmp": "pct_agd_ate_validacao_mp",
        "pct_etapa_valid_corte": "pct_validacao_mp_ate_corte",
        "pct_etapa_valid_corte_exec": "pct_validacao_corte_ate_execucao_corte",
        "pct_etapa_costura": "pct_costura",
        "pct_etapa_inspecao": "pct_inspecao",
        "pct_etapa_fat_estoque": "pct_faturamento_ate_estoque",
    }

    df_decomposicao = df_decomposicao.rename(columns=rename_etapas)

    colunas_numericas = df_decomposicao.select_dtypes(include=["number"]).columns
    df_decomposicao[colunas_numericas] = df_decomposicao[colunas_numericas].round(2)

    pasta = Path(pasta_saida)
    pasta.mkdir(parents=True, exist_ok=True)

    # Garantir datas estáveis no sufixo (usa util da célula 3.1b, se existir)
    def _to_iso_date_str_local(x):
        try:
            return _to_iso_date_str(x)
        except NameError:
            # fallback mínimo
            try:
                if isinstance(x, (pd.Timestamp,)):
                    return x.strftime("%Y-%m-%d")
            except Exception:
                pass
            try:
                from datetime import date, datetime
                if isinstance(x, (date, datetime)):
                    return x.strftime("%Y-%m-%d")
            except Exception:
                pass
            return str(x)

    inicio_str = _to_iso_date_str_local(globals().get('data_inicio', ''))
    fim_str    = _to_iso_date_str_local(globals().get('data_fim', ''))

    def _slug_local(v):
        try:
            return _slug_filtro(v)
        except NameError:
            v = str(v or "todos").strip()
            if v == "(Todos)":
                return "todos"
            return (
                v.lower()
                .replace(" ", "_")
                .replace("/", "-")
                .replace("\\", "-")
                .replace(":", "-")
                .replace(",", "")
                .replace("(", "")
                .replace(")", "")
            )

    filtro_suffix = (
        f"fornecedor-{_slug_local(globals().get('supplier_filtro', '(Todos)'))}"
        f"__produto-{_slug_local(globals().get('product_filtro', '(Todos)'))}"
        f"__inicio-{_slug_local(inicio_str)}"
        f"__fim-{_slug_local(fim_str)}"
    )

    # Correção: usar operador / para compor o caminho, não o operador bitwise OR
    arquivo = pasta / Path(f"decomposicao_lead_time_por_etapa__{filtro_suffix}.csv")

    # Mensagem clara sobre criação/sobrescrita
    existed = arquivo.exists()
    if existed:
        print(f"Sobrescrevendo arquivo: {arquivo}")
    else:
        print(f"Criando arquivo: {arquivo}")

    df_decomposicao.to_csv(arquivo, index=False, encoding="utf-8-sig")

    print("CSV completo de decomposição por etapa gerado com sucesso (comportamento determinístico por filtro):")
    print(f"- {arquivo} ({len(df_decomposicao):,} OPs)")

    display(FileLink(str(arquivo)))

    # Não é necessário exibir DataFrames longos aqui
    return df_decomposicao


# Executar a função
df_decomposicao_lead_time_etapas = gerar_csv_decomposicao_lead_time_por_etapa()

Sobrescrevendo arquivo: relatorios_csv/decomposicao_lead_time_por_etapa__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv
CSV completo de decomposição por etapa gerado com sucesso (comportamento determinístico por filtro):
- relatorios_csv/decomposicao_lead_time_por_etapa__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv (756 OPs)


/datasets/_deepnote_work/relatorios_csv/decomposicao_lead_time_por_etapa__fornecedor-todos__produto-todos__inicio-2026-01-01__fim-2026-06-30.csv

In [23]:
# Célula 3.2 — Validação: lead time limpo

total = len(df_lead_time_clean)
n_post = df_lead_time_clean["flag_teve_postergacao"].sum()
pct_post = n_post / total if total else 0

med_dias = df_lead_time_clean.loc[
    df_lead_time_clean["flag_teve_postergacao"], "qt_dias_postergacao_intencional"
].median()

baseline   = df_ops_enriched[~df_ops_enriched["flag_teve_postergacao"]].copy()
group_cols = ["supplier_name", "product_names", "is_finished_product_order"]
n_baseline = len(baseline)

SEP = "─" * 60
print(SEP)
print(f"Total de OPs processadas:          {total:>6,}")
print(f"OPs com postergação intencional:   {n_post:>6,}  ({pct_post:.1%})")
print(f"Mediana de dias postergados:       {med_dias:>6.1f}  (apenas OPs com postergação)")
print(
    f"OPs na baseline limpa p/ medianas: {n_baseline:>6,}  "
    f"({'✅ OK' if n_baseline >= 30 else '⚠️  ATENÇÃO: baseline pequena'})"
)
print(SEP)

baseline_sizes = baseline.groupby(group_cols).size().rename("n_ops").sort_values()
n_grupos_pequenos = (baseline_sizes < 5).sum()
print("\nBaseline por grupo (supplier + produto + tipo) — menores grupos:")
print(baseline_sizes.head(10).to_string())
if n_grupos_pequenos:
    print(f"\n⚠️  {n_grupos_pequenos} grupo(s) com < 5 OPs na baseline — medianas instáveis nesses combos.")
else:
    print("\n✅ Todos os grupos têm ≥ 5 OPs na baseline.")
print(SEP)

n_indeterminado = df_lead_time_clean.loc[
    df_lead_time_clean["flag_teve_postergacao"], "etapa_inflada"
].isna().sum()
print("\nDistribuição da etapa inflada (OPs com postergação):")
print(df_lead_time_clean["etapa_inflada"].value_counts(dropna=True).to_string())
if n_indeterminado:
    print(f"  (indeterminado/None): {n_indeterminado}  — nenhuma etapa teve desvio positivo")
print(SEP)

print("\nSanity check — lead time (dias):")
comp = pd.DataFrame({
    "bruto":    df_lead_time_clean["lead_time_realizado_bruto"].describe(),
    "ajustado":  df_lead_time_clean["lead_time_ajustado"].describe(),
})
print(comp.round(1).to_string())


────────────────────────────────────────────────────────────
Total de OPs processadas:             756
OPs com postergação intencional:      164  (21.7%)
Mediana de dias postergados:         24.0  (apenas OPs com postergação)
OPs na baseline limpa p/ medianas:    592  (✅ OK)
────────────────────────────────────────────────────────────

Baseline por grupo (supplier + produto + tipo) — menores grupos:
supplier_name                                           product_names                           is_finished_product_order
NATURAL COMPANY CONFECCOES LTDA                         Camiseta Polo Core Masculino            False                        1
BAE BRASIL                                              Cueca Boxer Comfort Simples Masculino   False                        1
LUTESTIL                                                The Perfect Top Feminino                True                         1
                                                        Undershirt Anti Suor Gola U Masculino 

In [24]:
# Célula 4 — Nível 1: KPI Cards (com variação MoM)

from IPython.display import display, HTML

# Valor atual (último mês fechado completo)
mes_atual = df_ops["mes_fechamento"].max()
mes_anterior = mes_atual - pd.DateOffset(months=1)


def mediana_mes(df, mes):
    sub = df[df["mes_fechamento"] == mes]
    return sub["lead_time_realizado"].median() if len(sub) >= 10 else None


lt_geral_atual = df_ops["lead_time_realizado"].median()
lt_tri_atual = df_tri["lead_time_realizado"].median()
lt_pa_atual = df_pa["lead_time_realizado"].median()
pct_dentro = df_ops["dentro_do_prazo"].mean() * 100

# Novo indicador: % dentro do prazo de 90 dias
pct_dentro_90 = (df_ops["lead_time_realizado"] <= 90).mean() * 100

# Variação MoM
lt_geral_ant = mediana_mes(df_ops, mes_anterior)
lt_tri_ant = mediana_mes(df_tri, mes_anterior)
lt_pa_ant = mediana_mes(df_pa, mes_anterior)


def delta_mom(atual, anterior):
    if anterior is None or pd.isna(anterior) or pd.isna(atual):
        return ""

    delta = atual - anterior
    arrow = "▼" if delta < 0 else ("▲" if delta > 0 else "→")

    color = (
        COLORS["status_ok"]
        if delta < 0
        else (
            COLORS["status_critico"]
            if delta > 0
            else COLORS["neutro_escuro"]
        )
    )

    return f"<span style='color:{color}'>{arrow} {abs(delta):.0f}d MoM</span>"


def kpi_card(label, value, unit="", color="#374151", sub=None):
    sub_html = (
        f"<div style='font-size:12px;color:#666;margin-top:6px'>{sub}</div>"
        if sub
        else ""
    )

    return f"""
    <div style='display:inline-block;background:#fafafa;border:1px solid #e5e7eb;
                border-radius:10px;padding:18px 26px;margin:8px;min-width:170px;text-align:center'>
        <div style='font-size:12px;color:#6b7280;font-weight:600;text-transform:uppercase;letter-spacing:0.5px'>{label}</div>
        <div style='font-size:34px;font-weight:700;color:{color};margin-top:4px'>{value}<span style='font-size:14px;font-weight:500'>{unit}</span></div>
        {sub_html}
    </div>
    """


# Cor de status para %120d
if pct_dentro >= 65:
    cor_pct = COLORS["status_ok"]
elif pct_dentro >= 50:
    cor_pct = COLORS["status_atencao"]
else:
    cor_pct = COLORS["status_critico"]

# Cor de status para %90d
if pct_dentro_90 >= 65:
    cor_pct_90 = COLORS["status_ok"]
elif pct_dentro_90 >= 50:
    cor_pct_90 = COLORS["status_atencao"]
else:
    cor_pct_90 = COLORS["status_critico"]

# Rótulo do cohort: jan/2026 → mês atual (data planejada de entrega)
cohort_label = f"Cohort: jan/2026–{mes_atual.strftime('%b/%Y').lower()}"

cards_html = "".join([
    kpi_card(
        "Mediana Geral",
        f"{lt_geral_atual:.0f}",
        "d",
        COLORS["neutro_escuro"],
        f"Target: {TARGET_LEAD_TIME}d &nbsp;|&nbsp; {delta_mom(lt_geral_atual, lt_geral_ant)}",
    ),
    kpi_card(
        "Mediana Triangulação",
        f"{lt_tri_atual:.0f}",
        "d",
        COLORS["fluxo_tri"],
        delta_mom(lt_tri_atual, lt_tri_ant),
    ),
    kpi_card(
        "Mediana Produto Acabado",
        f"{lt_pa_atual:.0f}",
        "d",
        COLORS["fluxo_pa"],
        delta_mom(lt_pa_atual, lt_pa_ant),
    ),
    kpi_card(
        "Dentro do Prazo (120d)",
        f"{pct_dentro:.1f}",
        "%",
        cor_pct,
        sub=None,
    ),
    kpi_card(
        "Dentro do Prazo (90d)",
        f"{pct_dentro_90:.1f}",
        "%",
        cor_pct_90,
        sub=None,
    ),
])

display(HTML(f"""
<div style='font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif'>

    <div style='margin:4px 8px 18px 8px'>
        <div style='font-size:22px;font-weight:800;color:{COLORS.get("neutro_escuro", "#374151")};letter-spacing:-0.3px'>
            📊 Visão Executiva
        </div>
    </div>

    <div style='display:flex;flex-wrap:wrap;gap:6px'>
        {cards_html}
    </div>

</div>
"""))

In [25]:
# Célula 4.1b.1 — KPI por Etapa | Triangulação
# Decomposição proporcional às frações medianas por OP no período filtrado atual.

from IPython.display import display, HTML
import numpy as np
import pandas as pd

# Verificação de pré-requisitos
if 'df_ops' not in globals() or 'ETAPAS_COLS' not in globals() or 'ETAPAS_LABELS' not in globals():
    print("Pré-requisitos não encontrados (df_ops / ETAPAS_COLS / ETAPAS_LABELS). Execute as células de preparação (Células 3, 3.1 e 3.2) e tente novamente.")

else:
    # Fallback local para kpi_card, caso não exista
    try:
        _ = kpi_card

    except NameError:
        def kpi_card(label, value, unit="", color="#374151", sub=None):
            sub_html = f"<div style='font-size:12px;color:#666;margin-top:6px'>{sub}</div>" if sub else ""

            return f"""
            <div style='display:inline-block;background:#fafafa;border:1px solid #E5E7EB;border-radius:10px;padding:18px 18px;margin:8px;min-width:160px;text-align:center'>
                <div style='font-size:11px;color:#6b7280;font-weight:700;text-transform:uppercase;letter-spacing:0.5px'>{label}</div>
                <div style='font-size:30px;font-weight:700;color:{color};margin-top:4px'>{value}<span style='font-size:14px;font-weight:500'>{unit}</span></div>
                {sub_html}
            </div>
            """

    # Nome da etapa a ser removida do card: 'etapa_criacao_agd'
    ETAPA_REMOVER = 'etapa_criacao_agd'

    def calcular_pesos_fluxo(df_fluxo, is_pa):
        soma_k = df_fluxo[ETAPAS_COLS].sum(axis=1, min_count=1)
        valid_mask = soma_k.notna() & (soma_k > 0)
        sub_valid = df_fluxo.loc[valid_mask].copy()

        if len(sub_valid) == 0:
            return None, None, True

        soma_k = soma_k.loc[valid_mask]

        fracs = {}

        for col in ETAPAS_COLS:
            vals = sub_valid[col]
            frac_col = (vals / soma_k).where(vals.notna() & (soma_k > 0))
            fracs[col] = frac_col

        fracs_df = pd.DataFrame(fracs)

        pesos_raw = fracs_df.median(axis=0, skipna=True)

        # Não atribui peso à etapa de criação→agd, pois o card será removido
        if ETAPA_REMOVER in pesos_raw.index:
            pesos_raw.loc[ETAPA_REMOVER] = 0.0

        positivos = pesos_raw[pesos_raw > 0].dropna()
        fallback_flag = False

        if positivos.empty or positivos.sum() <= 0:
            etapas_validas = [
                c for c in ETAPAS_COLS
                if c != ETAPA_REMOVER
            ]

            if len(etapas_validas) == 0:
                return None, None, True

            w = pd.Series(0.0, index=ETAPAS_COLS)
            w.loc[etapas_validas] = 1.0 / len(etapas_validas)
            fallback_flag = True

            return w, sub_valid, fallback_flag

        w = pesos_raw.copy()
        w = w.clip(lower=0).fillna(0.0)
        s = w.sum()

        if s > 0:
            w = w / s
        else:
            etapas_validas = [
                c for c in ETAPAS_COLS
                if c != ETAPA_REMOVER
            ]

            w = pd.Series(0.0, index=ETAPAS_COLS)

            if len(etapas_validas) > 0:
                w.loc[etapas_validas] = 1.0 / len(etapas_validas)

            fallback_flag = True

        return w, sub_valid, fallback_flag

    def decompor_T(T, w):
        if T is None or pd.isna(T):
            return {c: None for c in ETAPAS_COLS}

        contrib = {
            c: float(w.get(c, 0.0)) * float(T)
            for c in ETAPAS_COLS
        }

        arred = {
            c: int(round(v))
            for c, v in contrib.items()
        }

        arred = {
            c: max(0, v)
            for c, v in arred.items()
        }

        soma_arred = sum(v for v in arred.values() if v is not None)
        diff = int(round(float(T) - soma_arred))

        if diff != 0:
            # distribui o ajuste na maior fração entre as etapas exibidas (exclui a removida)
            exibiveis = [c for c in ETAPAS_COLS if c != ETAPA_REMOVER]
            maior = max(exibiveis, key=lambda c: w.get(c, 0.0)) if exibiveis else None
            if maior is not None:
                arred[maior] = max(0, arred.get(maior, 0) + diff)

        return arred

    def kpis_fluxo(df_fluxo, titulo_fluxo, cor_titulo="#374151"):
        is_pa = df_fluxo["is_finished_product_order"].iloc[0] if len(df_fluxo) else True

        T = (
            float(df_fluxo["lead_time_realizado"].median())
            if df_fluxo["lead_time_realizado"].notna().sum() > 0
            else None
        )

        soma_cols = df_fluxo[ETAPAS_COLS].sum(axis=1, min_count=1)
        mediana_soma = (
            float(soma_cols.median())
            if soma_cols.notna().sum() > 0
            else None
        )

        if T is not None:
            w, sub_valid, fallback_flag = calcular_pesos_fluxo(df_fluxo, is_pa=is_pa)

            if w is None:
                etapas_validas = [
                    c for c in ETAPAS_COLS
                    if c != ETAPA_REMOVER
                ]

                w = pd.Series(0.0, index=ETAPAS_COLS)

                if len(etapas_validas) > 0:
                    w.loc[etapas_validas] = 1.0 / len(etapas_validas)

                fallback_flag = True

            Ei = decompor_T(T, w)

        else:
            w = pd.Series({c: None for c in ETAPAS_COLS})
            Ei = {c: None for c in ETAPAS_COLS}
            fallback_flag = False

        total_str = "—" if (T is None or pd.isna(T)) else f"{T:.0f}"

        resumo_cards = [
            kpi_card(
                "Total LDT (mediana)",
                total_str,
                unit="d",
                color=COLORS.get("neutro_escuro", "#374151"),
                sub=None,
            )
        ]

        cards_html = []

        # Itera sobre as etapas, mas pula a etapa de criação→agd
        for col, label in zip(ETAPAS_COLS, ETAPAS_LABELS):
            if col == ETAPA_REMOVER:
                continue

            v = Ei.get(col)

            if v is None or pd.isna(v):
                valor_str = "—"
                sub_txt = None

            else:
                valor_str = f"{int(v)}"
                sub_txt = None

            cards_html.append(
                kpi_card(
                    label,
                    valor_str,
                    unit="d",
                    color=COLORS.get("neutro_escuro", "#374151"),
                    sub=sub_txt,
                )
            )

        microtexto = "Decomposição proporcional às frações medianas por OP — soma fecha com o Total LDT (base: período filtrado atual)"

        if 'fallback_flag' in locals() and fallback_flag:
            microtexto += " — fallback uniforme"

        bloco_html = f"""
        <div style='margin:8px 0 18px 0'>
          <div style='font-size:16px;font-weight:700;color:{cor_titulo};margin:4px 8px 2px 8px'>{titulo_fluxo}</div>
          <div style='font-size:11px;color:#6b7280;margin:0 8px 10px 8px'>{microtexto}</div>

          <div style='display:flex;flex-wrap:wrap;gap:6px;margin:0 8px 10px 8px'>
            {''.join(resumo_cards)}
          </div>

          <div style='display:flex;flex-wrap:wrap;gap:6px'>
            {''.join(cards_html)}
          </div>
        </div>
        """

        return bloco_html

    # Base Triangulação
    df_tri_local = df_ops[df_ops["is_finished_product_order"] == False].copy()

    html_tri = kpis_fluxo(
        df_tri_local,
        "Triangulação",
        cor_titulo=COLORS.get("fluxo_tri", "#EAB308")
    )

    container_html = f"""
    <div style='font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif'>

      <div style='margin:4px 8px 18px 8px'>
        <div style='font-size:22px;font-weight:800;color:{COLORS.get("neutro_escuro", "#374151")};letter-spacing:-0.3px'>
          📊 Visão Executiva - Quebrada por etapas | Triangulação
        </div>
      </div>

      {html_tri}

    </div>
    """

    display(HTML(container_html))

In [26]:
# Célula 4.1b.2 — KPI por Etapa | Produto Acabado
# Reaproveita as funções definidas no bloco anterior.

from IPython.display import display, HTML
import numpy as np
import pandas as pd

# Verificação de pré-requisitos
if (
    'df_ops' not in globals()
    or 'ETAPAS_COLS' not in globals()
    or 'ETAPAS_LABELS' not in globals()
    or 'kpis_fluxo' not in globals()
):
    print("Pré-requisitos não encontrados. Execute primeiro as células de preparação e o bloco 4.1b.1 — Triangulação.")

else:
    # Base Produto Acabado
    df_pa_local = df_ops[df_ops["is_finished_product_order"] == True].copy()

    # Remover a etapa 'Criação→agd' do gráfico (card) na chamada para Produto Acabado.
    # A função kpis_fluxo definida no bloco 4.1b.1 já ignora a etapa
    # 'etapa_criacao_agd' ao montar os cards, portanto apenas chamamos normalmente.
    html_pa = kpis_fluxo(
        df_pa_local,
        "Produto Acabado",
        cor_titulo=COLORS.get("fluxo_pa", "#6B46C1")
    )

    container_html = f"""
    <div style='font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif'>

      <div style='margin:4px 8px 18px 8px'>
        <div style='font-size:22px;font-weight:800;color:{COLORS.get("neutro_escuro", "#374151")};letter-spacing:-0.3px'>
          📊 Visão Executiva - Quebrada por etapas | Produto Acabado
        </div>
      </div>

      {html_pa}

    </div>
    """

    display(HTML(container_html))

## Lead Time quebrado por etapas

In [27]:
# Célula 5 — Nível 1: Série Temporal Mensal por Fluxo
# Substitui o snapshot Mediana×P75 estático: o último ponto da série já entrega o snapshot
# e ainda dá tendência. Faixa P25–P75 sombreada + barras de %120d no eixo secundário.

# Cortar pelo cohort configurado (jan/2026 em diante, respeitando data_inicio do filtro)
df_ts = df_ops.copy()


def serie_por_fluxo(df_subset):
    agg = (
        df_subset.groupby("mes_fechamento")
        .agg(
            mediana=("lead_time_realizado", "median"),
            p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
            p90=("lead_time_realizado", lambda x: x.quantile(0.90)),
            pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            n_ops=("op_code", "count"),
        )
        .reset_index()
    )
    # Filtro de robustez: meses com < 10 OPs viram NaN
    agg.loc[agg["n_ops"] < 10, ["mediana", "p75", "p90", "pct_no_prazo"]] = None
    return agg


ts_tri = serie_por_fluxo(df_ts[~df_ts["is_finished_product_order"]])
ts_pa  = serie_por_fluxo(df_ts[df_ts["is_finished_product_order"]])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Triangulação", "Produto Acabado"),
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    horizontal_spacing=0.12,
)


def add_fluxo(fig, ts, col, cor_fluxo, nome):
    # Faixa P25–P75 (sombreada)
    fig.add_trace(go.Scatter(
        x=list(ts["mes_fechamento"]) + list(ts["mes_fechamento"])[::-1],
        y=list(ts["p90"]) + list(ts["p75"])[::-1],
        fill="toself", fillcolor=cor_fluxo, opacity=0.15,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
        name=f"P75–P90 {nome}",
    ), row=1, col=col, secondary_y=False)

    # Linha mediana (forte)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["mediana"], mode="lines+markers",
        line=dict(color=cor_fluxo, width=3), marker=dict(size=8),
        name=f"Mediana {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Linha P75 (pontilhada)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["p75"], mode="lines",
        line=dict(color=cor_fluxo, width=1.5, dash="dot"),
        name=f"P75 {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Linha P90 (tracejada)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["p90"], mode="lines+markers",
        line=dict(color=cor_fluxo, width=1.5, dash="dash"),
        marker=dict(size=6, symbol="diamond"),
        name=f"P90 {nome}", legendgroup=nome,
        hovertemplate="%{x|%b %Y}<br>P90: <b>%{y:.0f}d</b><extra></extra>",
    ), row=1, col=col, secondary_y=False)

    # Barras %dentro120d no eixo secundário
    fig.add_trace(go.Bar(
        x=ts["mes_fechamento"], y=ts["pct_no_prazo"],
        marker_color=cor_fluxo, opacity=0.25,
        name=f"% no prazo {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=True)

    # Linha target 120d
    fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash",
                  line_color=COLORS["target_line"], row=1, col=col, secondary_y=False)

    # Anotação da variação MoM no último ponto válido
    validos = ts.dropna(subset=["mediana"])
    if len(validos) >= 2:
        ultimo = validos.iloc[-1]
        penultimo = validos.iloc[-2]
        delta = ultimo["mediana"] - penultimo["mediana"]
        arrow = "▼" if delta < 0 else "▲"
        cor_anot = COLORS["status_ok"] if delta < 0 else COLORS["status_critico"]
        fig.add_annotation(
            x=ultimo["mes_fechamento"], y=ultimo["mediana"],
            text=f"{arrow} {abs(delta):.0f}d MoM",
            showarrow=True, arrowhead=2, ax=30, ay=-30,
            font=dict(color=cor_anot, size=11, family="sans-serif"),
            row=1, col=col,
        )


add_fluxo(fig, ts_tri, col=1, cor_fluxo=COLORS["fluxo_tri"], nome="Tri")
add_fluxo(fig, ts_pa,  col=2, cor_fluxo=COLORS["fluxo_pa"],  nome="PA")

fig.update_xaxes(title_text="Mês", row=1, col=1)
fig.update_xaxes(title_text="Mês", row=1, col=2)
fig.update_yaxes(title_text="Dias", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Dias", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="% no prazo", row=1, col=1, secondary_y=True, range=[0, 100])
fig.update_yaxes(title_text="% no prazo", row=1, col=2, secondary_y=True, range=[0, 100])

fig.update_layout(
    title=f"Evolução do Lead Time — Cohort: {data_inicio} → {data_fim}",
    template=TEMPLATE, height=480,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
)
fig.show()


### Decomposição do Lead Time por Fornecedor

In [28]:
# Célula 6 — Nível 2: Decomposição de Lead Time por Etapa (6 etapas, PA e Tri)
# Parâmetro: trocar para "PA", "Tri" ou "Ambos"

FLUXO_FILTRO = "Ambos"

# Escala sequencial para as etapas
ETAPAS_CORES = ["#DBEAFE", "#93C5FD", "#60A5FA", "#2563EB", "#3B82F6", "#1D4ED8", "#1E3A8A"]


def montar_etapas(df_subset, label_fluxo):
    if len(df_subset) == 0:
        return None

    agg = (
        df_subset.groupby("supplier_name")
        .agg(
            **{col: (col, "median") for col in ETAPAS_COLS},
            n_ops=("op_code", "count"),
            lt_total=("lead_time_realizado", "median"),
        )
        .reset_index()
    )

    agg = agg[agg["n_ops"] >= CONFIG["min_ops_grafico"]]

    if len(agg) == 0:
        return None

    agg = agg.nlargest(CONFIG["top_n_fornecedores_grafico"], "n_ops")

    # Soma real do que está sendo exibido na barra empilhada.
    # Esse é o valor correto para o rótulo à direita de cada barra.
    agg["lt_soma_etapas"] = agg[ETAPAS_COLS].fillna(0).sum(axis=1)

    agg["fluxo"] = label_fluxo

    # Ordenar pela soma das etapas.
    # Como é barh, ascending=True deixa os maiores no topo.
    agg = agg.sort_values("lt_soma_etapas", ascending=True)

    return agg


dfs_para_plotar = []

if FLUXO_FILTRO in ("PA", "Ambos"):
    pa_etapas = montar_etapas(df_pa, "PA")
    if pa_etapas is not None:
        dfs_para_plotar.append(pa_etapas)

if FLUXO_FILTRO in ("Tri", "Ambos"):
    tri_etapas = montar_etapas(df_tri, "Tri")
    if tri_etapas is not None:
        dfs_para_plotar.append(tri_etapas)


if not dfs_para_plotar:
    print(f"⚠ Sem dados suficientes para o filtro FLUXO_FILTRO={FLUXO_FILTRO!r}")

else:
    n_subplots = len(dfs_para_plotar)

    fig = make_subplots(
        rows=1,
        cols=n_subplots,
        subplot_titles=[
            f"{d['fluxo'].iloc[0]} (top {len(d)} por volume)"
            for d in dfs_para_plotar
        ],
        shared_yaxes=False,
        horizontal_spacing=0.18,
    )

    for idx, agg in enumerate(dfs_para_plotar, start=1):
        max_total = agg["lt_soma_etapas"].max()

        for i, (col, label) in enumerate(zip(ETAPAS_COLS, ETAPAS_LABELS)):
            if agg[col].isna().all():
                continue

            fig.add_trace(
                go.Bar(
                    y=agg["supplier_name"],
                    x=agg[col],
                    name=label,
                    orientation="h",
                    marker_color=ETAPAS_CORES[i],
                    text=[
                        f"{v:.0f}d" if pd.notna(v) and v >= 10 else ""
                        for v in agg[col]
                    ],
                    textposition="inside",
                    insidetextanchor="middle",
                    textfont=dict(
                        size=10,
                        color="white" if i >= 3 else "#374151",
                    ),
                    hovertemplate=(
                        f"<b>%{{y}}</b><br>"
                        f"{label}: %{{x:.0f}}d"
                        "<extra></extra>"
                    ),
                    showlegend=(idx == 1),
                    legendgroup=label,
                ),
                row=1,
                col=idx,
            )

        # Rótulo da soma total exatamente à direita do fim da barra empilhada.
        # Usa x = lt_soma_etapas e xshift em pixels, evitando deslocamento em "dias".
        for _, linha in agg.iterrows():
            total_barra = linha["lt_soma_etapas"]

            if pd.notna(total_barra):
                fig.add_annotation(
                    x=total_barra,
                    y=linha["supplier_name"],
                    text=f"<b>{total_barra:.0f}d</b>",
                    showarrow=False,
                    xanchor="left",
                    yanchor="middle",
                    xshift=8,
                    font=dict(size=12, color="#111827"),
                    row=1,
                    col=idx,
                )

        fig.add_vline(
            x=TARGET_LEAD_TIME,
            line_dash="dash",
            line_color=COLORS["target_line"],
            row=1,
            col=idx,
        )

        # Abre espaço à direita para os rótulos da soma total.
        fig.update_xaxes(
            title_text="Dias (mediana)",
            range=[
                0,
                max(
                    max_total * 1.25,
                    TARGET_LEAD_TIME * 1.10,
                ),
            ],
            row=1,
            col=idx,
        )

    fig.update_layout(
        title=f"Decomposição de Lead Time por Etapa — {FLUXO_FILTRO}",
        barmode="stack",
        template=TEMPLATE,
        height=max(450, 30 * max(len(d) for d in dfs_para_plotar) + 120),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.24,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(
            l=40,
            r=100,
            t=80,
            b=130,
        ),
    )

    fig.show()

In [29]:
# Ranking de Fornecedores por Lead Time Médio
# Robusto para filtros combinados de fornecedor + produto, inclusive quando
# há apenas PA, apenas Tri, ou nenhuma amostra acima do mínimo de OPs.

MIN_OPS_R1 = globals().get("MIN_OPS_R1", CONFIG.get("min_ops_grafico", 3))
_display_r1 = [
    "rank",
    "supplier_name",
    "n_ops",
    "lt_medio",
    "lt_p75",
    "lt_medio_pa",
    "lt_medio_tri",
    "pct_no_prazo",
]

_ranking = pd.DataFrame(columns=_display_r1)
_base_r1 = df_ops.copy()

if len(_base_r1) == 0:
    print("Sem OPs para o filtro selecionado.")
else:
    _base_r1 = _base_r1.dropna(subset=["supplier_name", "lead_time_realizado"])

    if len(_base_r1) == 0:
        print("Sem OPs com lead time realizado para o filtro selecionado.")
    else:
        _ranking = (
            _base_r1
            .groupby("supplier_name", dropna=False)
            .agg(
                n_ops=("op_code", "count"),
                lt_medio=("lead_time_realizado", "mean"),
                lt_p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
                pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            )
            .reset_index()
        )

        _fluxo = (
            _base_r1
            .groupby(["supplier_name", "is_finished_product_order"], dropna=False)["lead_time_realizado"]
            .mean()
            .unstack()
        )

        _ranking["lt_medio_pa"] = _ranking["supplier_name"].map(
            _fluxo[True] if True in _fluxo.columns else pd.Series(dtype="float64")
        )
        _ranking["lt_medio_tri"] = _ranking["supplier_name"].map(
            _fluxo[False] if False in _fluxo.columns else pd.Series(dtype="float64")
        )

        _ranking = _ranking[_ranking["n_ops"] >= MIN_OPS_R1].copy()
        _ranking = _ranking.sort_values("lt_medio", ascending=False).reset_index(drop=True)
        _ranking["rank"] = np.arange(1, len(_ranking) + 1)
        _ranking = _ranking.reindex(columns=_display_r1)

print(f"Ranking de Fornecedores por Lead Time Médio | {len(_ranking)} fornecedores | mín {MIN_OPS_R1} OPs")

if _ranking.empty:
    print("Sem fornecedores com amostra suficiente para o filtro selecionado.")
    display(_ranking[_display_r1])
else:
    (
        _ranking[_display_r1]
        .style
        .background_gradient(subset=["lt_medio", "lt_p75", "lt_medio_pa", "lt_medio_tri"], cmap="Reds", axis=None)
        .background_gradient(subset=["pct_no_prazo"], cmap="Greens", axis=None)
        .format({
            "lt_medio":     "{:.0f}d",
            "lt_p75":       "{:.0f}d",
            "lt_medio_pa":  "{:.0f}d",
            "lt_medio_tri": "{:.0f}d",
            "pct_no_prazo": "{:.1f}%",
        }, na_rep="-")
        .hide(axis="index")
    )

Ranking de Fornecedores por Lead Time Médio | 27 fornecedores | mín 3 OPs


In [30]:
# Célula 6.1 — Tabela detalhada de etapas: Fornecedor × Produto × Fluxo
# Reproduz o formato da tabela "Gargalo por etapa" da guilda de LDT.

PRODUTO_FILTRO = None         # None = todos | ou string parcial (ex: "Tech T-shirt")
FLUXO_FILTRO_TABELA = "Ambos" # "PA", "Tri" ou "Ambos"
MIN_OPS_TABELA = 3

_agg_spec = {col: (col, "median") for col in ETAPAS_COLS}
_agg_spec["n_ops"] = ("op_code", "count")
_agg_spec["lt_total"] = ("lead_time_realizado", "median")

base = df_ops.copy()

if FLUXO_FILTRO_TABELA == "PA":
    base = base[base["is_finished_product_order"] == True]
elif FLUXO_FILTRO_TABELA == "Tri":
    base = base[base["is_finished_product_order"] == False]

if PRODUTO_FILTRO:
    base = base[base["product_names"].str.contains(PRODUTO_FILTRO, case=False, na=False)]

# Remover a coluna de criação -> agd das etapas ANTES da agregação/renomeação para que não apareça em nenhuma parte da tabela
ETAPA_REMOVER_COL = "etapa_criacao_agd"
ETAPA_REMOVER_LABELS = {"Criação->agd", "Criação→agd", "Criação→Agd", "Criação -> Agd", "Criação → Agd", "Criação→AGD", "Criação-›agd"}

# Garante que a coluna exista na base; se existir, zera a influência removendo-a da lista de etapas
if ETAPA_REMOVER_COL in ETAPAS_COLS:
    ETAPAS_COLS_TABELA = [c for c in ETAPAS_COLS if c != ETAPA_REMOVER_COL]
    ETAPAS_LABELS_TABELA_BASE = [label for col, label in zip(ETAPAS_COLS, ETAPAS_LABELS) if col != ETAPA_REMOVER_COL]
else:
    ETAPAS_COLS_TABELA = ETAPAS_COLS.copy()
    ETAPAS_LABELS_TABELA_BASE = ETAPAS_LABELS.copy()

# Construir spec de agregação apenas com as etapas desejadas
_agg_spec = {col: (col, "median") for col in ETAPAS_COLS_TABELA}
_agg_spec["n_ops"] = ("op_code", "count")
_agg_spec["lt_total"] = ("lead_time_realizado", "median")

# Agregar
tabela = (
    base.groupby(["supplier_name", "product_names", "is_finished_product_order"], dropna=False)
    .agg(**_agg_spec)
    .reset_index()
)

# Filtrar mínimo de OPs
tabela = tabela[tabela["n_ops"] >= MIN_OPS_TABELA].copy()

# Fluxo legível
tabela["fluxo"] = tabela["is_finished_product_order"].map({True: "PA", False: "Tri"})

# Identificar gargalo principal por linha (apenas entre as etapas exibidas)
# Importante: continua ignorando a etapa de Criação→agd ao buscar o gargalo
if ETAPAS_COLS_TABELA:
    tabela["gargalo_etapa"] = tabela[ETAPAS_COLS_TABELA].idxmax(axis=1).map(
        dict(zip(ETAPAS_COLS_TABELA, ETAPAS_LABELS_TABELA_BASE))
    )
    tabela["gargalo_dias"] = tabela[ETAPAS_COLS_TABELA].max(axis=1)
else:
    tabela["gargalo_etapa"] = np.nan
    tabela["gargalo_dias"] = np.nan

# Texto do gargalo principal
tabela["gargalo_principal"] = (
    tabela["gargalo_etapa"].astype(str)
    + " ("
    + tabela["gargalo_dias"].round(0).astype("Int64").astype(str)
    + "d)"
)

# Ordenar por lt_total decrescente
tabela = tabela.sort_values("lt_total", ascending=False)

# Truncar nome de produto
tabela["produto"] = tabela["product_names"].astype(str).str[:35]

# Renomear colunas de etapas para os labels finais, mantendo apenas as exibidas
rename_map = dict(zip(ETAPAS_COLS_TABELA, ETAPAS_LABELS_TABELA_BASE))
tabela = tabela.rename(columns=rename_map)

# Labels finais que serão mostrados (já sem a etapa de Criação→agd)
ETAPAS_LABELS_TABELA = ETAPAS_LABELS_TABELA_BASE

# Colunas para exibição — não incluir a etapa de Criação→agd em nenhuma hipótese
display_cols = [
    "supplier_name",
    "produto",
    "fluxo",
    "n_ops",
    "lt_total",
    *ETAPAS_LABELS_TABELA,
    "gargalo_principal",
]


def color_fluxo(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""

print(
    f"Tabela: {len(tabela)} pares (fornecedor × produto × fluxo) — "
    f"filtro: {FLUXO_FILTRO_TABELA}, mín {MIN_OPS_TABELA} OPs"
)

(
    tabela[display_cols]
    .style
    .background_gradient(subset=ETAPAS_LABELS_TABELA, cmap="Blues", axis=None)
    .map(color_fluxo, subset=["fluxo"])
    .format({
        "lt_total": "{:.0f}d",
        **{label: "{:.0f}d" for label in ETAPAS_LABELS_TABELA},
    }, na_rep="—")
    .hide(axis="index")
)

Tabela: 64 pares (fornecedor × produto × fluxo) — filtro: Ambos, mín 3 OPs


supplier_name,produto,fluxo,n_ops,lt_total,Agd da OP,Aguardo MP,Aguardo Corte,Costura,Inspeção,Faturamento,gargalo_principal
Conceitun,Calça Director Masculino,PA,6,218d,—,120d,53d,34d,7d,4d,Aguardo MP (120d)
NOVA FORMULA,Camiseta Henley Core Masculino,Tri,3,192d,32d,40d,73d,22d,6d,3d,Aguardo Corte (73d)
RIZLLEP,The Perfect Top V Feminino,PA,3,188d,—,30d,45d,24d,79d,10d,Inspeção (79d)
LUTESTIL,Calcinha Minimal Corte a Laser Femi,PA,4,171d,0d,24d,104d,42d,2d,1d,Aguardo Corte (104d)
ART LIVRE,Tech T-shirt Heavy Slim Masculino,PA,4,164d,14d,102d,39d,9d,9d,4d,Aguardo MP (102d)
LUTESTIL,Undershirt Anti Suor Gola V Masculi,PA,3,159d,—,43d,74d,41d,6d,1d,Aguardo Corte (74d)
RIZLLEP,Saia Envelope Breeze Feminino,PA,5,152d,—,30d,39d,53d,31d,6d,Costura (53d)
DDAL,Wingsuit Feminino,Tri,33,149d,4d,80d,20d,25d,7d,3d,Aguardo MP (80d)
MC & MC,Camisa FutureForm Feminino,PA,5,149d,37d,82d,56d,12d,6d,1d,Aguardo MP (82d)
NATURAL COMPANY CONFECCOES LTDA,Camiseta Grafeno Edition Masculino,PA,4,148d,—,14d,36d,88d,7d,4d,Costura (88d)


### Decomposição do Lead Time por Matéria Prima

In [31]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_mpp = _dntk.execute_sql(
  'WITH fabric_costs AS (\n        SELECT\n            mfs.id AS fabric_sku_id,\n            mfs.fabric_id,\n            mfs.knitting_factory_id,\n            mfs.sku AS fabric_sku,\n            mfs.invoice_fabric_name AS factory_fabric_name,\n            mfs.unit_price,\n            mfs.minimum_volume_per_order,\n            mfs.multiple_volume_per_order,\n            mf.name AS fabric_name,\n            mf.article_id,\n            ma.name AS article_name,\n            ma.unit AS article_unit,\n            mkf.supplier_id,\n            ms.alias AS knitting_factory_name\n        FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs\n        LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id\n        LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\n        LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id\n        LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id\n    ),\n    fabric_min_max_cost AS (\n        SELECT\n            fc.fabric_id,\n            fc.fabric_name,\n            MIN(fc.unit_price) AS min_fabric_cost,\n            MAX(fc.unit_price) AS max_fabric_cost,\n            MIN(fc.minimum_volume_per_order) AS minimum_volume_per_order,\n            COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,\n            ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,\n        FROM fabric_costs AS fc\n        GROUP BY fc.fabric_id, fc.fabric_name\n    ),\n    skp_with_sales_l8m AS (\n        SELECT DISTINCT s.product_name\n        FROM `insider-data-lake.fpa.analytical_dre` d\n        LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)\n        WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)\n        AND d.order_status != \'Not authorized\'\n        AND d.quantity > 0\n        AND s.product_name IS NOT NULL\n    ),\n    skp_status AS (\n        SELECT\n            s.product_name,\n            CASE\n                WHEN COUNTIF(s.sku_state = \'ativo_perene\') > 0         THEN \'ativo_perene\'\n                WHEN COUNTIF(s.sku_state = \'ativo_em_lancamento\') > 0  THEN \'ativo_em_lancamento\'\n                WHEN COUNTIF(s.sku_state = \'ativo_capsula\') > 0        THEN \'ativo_capsula\'\n                WHEN COUNTIF(s.sku_state = \'personalizacao\') > 0       THEN \'personalizacao\'\n                WHEN COUNTIF(s.sku_state = \'kit\') > 0                  THEN \'kit\'\n                ELSE \'desativado\'\n            END AS product_status\n        FROM `insider-data-lake.integrated.skus` s\n        INNER JOIN skp_with_sales_l8m l8m USING(product_name)\n        GROUP BY s.product_name\n    ),\n    sku_fabrics AS (\n        SELECT\n            mps.sku,\n            mps.sku_name,\n            s.sku_state,\n            s.gender,\n            s.color,\n            s.size,\n            s.product_name,\n            mpsf.fabric_id,\n            mf.name AS fabric_name,\n            mpsf.consumption,\n            fc.min_fabric_cost AS min_fabric_unitary_cost,\n            fc.max_fabric_cost AS max_fabric_unitary_cost,\n            fc.minimum_volume_per_order,\n            fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,\n            fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,\n            ma.unit AS article_unit,\n            ma.name AS article_name,\n            mf.article_id,\n            fc.number_knitting_factories,\n            fc.knitting_factories_names\n        FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf\n        LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id\n        LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id\n        LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id\n        LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku\n        LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id\n        INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name\n    ),\n    product_article_agg AS (\n        SELECT\n            sf.product_name,\n            ss.product_status,\n            REGEXP_REPLACE(sf.article_name, r\'Modal \\\\(\\\\d+\\\\)\', \'Modal\') AS article_name,\n            sf.article_unit,\n            MIN(sf.minimum_volume_per_order) AS minimum_volume_per_order,\n            APPROX_QUANTILES(sf.consumption, 2)[OFFSET(1)] AS median_article_consumption\n        FROM sku_fabrics AS sf\n        INNER JOIN skp_status AS ss ON ss.product_name = sf.product_name\n        WHERE ss.product_status IN (\'ativo_perene\', \'ativo_em_lancamento\', \'desativado\')\n        AND LOWER(sf.product_name) NOT LIKE \'%ziraldo%\'\n        AND LOWER(sf.product_name) NOT LIKE \'% xp%\'\n        AND LOWER(sf.product_name) NOT LIKE \'%maluquinho%\'\n        AND LOWER(sf.product_name) NOT LIKE \'% b2b %\'\n        GROUP BY sf.product_name, ss.product_status, article_name, sf.article_unit\n    )\n\n    SELECT\n        product_name,\n        product_status,\n        article_name AS tecido_principal,\n        article_unit,\n        ROUND(CAST(median_article_consumption AS FLOAT64), 4) AS consumo_mediano,\n        minimum_volume_per_order,\n        COUNT(*) OVER (PARTITION BY product_name) AS qtd_tecidos_total\n    FROM product_article_agg\n    QUALIFY ROW_NUMBER() OVER (\n        PARTITION BY product_name\n        ORDER BY median_article_consumption DESC\n    ) = 1\n    ORDER BY product_name ASC',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_mpp

,product_name,product_status,tecido_principal,article_unit,consumo_mediano,minimum_volume_per_order,qtd_tecidos_total
0,Action Top Feminino,desativado,Sportiva Pro,kg,0.150,97.0,1
1,Air Blouse Feminino,desativado,String Stretch,kg,0.392,153.0,1
2,Air Loop Top 2.0 Feminino,desativado,New String Stretch,m,0.330,78.0,1
3,Air Loop Top Feminino,desativado,String Stretch,kg,0.330,153.0,1
4,Bermuda Kyoto Feminino,ativo_perene,Nylon WR 50+,m,0.900,60.0,1
...,...,...,...,...,...,...,...
161,Vestido Tube Dress Curto Feminino,desativado,Staff Special,kg,0.327,140.0,1
162,Vestido Wingsuit Feminino,ativo_perene,Top Visco Comfort,kg,1.128,84.0,1
163,Viseira Esportiva JoggIn,ativo_perene,Mac Power,kg,0.036,NaN,1
164,Wingsuit Feminino,ativo_perene,Boucle,kg,0.610,136.0,1


In [32]:
# Bloco 6-MP — Decomposição de Lead Time por Matéria-prima (tecido principal)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Verificações básicas
missing = [v for v in ["df_ops", "df_mpp", "ETAPAS_COLS", "ETAPAS_LABELS", "CONFIG", "TARGET_LEAD_TIME", "TEMPLATE", "COLORS"] if v not in globals()]
if missing:
    print(f"Variáveis ausentes: {missing}. Execute as células anteriores.")
else:
    # Preparar base com tecido principal por OP (mapeia product_names -> tecido_principal)
    map_tecido = df_mpp[["product_name", "tecido_principal"]].drop_duplicates().rename(columns={"product_name": "product_names"})

    base = df_ops.merge(map_tecido, on="product_names", how="left")

    # Se não encontrou tecido, marca como 'Sem mapeamento'
    base["tecido_principal"] = base["tecido_principal"].fillna("Sem mapeamento")

    def montar_etapas_mp(df_subset, label_fluxo):
        if len(df_subset) == 0:
            return None
        agg = (
            df_subset.groupby("tecido_principal")
            .agg(**{col: (col, "median") for col in ETAPAS_COLS},
                 n_ops=("op_code", "count"),
                 lt_total=("lead_time_realizado", "median"))
            .reset_index()
        )
        # Mantém apenas grupos com volume mínimo, mas NÃO limita ao top-N
        agg = agg[agg["n_ops"] >= CONFIG["min_ops_grafico"]]
        if len(agg) == 0:
            return None
        # Soma das etapas exibidas
        agg["lt_soma_etapas"] = agg[ETAPAS_COLS].fillna(0).sum(axis=1)
        agg["fluxo"] = label_fluxo
        # Ordenar pela soma
        agg = agg.sort_values("lt_soma_etapas", ascending=True)
        return agg

    FLUXO_FILTRO_MP = "Ambos"  # "PA", "Tri" ou "Ambos"

    dfs_plot = []
    if FLUXO_FILTRO_MP in ("PA", "Ambos"):
        pa = montar_etapas_mp(base[base["is_finished_product_order"]], "PA")
        if pa is not None:
            dfs_plot.append(pa)
    if FLUXO_FILTRO_MP in ("Tri", "Ambos"):
        tri = montar_etapas_mp(base[~base["is_finished_product_order"]], "Tri")
        if tri is not None:
            dfs_plot.append(tri)

    if not dfs_plot:
        print("⚠ Sem dados suficientes para decompor por matéria-prima com os filtros atuais.")
    else:
        fig = make_subplots(
            rows=1,
            cols=len(dfs_plot),
            subplot_titles=[f"{d['fluxo'].iloc[0]} (total {len(d)} materiais)" for d in dfs_plot],
            shared_yaxes=False,
            horizontal_spacing=0.18,
        )

        ETAPAS_CORES = ["#DBEAFE", "#93C5FD", "#60A5FA", "#2563EB", "#3B82F6", "#1D4ED8", "#1E3A8A"]

        for idx, agg in enumerate(dfs_plot, start=1):
            max_total = agg["lt_soma_etapas"].max()
            for i, (col, label) in enumerate(zip(ETAPAS_COLS, ETAPAS_LABELS)):
                if agg[col].isna().all():
                    continue
                fig.add_trace(
                    go.Bar(
                        y=agg["tecido_principal"],
                        x=agg[col],
                        name=label,
                        orientation="h",
                        marker_color=ETAPAS_CORES[i],
                        text=[f"{v:.0f}d" if pd.notna(v) and v >= 10 else "" for v in agg[col]],
                        textposition="inside",
                        insidetextanchor="middle",
                        textfont=dict(size=10, color="#111827" if i < 3 else "white"),
                        hovertemplate=(f"<b>%{{y}}</b><br>{label}: %{{x:.0f}}d<extra></extra>"),
                        showlegend=(idx == 1),
                        legendgroup=label,
                    ),
                    row=1, col=idx,
                )

            # Rótulo total da barra
            for _, linha in agg.iterrows():
                total_barra = linha["lt_soma_etapas"]
                if pd.notna(total_barra):
                    fig.add_annotation(
                        x=total_barra,
                        y=linha["tecido_principal"],
                        text=f"<b>{total_barra:.0f}d</b>",
                        showarrow=False,
                        xanchor="left",
                        yanchor="middle",
                        xshift=8,
                        font=dict(size=12, color="#111827"),
                        row=1, col=idx,
                    )

            fig.add_vline(x=TARGET_LEAD_TIME, line_dash="dash", line_color=COLORS["target_line"], row=1, col=idx)

            fig.update_xaxes(
                title_text="Dias (mediana)",
                range=[0, max(max_total * 1.25, TARGET_LEAD_TIME * 1.10)],
                row=1, col=idx,
            )

        fig.update_layout(
            title="Decomposição de Lead Time por Etapa — por Matéria-prima (tecido principal)",
            barmode="stack",
            template=TEMPLATE,
            height=max(450, 30 * max(len(d) for d in dfs_plot) + 120),
            legend=dict(orientation="h", yanchor="bottom", y=-0.24, xanchor="center", x=0.5),
            margin=dict(l=40, r=100, t=80, b=130),
        )

        fig.show()

    # Também gerar uma tabela resumo para export/insumo adicional
    def tabela_resumo_mp(df_base):
        agg = (
            df_base.groupby(["tecido_principal", "is_finished_product_order"]) 
            .agg(
                n_ops=("op_code", "nunique"),
                lt_mediana=("lead_time_realizado", "median"),
                lt_p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
                lt_p90=("lead_time_realizado", lambda x: x.quantile(0.90)),
                pct_no_prazo=("dentro_do_prazo", "mean"),
            ).reset_index()
        )
        agg["fluxo"] = np.where(agg["is_finished_product_order"], "PA", "Tri")
        agg = agg.drop(columns=["is_finished_product_order"]) 
        agg["pct_no_prazo"] = (agg["pct_no_prazo"] * 100).round(1)
        num_cols = ["lt_mediana", "lt_p75", "lt_p90"]
        agg[num_cols] = agg[num_cols].round(1)
        return agg.sort_values(["fluxo", "lt_mediana"], ascending=[True, True])

    df_leadtime_por_mp = tabela_resumo_mp(base)
    df_leadtime_por_mp = df_leadtime_por_mp[df_leadtime_por_mp["n_ops"] >= CONFIG["min_ops_grafico"]]

    df_leadtime_por_mp

In [33]:
# Reexecutar dependências mínimas caso a célula anterior acuse variáveis ausentes
import pandas as pd
import numpy as np

# Tentar recuperar variáveis já definidas em células anteriores do notebook
need = [v for v in ["df_ops", "ETAPAS_COLS", "ETAPAS_LABELS", "CONFIG", "TARGET_LEAD_TIME", "TEMPLATE", "COLORS"] if v not in globals()]
print("Faltando:", need)


Faltando: []


## Lead time quebrado por volume

In [34]:
# Célula 8.1 — Nível 1: LDT mediano por bucket de volume × fluxo (agregado executivo)
matrix = (
    df_ops.groupby(["volume_bucket", "is_finished_product_order"], observed=True)["lead_time_realizado"]
    .agg(["median", "count"])
    .reset_index()
)
matrix.columns = ["volume_bucket", "is_finished_product_order", "mediana", "n_ops"]
matrix["fluxo"] = matrix["is_finished_product_order"].map({True: "Produto Acabado", False: "Triangulação"})
matrix = matrix[matrix["n_ops"] >= CONFIG["min_ops_grafico"]]

fig = px.bar(
    matrix, x="volume_bucket", y="mediana", color="fluxo", barmode="group",
    text=matrix.apply(lambda r: f"{r['mediana']:.0f}d (n={r['n_ops']})" if r['n_ops'] >= 10 else f"{r['mediana']:.0f}d", axis=1),
    color_discrete_map={
        "Triangulação": COLORS["fluxo_tri"],
        "Produto Acabado": COLORS["fluxo_pa"],
    },
    labels={"mediana": "Lead Time Mediano (dias)", "volume_bucket": "Faixa de Volume"},
    title="LDT Mediano por Faixa de Volume × Fluxo",
    template=TEMPLATE,
    category_orders={"volume_bucket": CONFIG["labels_volume"]},
)
fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash", line_color=COLORS["target_line"],
              annotation_text=f"Target {TARGET_LEAD_TIME}d")
fig.update_traces(textposition="outside")
fig.update_layout(height=420)
fig.show()


In [35]:
# Célula 8.2 — Nível 2: Heatmap Fornecedor × Faixa de Volume (PA e Tri lado a lado)


def montar_heatmap(df_subset, label):
    top_forn = (
        df_subset.groupby("supplier_name")["op_code"].count()
        .nlargest(CONFIG["top_n_fornecedores_heatmap"]).index.tolist()
    )
    pivot_mediana = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["lead_time_realizado"]
        .median().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["op_code"]
        .count().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    # Mascarar células com n_ops < min_ops_grafico
    pivot_mediana = pivot_mediana.where(pivot_n >= CONFIG["min_ops_grafico"])
    # Ordenar por mediana geral do fornecedor (asc = melhor no topo)
    ordem = pivot_mediana.median(axis=1).sort_values(ascending=True).index
    return pivot_mediana.loc[ordem], pivot_n.loc[ordem], label


hm_tri, n_tri, _ = montar_heatmap(df_tri, "Triangulação")
hm_pa,  n_pa,  _ = montar_heatmap(df_pa,  "Produto Acabado")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Triangulação", "Produto Acabado"),
                    horizontal_spacing=0.18)

for idx, (hm, n_df, _) in enumerate([(hm_tri, n_tri, "Tri"), (hm_pa, n_pa, "PA")], start=1):
    texto = hm.copy().astype(object)
    for i in hm.index:
        for j in hm.columns:
            v = hm.loc[i, j]
            n_val = n_df.loc[i, j] if not pd.isna(n_df.loc[i, j]) else 0
            texto.loc[i, j] = f"{v:.0f}d<br>n={int(n_val)}" if pd.notna(v) else "—"

    fig.add_trace(go.Heatmap(
        z=hm.values, x=list(hm.columns), y=list(hm.index),
        text=texto.values, texttemplate="%{text}",
        textfont=dict(size=10),
        colorscale=[
            [0.0,  COLORS["status_ok"]],
            [0.4,  "#FEF3C7"],
            [0.5,  COLORS["target_line"]],
            [0.6,  COLORS["status_atencao"]],
            [1.0,  COLORS["status_critico"]],
        ],
        zmid=TARGET_LEAD_TIME,
        zmin=30, zmax=210,
        showscale=(idx == 2),
        colorbar=dict(title="LDT (d)", x=1.02) if idx == 2 else None,
        hovertemplate="<b>%{y}</b><br>Volume: %{x}<br>LDT: %{z:.0f}d<extra></extra>",
    ), row=1, col=idx)

fig.update_layout(
    title="Heatmap: Lead Time Mediano por Fornecedor × Faixa de Volume",
    template=TEMPLATE,
    height=max(500, 25 * max(len(hm_tri), len(hm_pa)) + 100),
)
fig.update_xaxes(title_text="Faixa de Volume")
fig.show()


In [36]:
# Célula 8.5 — Nível 3: Tabela de Recomendação de Prazo por Fornecedor × Produto × Fluxo
# Output direto da ação P1 da guilda: substituir o prazo padrão de 120d pelo prazo real recomendado.
# Granularidade: fornecedor × product_names × fluxo (PA / Tri)

PERCENTIL = CONFIG["percentil_recomendacao"]
JANELA_LONGA = CONFIG["janela_meses"]
JANELA_CURTA = CONFIG["janela_meses_curta"]

MIN_OPS_REC = 3  # mínimo local — mais permissivo que o global de 5,
                 # pois granularidade por produto reduz n por célula


def recomendacao_por_produto(df_subset, janela_meses, min_ops=MIN_OPS_REC):
    data_corte = (
        pd.Timestamp.now(tz="UTC").normalize()
        - pd.DateOffset(months=janela_meses)
    ).tz_localize(None)

    sub = df_subset[df_subset["mes_fechamento"] >= data_corte].copy()

    agg = (
        sub.groupby(
            [
                "supplier_name",
                "product_names",
                "is_finished_product_order",
            ]
        )
        .agg(
            n_ops=("op_code", "count"),
            lt_realizado_p50=("lead_time_realizado", "median"),
            lt_recomendado_raw=(
                "lead_time_realizado",
                lambda x: x.quantile(PERCENTIL),
            ),
            pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            lt_teorico_cadastrado=("lead_time_teorico", "median"),
        )
        .reset_index()
    )

    agg = agg[agg["n_ops"] >= min_ops].copy()
    agg["lt_recomendado"] = (agg["lt_recomendado_raw"] / 5).round() * 5

    return agg.drop(columns="lt_recomendado_raw")


rec_longa = recomendacao_por_produto(df_ops, JANELA_LONGA)

rec_curta = (
    recomendacao_por_produto(df_ops, JANELA_CURTA)[
        [
            "supplier_name",
            "product_names",
            "is_finished_product_order",
            "lt_recomendado",
            "n_ops",
        ]
    ]
    .rename(
        columns={
            "lt_recomendado": "lt_recomendado_curta",
            "n_ops": "n_ops_curta",
        }
    )
)

rec = rec_longa.merge(
    rec_curta,
    on=[
        "supplier_name",
        "product_names",
        "is_finished_product_order",
    ],
    how="left",
)

rec["fluxo"] = rec["is_finished_product_order"].map(
    {
        True: "PA",
        False: "Tri",
    }
)

# Ajuste Tri já incorporado em lead_time_teorico (Cell 3 / Célula 2.1)
rec["delta_vs_cadastrado"] = (
    rec["lt_recomendado"] - rec["lt_teorico_cadastrado"]
).round(0)


def direcao(d):
    if pd.isna(d):
        return "—"

    if d > 10:
        return "▲ Aumentar"

    if d < -10:
        return "▼ Reduzir"

    return "→ Manter"


def confianca(n):
    if n >= 30:
        return "🟢 Alta"

    if n >= 10:
        return "🟡 Média"

    return "🔴 Baixa"


def tendencia(row):
    if (
        pd.isna(row.get("lt_recomendado_curta"))
        or row.get("n_ops_curta", 0) < 3
    ):
        return "—"

    diff = row["lt_recomendado_curta"] - row["lt_recomendado"]

    if diff < -5:
        return "📉 Melhorando"

    if diff > 5:
        return "📈 Piorando"

    return "≡ Estável"


rec["direcao"] = rec["delta_vs_cadastrado"].apply(direcao)
rec["confianca"] = rec["n_ops"].apply(confianca)
rec["tendencia_3m"] = rec.apply(tendencia, axis=1)

rec["produto"] = rec["product_names"].str[:40]

rec = rec.sort_values(
    "delta_vs_cadastrado",
    key=lambda x: x.abs(),
    ascending=False,
)

display_cols = [
    "supplier_name",
    "produto",
    "fluxo",
    "n_ops",
    "lt_teorico_cadastrado",
    "lt_recomendado",
    "lt_recomendado_curta",
    "delta_vs_cadastrado",
    "direcao",
    "pct_no_prazo",
    "tendencia_3m",
    "confianca",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (
            f"background-color:{COLORS['fluxo_pa']};"
            "color:white; font-weight:600; text-align:center"
        )

    if v == "Tri":
        return (
            f"background-color:{COLORS['fluxo_tri']};"
            "color:#374151; font-weight:600; text-align:center"
        )

    return ""


def color_direcao(v):
    if "Aumentar" in str(v):
        return (
            f"background-color:{COLORS['status_critico']}; "
            "color:white; font-weight:600"
        )

    if "Reduzir" in str(v):
        return (
            f"background-color:{COLORS['status_ok']}; "
            "color:white; font-weight:600"
        )

    return ""


print(
    f"📋 Recomendação de Prazo — P{int(PERCENTIL * 100)} | "
    f"janela {JANELA_LONGA}m "
    f"(curta: {JANELA_CURTA}m) | "
    f"mín {MIN_OPS_REC} OPs por par"
)
print("   Granularidade: fornecedor × produto × fluxo")
print(f"   Pares com amostra suficiente: {len(rec)}")
print(
    f"   ▲ Aumentar: {(rec['direcao'] == '▲ Aumentar').sum()} | "
    f"▼ Reduzir: {(rec['direcao'] == '▼ Reduzir').sum()} | "
    f"→ Manter: {(rec['direcao'] == '→ Manter').sum()}"
)

(
    rec[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .map(color_direcao, subset=["direcao"])
    .background_gradient(
        subset=["delta_vs_cadastrado"],
        cmap="RdYlGn_r",
        vmin=-40,
        vmax=90,
    )
    .format(
        {
            "lt_teorico_cadastrado": "{:.0f}d",
            "lt_recomendado": "{:.0f}d",
            "lt_recomendado_curta": "{:.0f}d",
            "delta_vs_cadastrado": "{:+.0f}d",
            "pct_no_prazo": "{:.1f}%",
        },
        na_rep="—",
    )
    .hide(axis="index")
)

📋 Recomendação de Prazo — P75 | janela 12m (curta: 3m) | mín 3 OPs por par
   Granularidade: fornecedor × produto × fluxo
   Pares com amostra suficiente: 64
   ▲ Aumentar: 46 | ▼ Reduzir: 5 | → Manter: 13


supplier_name,produto,fluxo,n_ops,lt_teorico_cadastrado,lt_recomendado,lt_recomendado_curta,delta_vs_cadastrado,direcao,pct_no_prazo,tendencia_3m,confianca
Conceitun,Calça Director Masculino,PA,6,90d,235d,235d,+145d,▲ Aumentar,0.0%,≡ Estável,🔴 Baixa
RIZLLEP,The Perfect Top V Feminino,PA,3,90d,200d,200d,+110d,▲ Aumentar,0.0%,≡ Estável,🔴 Baixa
NOVA FORMULA,Camiseta Henley Core Masculino,Tri,3,115d,215d,—,+100d,▲ Aumentar,0.0%,—,🔴 Baixa
MC & MC,Camisa FutureForm Feminino,PA,5,90d,180d,170d,+90d,▲ Aumentar,20.0%,📉 Melhorando,🔴 Baixa
LUTESTIL,Calcinha Minimal Corte a Laser Feminino,PA,4,90d,175d,—,+85d,▲ Aumentar,0.0%,—,🔴 Baixa
BAE BRASIL,NoHo Socks,PA,9,45d,130d,—,+85d,▲ Aumentar,66.7%,—,🔴 Baixa
INDÚSTRIA TEXTIL BETILHA LTDA,Shorts Kyoto Feminino,PA,7,60d,140d,145d,+80d,▲ Aumentar,57.1%,≡ Estável,🔴 Baixa
LUTESTIL,Undershirt Anti Suor Gola V Masculino,PA,3,90d,170d,—,+80d,▲ Aumentar,0.0%,—,🔴 Baixa
BAE BRASIL,Cueca Boxer Performance Simples Masculin,PA,13,75d,150d,155d,+75d,▲ Aumentar,30.8%,≡ Estável,🟡 Média
NATURAL COMPANY CONFECCOES LTDA,Camiseta Grafeno Edition Masculino,PA,4,90d,165d,—,+75d,▲ Aumentar,0.0%,—,🔴 Baixa


## Tabela detalhada — Lead time por fornecedor × produto × faixa de volume

Benchmark: nesta tabela, o benchmark representa a mediana geral de lead time das OPs que pertencem à mesma faixa de volume e ao mesmo fluxo \(Triangulação ou Produto Acabado\), independentemente do fornecedor e do produto\. O campo "Delta vs benchmark" mostra quanto o par fornecedor × produto está acima ou abaixo desse comportamento esperado para OPs comparáveis em volume e fluxo: valores positivos indicam lead time pior que o benchmark; valores negativos indicam desempenho melhor que a referência\.

In [37]:
# Célula 8.3 — Tabela detalhada: lead time por fornecedor × produto × faixa de volume
# Complementa os gráficos 8.1 e 8.2 com o detalhe do par fornecedor-produto.

colunas_base = [
    "op_code",
    "supplier_name",
    "product_names",
    "is_finished_product_order",
    "volume_bucket",
    "lead_time_realizado",
]
colunas_opcionais = [
    "planned_quantity_op",
    "lead_time_teorico",
    "desvio_lt",
    "dentro_do_prazo",
    "dt_planned_entry_warehouse",
]
colunas_disponiveis = [c for c in colunas_base + colunas_opcionais if c in df_ops.columns]

missing = [c for c in colunas_base if c not in df_ops.columns]
if missing:
    raise ValueError(f"Colunas obrigatórias ausentes em df_ops: {missing}")

df_ltd = df_ops[colunas_disponiveis].copy()
df_ltd = df_ltd.dropna(
    subset=[
        "supplier_name",
        "product_names",
        "volume_bucket",
        "lead_time_realizado",
    ]
)

# Garante a ordem executiva dos buckets usada nos gráficos.
df_ltd["volume_bucket"] = pd.Categorical(
    df_ltd["volume_bucket"],
    categories=CONFIG["labels_volume"],
    ordered=True,
)

agg_dict = {
    "n_ops": ("op_code", "count"),
    "lt_p25": ("lead_time_realizado", lambda x: x.quantile(0.25)),
    "lt_mediano": ("lead_time_realizado", "median"),
    "lt_p75": ("lead_time_realizado", lambda x: x.quantile(0.75)),
    "lt_min": ("lead_time_realizado", "min"),
    "lt_max": ("lead_time_realizado", "max"),
}

if "planned_quantity_op" in df_ltd.columns:
    agg_dict.update({
        "qtd_mediana_op": ("planned_quantity_op", "median"),
        "qtd_min_op": ("planned_quantity_op", "min"),
        "qtd_max_op": ("planned_quantity_op", "max"),
    })

if "lead_time_teorico" in df_ltd.columns:
    agg_dict["lt_teorico_mediano"] = ("lead_time_teorico", "median")

if "desvio_lt" in df_ltd.columns:
    agg_dict["desvio_mediano"] = ("desvio_lt", "median")

if "dentro_do_prazo" in df_ltd.columns:
    agg_dict["pct_no_prazo"] = ("dentro_do_prazo", lambda x: x.mean() * 100)

if "dt_planned_entry_warehouse" in df_ltd.columns:
    agg_dict.update({
        "primeira_entrega_planejada": ("dt_planned_entry_warehouse", "min"),
        "ultima_entrega_planejada": ("dt_planned_entry_warehouse", "max"),
    })

tabela_ltd = (
    df_ltd.groupby(
        [
            "supplier_name",
            "product_names",
            "is_finished_product_order",
            "volume_bucket",
        ],
        observed=True,
    )
    .agg(**agg_dict)
    .reset_index()
)

tabela_ltd = tabela_ltd[tabela_ltd["n_ops"] >= CONFIG["min_ops_grafico"]].copy()
tabela_ltd["fluxo"] = tabela_ltd["is_finished_product_order"].map({
    True: "PA",
    False: "Tri",
})

benchmark_bucket_fluxo = (
    df_ltd.groupby(["volume_bucket", "is_finished_product_order"], observed=True)
    .agg(lt_mediano_bucket_fluxo=("lead_time_realizado", "median"))
    .reset_index()
)

resumo_par = (
    tabela_ltd.groupby(
        ["supplier_name", "product_names", "is_finished_product_order"],
        observed=True,
    )
    .agg(
        n_ops_par=("n_ops", "sum"),
        lt_mediano_par=("lt_mediano", "median"),
        buckets_com_dados=("volume_bucket", "nunique"),
    )
    .reset_index()
)

if "qtd_min_op" in tabela_ltd.columns and "qtd_max_op" in tabela_ltd.columns:
    faixa_qtd_par = (
        tabela_ltd.groupby(
            ["supplier_name", "product_names", "is_finished_product_order"],
            observed=True,
        )
        .agg(
            qtd_min_par=("qtd_min_op", "min"),
            qtd_max_par=("qtd_max_op", "max"),
        )
        .reset_index()
    )
    resumo_par = resumo_par.merge(
        faixa_qtd_par,
        on=["supplier_name", "product_names", "is_finished_product_order"],
        how="left",
    )

tabela_ltd = (
    tabela_ltd
    .merge(
        benchmark_bucket_fluxo,
        on=["volume_bucket", "is_finished_product_order"],
        how="left",
    )
    .merge(
        resumo_par,
        on=["supplier_name", "product_names", "is_finished_product_order"],
        how="left",
    )
)

tabela_ltd["delta_vs_bucket_fluxo"] = (
    tabela_ltd["lt_mediano"] - tabela_ltd["lt_mediano_bucket_fluxo"]
)
tabela_ltd["amplitude_p25_p75"] = tabela_ltd["lt_p75"] - tabela_ltd["lt_p25"]

def status_ltd(row):
    if pd.isna(row["lt_mediano"]):
        return "Sem dado"
    if row["lt_mediano"] >= TARGET_LEAD_TIME + CONFIG["threshold_desvio_critico"]:
        return "Crítico"
    if row["lt_mediano"] > TARGET_LEAD_TIME:
        return "Atenção"
    return "OK"


tabela_ltd["status_vs_target"] = tabela_ltd.apply(status_ltd, axis=1)

# Em visão geral, limita a tabela aos pares com mais evidência para manter o notebook leve.
# Quando fornecedor/produto estão filtrados, naturalmente quase tudo entra.
pares_prioritarios = (
    tabela_ltd[[
        "supplier_name",
        "product_names",
        "is_finished_product_order",
        "n_ops_par",
        "buckets_com_dados",
        "lt_mediano_par",
    ]]
    .drop_duplicates()
    .sort_values(
        ["buckets_com_dados", "n_ops_par", "lt_mediano_par"],
        ascending=[False, False, False],
    )
    .head(80)
)

tabela_ltd_view = tabela_ltd.merge(
    pares_prioritarios[["supplier_name", "product_names", "is_finished_product_order"]],
    on=["supplier_name", "product_names", "is_finished_product_order"],
    how="inner",
)

tabela_ltd_view = tabela_ltd_view.sort_values(
    ["supplier_name", "product_names", "fluxo", "volume_bucket"],
    ascending=[True, True, True, True],
)

display_cols_ltd = [
    "supplier_name",
    "product_names",
    "fluxo",
    "volume_bucket",
    "n_ops",
]

for c in ["qtd_mediana_op", "qtd_min_op", "qtd_max_op"]:
    if c in tabela_ltd_view.columns:
        display_cols_ltd.append(c)

display_cols_ltd += [
    "lt_mediano",
    "lt_p25",
    "lt_p75",
    "amplitude_p25_p75",
    "lt_mediano_bucket_fluxo",
    "delta_vs_bucket_fluxo",
]

for c in ["lt_teorico_mediano", "desvio_mediano", "pct_no_prazo"]:
    if c in tabela_ltd_view.columns:
        display_cols_ltd.append(c)

display_cols_ltd += ["buckets_com_dados", "n_ops_par", "status_vs_target"]

rename_ltd = {
    "supplier_name": "Fornecedor",
    "product_names": "Produto",
    "fluxo": "Fluxo",
    "volume_bucket": "Faixa volume OP",
    "n_ops": "OPs na faixa",
    "qtd_mediana_op": "Qtd mediana OP",
    "qtd_min_op": "Qtd mín OP",
    "qtd_max_op": "Qtd máx OP",
    "lt_mediano": "LT mediano",
    "lt_p25": "LT p25",
    "lt_p75": "LT p75",
    "amplitude_p25_p75": "Amplitude p25-p75",
    "lt_mediano_bucket_fluxo": "Benchmark bucket/fluxo",
    "delta_vs_bucket_fluxo": "Delta vs benchmark",
    "lt_teorico_mediano": "LT teórico",
    "desvio_mediano": "Desvio mediano",
    "pct_no_prazo": "% no prazo",
    "buckets_com_dados": "Buckets com dados",
    "n_ops_par": "OPs do par",
    "status_vs_target": "Status target",
}

format_ltd = {
    "Qtd mediana OP": "{:,.0f}",
    "Qtd mín OP": "{:,.0f}",
    "Qtd máx OP": "{:,.0f}",
    "LT mediano": "{:.0f}d",
    "LT p25": "{:.0f}d",
    "LT p75": "{:.0f}d",
    "Amplitude p25-p75": "{:.0f}d",
    "Benchmark bucket/fluxo": "{:.0f}d",
    "Delta vs benchmark": "{:+.0f}d",
    "LT teórico": "{:.0f}d",
    "Desvio mediano": "{:+.0f}d",
    "% no prazo": "{:.1f}%",
}

def color_fluxo_ltd(v):
    if v == "PA":
        return (
            f"background-color:{COLORS['fluxo_pa']};"
            "color:white; font-weight:600; text-align:center"
        )
    if v == "Tri":
        return (
            f"background-color:{COLORS['fluxo_tri']};"
            "color:#374151; font-weight:600; text-align:center"
        )
    return ""


def color_status_ltd(v):
    if v == "Crítico":
        return (
            f"background-color:{COLORS['status_critico']};"
            "color:white; font-weight:600; text-align:center"
        )
    if v == "Atenção":
        return (
            f"background-color:{COLORS['status_atencao']};"
            "color:white; font-weight:600; text-align:center"
        )
    if v == "OK":
        return (
            f"background-color:{COLORS['status_ok']};"
            "color:white; font-weight:600; text-align:center"
        )
    return ""

print(
    "Tabela detalhada de lead time por fornecedor × produto × faixa de volume"
    f"\nPares exibidos: {pares_prioritarios.shape[0]} | "
    f"linhas exibidas: {len(tabela_ltd_view)} | "
    f"mínimo: {CONFIG['min_ops_grafico']} OPs por célula"
)
print(
    "Leitura: Delta vs benchmark compara o LT mediano do par com a mediana geral "
    "do mesmo bucket de volume e fluxo."
)

styled_ltd = (
    tabela_ltd_view[display_cols_ltd]
    .rename(columns=rename_ltd)
    .style
    .map(color_fluxo_ltd, subset=["Fluxo"])
    .map(color_status_ltd, subset=["Status target"])
    .background_gradient(
        subset=["Delta vs benchmark"],
        cmap="RdYlGn_r",
        vmin=-60,
        vmax=120,
    )
    .background_gradient(
        subset=["Amplitude p25-p75"],
        cmap="YlOrRd",
        vmin=0,
        vmax=120,
    )
    .format(format_ltd, na_rep="—")
    .hide(axis="index")
)

styled_ltd

Tabela detalhada de lead time por fornecedor × produto × faixa de volume
Pares exibidos: 54 | linhas exibidas: 77 | mínimo: 3 OPs por célula
Leitura: Delta vs benchmark compara o LT mediano do par com a mediana geral do mesmo bucket de volume e fluxo.


Fornecedor,Produto,Fluxo,Faixa volume OP,OPs na faixa,Qtd mediana OP,Qtd mín OP,Qtd máx OP,LT mediano,LT p25,LT p75,Amplitude p25-p75,Benchmark bucket/fluxo,Delta vs benchmark,LT teórico,Desvio mediano,% no prazo,Buckets com dados,OPs do par,Status target
ART LIVRE,Daily T-shirt Feminino,PA,1000-1999,4,"1,739","1,738","1,740",129d,103d,168d,64d,106d,+23d,100d,+29d,50.0%,1,4,Atenção
ART LIVRE,Daily T-shirt Masculino,PA,1000-1999,14,"1,678","1,002","1,990",111d,92d,124d,32d,106d,+5d,100d,+11d,71.4%,2,34,OK
ART LIVRE,Daily T-shirt Masculino,PA,2000-4999,20,"3,004","2,836","3,140",111d,85d,121d,36d,109d,+2d,100d,+11d,75.0%,2,34,OK
ART LIVRE,Tech T-shirt Gola U Feminino,PA,1000-1999,4,"1,256","1,160","1,863",122d,102d,149d,46d,106d,+16d,100d,+22d,50.0%,1,4,Atenção
ART LIVRE,Tech T-shirt Gola V Feminino,PA,1000-1999,3,"1,001","1,000","1,276",104d,102d,110d,8d,106d,-2d,100d,+4d,100.0%,1,3,OK
ART LIVRE,Tech T-shirt Heavy Masculino,PA,500-999,4,590,500,754,146d,136d,152d,16d,94d,+52d,100d,+46d,25.0%,1,4,Atenção
ART LIVRE,Tech T-shirt Heavy Slim Masculino,PA,500-999,4,500,500,500,164d,153d,174d,21d,94d,+70d,100d,+64d,0.0%,1,4,Crítico
ARTIGO X,Boné Outdoors,PA,1000-1999,4,"1,402","1,367","1,402",107d,99d,125d,26d,106d,+1d,120d,-13d,75.0%,1,4,OK
ARTIGO X,Boné Sixx,PA,<200,3,167,50,190,120d,118d,120d,2d,120d,+0d,113d,+7d,66.7%,3,19,OK
ARTIGO X,Boné Sixx,PA,200-499,12,414,261,469,118d,110d,120d,10d,108d,+11d,113d,+6d,75.0%,3,19,OK


### Risco de Fornecimento Único e Revisão de Leadtime teórico

Esta seção cruza dois olhares de governança: o risco de dependência quando um produto está concentrado em um único fornecedor e a necessidade de revisar o lead time teórico cadastrado quando o histórico realizado indica desvios relevantes\. A leitura ajuda a priorizar pares fornecedor × produto que combinam baixa redundância de fornecimento, criticidade comercial e performance operacional fora do esperado\.

In [38]:
# Célula 8.4 — Download CSV da tabela detalhada completa
# Exporta a versão completa da tabela 8.3, sem o limite visual dos pares prioritários.

from pathlib import Path
from IPython.display import FileLink, display

if "tabela_ltd" not in globals():
    raise ValueError("Execute a Célula 8.3 antes de gerar o CSV detalhado.")

csv_cols_ltd = [
    "supplier_name",
    "product_names",
    "fluxo",
    "volume_bucket",
    "n_ops",
]

for c in ["qtd_mediana_op", "qtd_min_op", "qtd_max_op"]:
    if c in tabela_ltd.columns:
        csv_cols_ltd.append(c)

csv_cols_ltd += [
    "lt_mediano",
    "lt_p25",
    "lt_p75",
    "amplitude_p25_p75",
    "lt_mediano_bucket_fluxo",
    "delta_vs_bucket_fluxo",
]

for c in [
    "lt_teorico_mediano",
    "desvio_mediano",
    "pct_no_prazo",
    "buckets_com_dados",
    "n_ops_par",
    "status_vs_target",
]:
    if c in tabela_ltd.columns:
        csv_cols_ltd.append(c)

csv_cols_ltd = [c for c in csv_cols_ltd if c in tabela_ltd.columns]

csv_rename_ltd = {
    "supplier_name": "Fornecedor",
    "product_names": "Produto",
    "fluxo": "Fluxo",
    "volume_bucket": "Faixa volume OP",
    "n_ops": "OPs na faixa",
    "qtd_mediana_op": "Qtd mediana OP",
    "qtd_min_op": "Qtd min OP",
    "qtd_max_op": "Qtd max OP",
    "lt_mediano": "LT mediano",
    "lt_p25": "LT p25",
    "lt_p75": "LT p75",
    "amplitude_p25_p75": "Amplitude p25-p75",
    "lt_mediano_bucket_fluxo": "Benchmark bucket/fluxo",
    "delta_vs_bucket_fluxo": "Delta vs benchmark",
    "lt_teorico_mediano": "LT teorico",
    "desvio_mediano": "Desvio mediano",
    "pct_no_prazo": "% no prazo",
    "buckets_com_dados": "Buckets com dados",
    "n_ops_par": "OPs do par",
    "status_vs_target": "Status target",
}

tabela_ltd_csv = (
    tabela_ltd[csv_cols_ltd]
    .rename(columns=csv_rename_ltd)
    .sort_values(["Fornecedor", "Produto", "Fluxo", "Faixa volume OP"])
)

export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)
csv_path = export_dir / "lead_time_detalhado_fornecedor_produto_volume.csv"

tabela_ltd_csv.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(
    "CSV detalhado gerado com sucesso"
    f"\nLinhas exportadas: {len(tabela_ltd_csv)}"
    f"\nArquivo: {csv_path}"
)

display(FileLink(str(csv_path), result_html_prefix="Baixar CSV: "))

CSV detalhado gerado com sucesso
Linhas exportadas: 77
Arquivo: exports/lead_time_detalhado_fornecedor_produto_volume.csv


/datasets/_deepnote_work/exports/lead_time_detalhado_fornecedor_produto_volume.csv

In [39]:
# Célula 9 — Nível 3: Risco Single-Source (matriz + tabela fornecedor × produto)

# Identificar produtos single-source
single_source_produtos = df_capacity[df_capacity["num_suppliers_per_product"] == 1][
    ["product_name", "alias", "is_finished_product", "lead_time", "tag_abc"]
].drop_duplicates().rename(columns={"alias": "fornecedor", "lead_time": "lt_teorico"})

# Enriquecer com dados de OPs (lead time realizado, desvio, % no prazo)
ops_por_par = (
    df_ops.groupby(["product_names", "supplier_name", "is_finished_product_order"])
    .agg(
        lt_realizado=("lead_time_realizado", "median"),
        desvio_mediano=("desvio_lt", "median"),
        pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
        n_ops=("op_code", "count"),
    )
    .reset_index()
    .rename(columns={
        "product_names": "product_name",
        "supplier_name": "fornecedor",
        "is_finished_product_order": "is_finished_product",
    })
)

single_source = single_source_produtos.merge(
    ops_por_par,
    how="left",
    on=["product_name", "fornecedor", "is_finished_product"],
)

single_source["fluxo"] = single_source["is_finished_product"].map({
    True: "PA",
    False: "Tri",
})


def nivel(row):
    if row.get("tag_abc") in ["A", "B"]:
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_critico"]:
            return "🔴 Alto"

        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_atencao"]:
            return "🟠 Moderado"

        return "🟡 Acompanhar"

    return "🟢 Baixo"


single_source["nivel_atencao"] = single_source.apply(nivel, axis=1)

# ====== MATRIZ DE RISCO (scatter) ======
matriz = single_source.dropna(subset=["desvio_mediano", "n_ops"]).copy()

simbolo_abc = {
    "A": "circle",
    "B": "square",
    "C": "diamond",
}

matriz["simbolo"] = matriz["tag_abc"].map(simbolo_abc).fillna("x")

fig = go.Figure()

for abc in ["A", "B", "C"]:
    sub = matriz[matriz["tag_abc"] == abc]

    if len(sub) == 0:
        continue

    fig.add_trace(
        go.Scatter(
            x=sub["n_ops"],
            y=sub["desvio_mediano"],
            mode="markers",
            marker=dict(
                symbol=simbolo_abc[abc],
                size=sub["n_ops"].clip(5, 50),
                color=sub["pct_no_prazo"],
                colorscale=[
                    [0.0, COLORS["status_critico"]],
                    [0.5, COLORS["status_atencao"]],
                    [1.0, COLORS["status_ok"]],
                ],
                cmin=0,
                cmax=100,
                showscale=(abc == "A"),
                colorbar=(
                    dict(
                        title=dict(
                            text="% dentro 120d",
                            side="top",
                        ),
                        x=1.025,
                        y=0.43,
                        len=0.72,
                        thickness=22,
                        xanchor="left",
                        yanchor="middle",
                        outlinewidth=0,
                    )
                    if abc == "A"
                    else None
                ),
                line=dict(
                    width=1,
                    color="#374151",
                ),
            ),
            name=f"Tag {abc}",
            text=(
                sub["fornecedor"].astype(str)
                + " — "
                + sub["product_name"].astype(str).str[:30]
                + " ("
                + sub["fluxo"].astype(str)
                + ")"
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "n_OPs: %{x}<br>"
                "Desvio: %{y:.0f}d<br>"
                f"Tag ABC: {abc}<extra></extra>"
            ),
        )
    )

fig.add_hline(
    y=CONFIG["threshold_desvio_critico"],
    line_dash="dash",
    line_color=COLORS["status_critico"],
    annotation_text="Crítico >30d",
    annotation_position="top right",
)

fig.add_hline(
    y=CONFIG["threshold_desvio_atencao"],
    line_dash="dot",
    line_color=COLORS["status_atencao"],
    annotation_text="Atenção >10d",
    annotation_position="top right",
)

fig.add_hline(
    y=0,
    line_color="#374151",
    line_width=0.5,
)

fig.update_layout(
    title="Matriz de Risco Single-Source — Par Fornecedor × Produto",
    template=TEMPLATE,
    xaxis=dict(
        title="Volume (n_OPs últimos 12m) — log",
        type="log",
    ),
    yaxis=dict(
        title="Desvio mediano vs teórico (dias)",
    ),
    height=520,

    # Legenda categórica ABC deslocada para a direita e para cima
    legend=dict(
        title="Criticidade ABC",
        orientation="v",
        x=1.15,
        y=1.00,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        borderwidth=0,
        font=dict(size=12),
        title_font=dict(size=13),
    ),

    # Margem direita maior para caber colorbar + legenda ABC
    margin=dict(
        l=80,
        r=300,
        t=90,
        b=80,
    ),
)

fig.show()

# ====== TABELA fornecedor × produto ======
display_cols = [
    "product_name",
    "fornecedor",
    "fluxo",
    "tag_abc",
    "lt_teorico",
    "lt_realizado",
    "desvio_mediano",
    "pct_no_prazo",
    "n_ops",
    "nivel_atencao",
]


def color_fluxo_cell(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"

    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"

    return ""


def color_atencao(v):
    if "Alto" in str(v):
        return "font-weight:600"

    return ""


ordem_atencao = {
    "🔴 Alto": 0,
    "🟠 Moderado": 1,
    "🟡 Acompanhar": 2,
    "🟢 Baixo": 3,
}

single_source["_ord"] = single_source["nivel_atencao"].map(ordem_atencao)

single_source = single_source.sort_values(
    ["_ord", "tag_abc", "desvio_mediano"],
    ascending=[True, True, False],
).drop(columns="_ord")

single_source = single_source[
    single_source["n_ops"].notna()
    & (single_source["n_ops"] > 0)
]

print(f"📋 Pares Single-Source com OPs (fornecedor × produto): {len(single_source)}")

(
    single_source[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .map(color_atencao, subset=["nivel_atencao"])
    .format({
        "lt_teorico": "{:.0f}d",
        "lt_realizado": "{:.0f}d",
        "desvio_mediano": "{:+.0f}d",
        "pct_no_prazo": "{:.1f}%",
    }, na_rep="—")
    .hide(axis="index")
)

📋 Pares Single-Source com OPs (fornecedor × produto): 33


product_name,fornecedor,fluxo,tag_abc,lt_teorico,lt_realizado,desvio_mediano,pct_no_prazo,n_ops,nivel_atencao
Undershirt Anti Suor Gola V Masculino,LUTESTIL,PA,A,90d,159d,+69d,0.0%,3.000000,🔴 Alto
Vestido Chemise Sem Mangas FutureForm Feminino,PIXIE,PA,A,90d,131d,+41d,0.0%,2.000000,🔴 Alto
Camisa FutureForm Masculino,MC & MC,PA,A,90d,128d,+38d,33.3%,3.000000,🔴 Alto
Cueca Boxer Comfort Anti Suor Masculino,BAE BRASIL,PA,A,75d,106d,+30d,68.2%,22.000000,🔴 Alto
Calça Director Masculino,Conceitun,PA,B,90d,218d,+128d,0.0%,6.000000,🔴 Alto
Vestido Curto Gola Canoa FutureForm Feminino,KABRIOLLI,PA,B,90d,139d,+49d,0.0%,3.000000,🔴 Alto
Skin Cropped Long Sleeve Feminino,BAE BRASIL,PA,A,75d,90d,+15d,84.6%,13.000000,🟠 Moderado
Spectrum Socks Low 2.0,MALHAS D'STEFANO,PA,B,60d,89d,+29d,81.5%,27.000000,🟠 Moderado
Calça Flare InSkin Feminino,CLARA BELLA,PA,B,90d,115d,+25d,100.0%,1.000000,🟠 Moderado
Vestido Midi de Alça FutureForm Feminino,KABRIOLLI,PA,B,90d,105d,+15d,100.0%,2.000000,🟠 Moderado


## Oportunidades de Realocação

In [40]:
# Célula 10 — Nível 3: Top Spreads — Mesmo Produto + Faixa de Volume
# Parte A: Gráfico de barras horizontais (top 20 por spread bruto)
# Parte B: Tabela companion sem coluna oportunidade_ops_dias

# ====== PREPARAÇÃO DOS DADOS (compartilhada com heatmap abaixo) ======

spread_base = (
    df_ops.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order", "supplier_name"],
        observed=True,
    )
    .agg(
        lt_mediano = ("lead_time_realizado", "median"),
        n_ops      = ("op_code",             "count"),
    )
    .reset_index()
)
spread_base = spread_base[spread_base["n_ops"] >= CONFIG["min_ops_grafico"]]

contagem_forn = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["supplier_name"].nunique().reset_index(name="n_fornecedores")
)
spread_base = spread_base.merge(
    contagem_forn,
    on=["product_names", "volume_bucket", "is_finished_product_order"],
)
spread_base = spread_base[spread_base["n_fornecedores"] >= 2]

spread_min = (
    spread_base.sort_values("lt_mediano")
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_rapido",
        "supplier_name": "fornecedor_rapido",
        "n_ops":         "n_rapido",
    })
)

spread_max = (
    spread_base.sort_values("lt_mediano", ascending=False)
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_lento",
        "supplier_name": "fornecedor_lento",
        "n_ops":         "n_lento",
    })
)

n_total_grupo = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["n_ops"].sum().reset_index(name="n_total_grupo")
)

spread_agg = (
    spread_min
    .merge(spread_max,    on=["product_names", "volume_bucket", "is_finished_product_order"])
    .merge(n_total_grupo, on=["product_names", "volume_bucket", "is_finished_product_order"])
)
spread_agg["spread_dias"] = spread_agg["lt_lento"] - spread_agg["lt_rapido"]
spread_agg["fluxo"]       = spread_agg["is_finished_product_order"].map({True: "PA", False: "Tri"})

# ====== PARTE A: GRÁFICO DE BARRAS (top 20 por spread bruto) ======

spread_top = spread_agg.sort_values("spread_dias", ascending=False).head(20).copy()

spread_top["label"] = (
    spread_top["product_names"].str[:28] + " | "
    + spread_top["volume_bucket"].astype(str) + " | "
    + spread_top["fluxo"]
)

fig = go.Figure(go.Bar(
    x=spread_top["spread_dias"],
    y=spread_top["label"],
    orientation="h",
    marker=dict(
        color=spread_top["spread_dias"],
        colorscale=[
            [0.0, COLORS["status_ok"]],
            [0.5, COLORS["status_atencao"]],
            [1.0, COLORS["status_critico"]],
        ],
        cmin=0, cmax=150,
        showscale=False,
    ),
    text=spread_top["spread_dias"].round(0).astype("Int64").astype(str) + "d",
    textposition="outside",
    customdata=spread_top[[
        "fornecedor_rapido", "lt_rapido",
        "fornecedor_lento",  "lt_lento",
        "n_total_grupo",
    ]].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Rápido: %{customdata[0]} (%{customdata[1]:.0f}d)<br>"
        "Lento:  %{customdata[2]} (%{customdata[3]:.0f}d)<br>"
        "n total no grupo: %{customdata[4]}<br>"
        "Spread: %{x:.0f}d<extra></extra>"
    ),
))
fig.update_layout(
    title="Top 20 Spreads — Oportunidade de Realocação (mesmo produto + faixa de volume)",
    template=TEMPLATE,
    xaxis_title="Spread de Lead Time (dias)",
    height=620,
    yaxis={"categoryorder": "total ascending"},
)
fig.show()

# ====== PARTE B: TABELA COMPANION (sem oportunidade_ops_dias) ======

display_cols_tab = [
    "product_names", "fluxo", "volume_bucket",
    "fornecedor_rapido", "lt_rapido", "n_rapido",
    "fornecedor_lento",  "lt_lento",  "n_lento",
    "spread_dias", "n_total_grupo",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (f"background-color:{COLORS['fluxo_pa']};"
                "color:white; font-weight:600; text-align:center")
    if v == "Tri":
        return (f"background-color:{COLORS['fluxo_tri']};"
                "color:#374151; font-weight:600; text-align:center")
    return ""


(spread_top[display_cols_tab]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .background_gradient(subset=["spread_dias"], cmap="RdYlGn_r", vmin=0, vmax=150)
    .format({
        "lt_rapido":   "{:.0f}d",
        "lt_lento":    "{:.0f}d",
        "spread_dias": "{:.0f}d",
    })
    .hide(axis="index")
)


product_names,fluxo,volume_bucket,fornecedor_rapido,lt_rapido,n_rapido,fornecedor_lento,lt_lento,n_lento,spread_dias,n_total_grupo
The Perfect Top Feminino,PA,1000-1999,RDM,63d,5,BAE BRASIL,146d,7,83d,16
Daily T-shirt Masculino,PA,500-999,DDAL,54d,6,BY COTTON,114d,4,59d,10
Daily Light T-shirt Masculino,PA,2000-4999,LUNELLI,68d,4,Lunelli Nordeste,119d,30,50d,34
Spectrum Socks High 2.0,PA,1000-1999,BAE BRASIL,46d,4,MALHAS D'STEFANO,96d,7,50d,11
Daily T-shirt Masculino,PA,2000-4999,ART LIVRE,111d,20,RIZLLEP,136d,21,25d,48
Tech T-shirt Gola U Masculino,PA,2000-4999,BAE BRASIL,90d,23,BY COTTON,104d,13,14d,36
Wingsuit Feminino,Tri,200-499,GOAT,142d,12,DDAL,152d,12,10d,24
Wingsuit Feminino,Tri,500-999,GOAT,130d,4,DDAL,139d,20,10d,24
Daily T-shirt Masculino,PA,1000-1999,BAE BRASIL,102d,7,ART LIVRE,111d,14,9d,33
The Perfect Top Feminino,PA,2000-4999,RDM,102d,8,BAE BRASIL,105d,17,4d,25


In [41]:
# Célula 10.1 — Heatmap de Spread por Produto × Faixa de Volume (PA e Tri separados)
# Ordenação crescente: produtos com menor spread ficam no topo.
# Células sem amostra suficiente exibem "—" em cinza.


def montar_heatmap_spread(df_spread_agg, label_fluxo):
    sub = df_spread_agg[df_spread_agg["fluxo"] == label_fluxo].copy()
    if len(sub) == 0:
        return None, None

    sub["produto_label"] = sub["product_names"].str[:35]

    pivot_spread = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="spread_dias",
            aggfunc="median",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="n_total_grupo",
            aggfunc="sum",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )

    pivot_spread = pivot_spread.where(pivot_n >= CONFIG["min_ops_grafico"])

    ordem = pivot_spread.max(axis=1).sort_values(ascending=True).index
    return pivot_spread.loc[ordem], pivot_n.loc[ordem]


hm_tri, n_tri_hm = montar_heatmap_spread(spread_agg, "Tri")
hm_pa,  n_pa_hm  = montar_heatmap_spread(spread_agg, "PA")

subplots_data = [
    (hm, n_hm, lbl)
    for hm, n_hm, lbl in [(hm_tri, n_tri_hm, "Triangulação"), (hm_pa, n_pa_hm, "Produto Acabado")]
    if hm is not None and len(hm) > 0
]
if len(subplots_data) == 0:
    print("⚠ Sem dados suficientes para gerar o heatmap de spread com os filtros atuais.")
    print(f"  hm_tri: {hm_tri is not None and len(hm_tri) > 0}")
    print(f"  hm_pa:  {hm_pa is not None and len(hm_pa) > 0}")
else:
    n_cols = len(subplots_data)
    fig_hm = make_subplots(
        rows=1, cols=n_cols,
        subplot_titles=[lbl for _, _, lbl in subplots_data],
        horizontal_spacing=0.18,
    )

    for idx, (hm, n_hm, lbl) in enumerate(subplots_data, start=1):
        texto = hm.copy().astype(object)
        for i in hm.index:
            for j in hm.columns:
                v = hm.loc[i, j]
                texto.loc[i, j] = f"{v:.0f}d" if pd.notna(v) else "—"

        fig_hm.add_trace(
            go.Heatmap(
                z=hm.values,
                x=list(hm.columns),
                y=list(hm.index),
                text=texto.values,
                texttemplate="%{text}",
                textfont=dict(size=10, color="#374151"),
                colorscale=[
                    [0.0,  COLORS["status_ok"]],
                    [0.33, "#FEF9C3"],
                    [0.66, COLORS["status_atencao"]],
                    [1.0,  COLORS["status_critico"]],
                ],
                zmid=50,
                zmin=0,
                zmax=150,
                showscale=(idx == n_cols),
                colorbar=dict(
                    title="Spread (d)",
                    x=1.02,
                    tickvals=[0, 30, 60, 90, 120, 150],
                    ticktext=["0d", "30d", "60d", "90d", "120d", "150d+"],
                ) if idx == n_cols else None,
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Volume: %{x}<br>"
                    "Spread: %{z:.0f}d<extra></extra>"
                ),
            ),
            row=1, col=idx,
        )
        fig_hm.update_xaxes(title_text="Faixa de Volume", row=1, col=idx)

    fig_hm.update_layout(
        title=(
            "Heatmap de Spread de Lead Time por Produto × Faixa de Volume<br>"
            "<sup>Ordenado por spread crescente — verde = menor spread</sup>"
        ),
        template=TEMPLATE,
        height=max(500, 22 * max(len(hm) for hm, _, _ in subplots_data) + 120),
    )
    fig_hm.show()

In [42]:
# Validação pontual da correção de postergação — Opção B
validacao_opf99n11 = df_postponement[df_postponement["op_code"] == "OPF99N11"]
print("Validação OPF99N11 — postergação apenas entre snapshots committed consecutivos")
display(validacao_opf99n11)

Validação OPF99N11 — postergação apenas entre snapshots committed consecutivos


,op_code,dt_planned_original,qt_dias_postergacao_intencional,flag_teve_postergacao
2934,OPF99N11,2025-12-15,7,True


In [43]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{}')
else:
  _deepnote_current_table_attrs = '{}'

df_7 = _dntk.execute_sql(
  'WITH daily_dedup AS (\n  SELECT\n    h.op_code,\n    h.ingestion_date,\n    h.current_production_stage,\n    h.supplier_name,\n    h.product_name,\n    h.cycle_name,\n    h.production_order_type,\n    h.dt_planned_entry_warehouse,\n    h.dt_reviewed_entry_warehouse,\n    h.dt_largest_entry_warehouse,\n    ROW_NUMBER() OVER (\n      PARTITION BY h.op_code, DATE(h.ingestion_date)\n      ORDER BY h.ingestion_date ASC\n    ) AS rn\n  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history` h\n  WHERE h.op_code = \'OPF99N11\'\n),\n\ndeduped AS (\n  SELECT *\n  FROM daily_dedup\n  WHERE rn = 1\n),\n\nwith_lag AS (\n  SELECT\n    *,\n    LAG(dt_planned_entry_warehouse) OVER (\n      PARTITION BY op_code\n      ORDER BY ingestion_date\n    ) AS prev_planned_entry_warehouse,\n    LAG(production_order_type) OVER (\n      PARTITION BY op_code\n      ORDER BY ingestion_date\n    ) AS prev_production_order_type\n  FROM deduped\n)\n\nSELECT\n  op_code,\n  ingestion_date,\n  current_production_stage,\n  cycle_name,\n  production_order_type,\n  prev_production_order_type,\n  prev_planned_entry_warehouse,\n  dt_planned_entry_warehouse,\n  DATE_DIFF(dt_planned_entry_warehouse, prev_planned_entry_warehouse, DAY) AS diff_dias,\n  dt_reviewed_entry_warehouse,\n  dt_largest_entry_warehouse\nFROM with_lag\nWHERE prev_planned_entry_warehouse IS NULL\n   OR dt_planned_entry_warehouse != prev_planned_entry_warehouse\n   OR production_order_type != prev_production_order_type\nORDER BY ingestion_date',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_7

,op_code,ingestion_date,current_production_stage,cycle_name,production_order_type,prev_production_order_type,prev_planned_entry_warehouse,dt_planned_entry_warehouse,diff_dias,dt_reviewed_entry_warehouse,dt_largest_entry_warehouse
0,OPF99N11,2025-08-13,Compras: Aguardando Faturamento de Tecidos,None,None,None,None,2025-12-01,NaN,None,None
1,OPF99N11,2025-10-25,fabric_delivery_and_validation,C122025,None,None,2025-12-01,2025-12-15,14.0,None,None
2,OPF99N11,2025-12-13,waiting_fabric_arrival,C122025,committed,committed,2025-12-15,2026-04-17,123.0,2026-04-17,None
3,OPF99N11,2026-01-14,cut_fabric_and_sewing_process,C122025,committed,committed,2026-04-17,2026-04-24,7.0,2026-04-24,None
4,OPF99N11,2026-02-07,cut_fabric_and_sewing_process,C122025,incubation,committed,2026-04-24,2026-04-24,0.0,2026-04-24,None
5,OPF99N11,2026-04-01,cut_fabric_and_sewing_process,C122025,incubation,incubation,2026-04-24,2026-04-27,3.0,2026-04-27,None


In [44]:
# Celula final - Resumo auditavel de lead time cadastrado
# Mantem os totais no fim do snapshot para revisao rapida apos execucao.

print("SUMMARY_LT_CADASTRO_START")
print(summary_lt_cadastro.to_string(index=False))
print("SUMMARY_LT_CADASTRO_END")
print(f"TOTAL_LINHAS_AUDITADAS={len(df_lead_time_cadastro_audit)}")
print(f"TOTAL_PRODUTOS_AUDITADOS={df_lead_time_cadastro_audit['product_id'].nunique()}")
print(f"TOTAL_CHAVES_LT_CADASTRO_VALIDAS={len(df_lead_time_cadastro)}")
print(f"TOTAL_ISSUES_CRITICAS_ATIVAS={int(df_lead_time_cadastro_audit['is_critical_lead_time_issue'].sum())}")
print(f"TOTAL_LINHAS_LT_CADASTRO_INVALIDAS={len(invalid_lead_time_cadastro_rows)}")
print(f"TOTAL_LINHAS_DF_CAPACITY_PRESERVADO={len(df_capacity)}")
if 'df_lead_time_join_audit' in globals():
    print(f"TOTAL_OPS_AUDITORIA_JOIN_LT={len(df_lead_time_join_audit)}")
    if 'motivo_provavel' in df_lead_time_join_audit.columns and len(df_lead_time_join_audit) > 0:
        print("SUMMARY_JOIN_LT_MOTIVOS_START")
        print(df_lead_time_join_audit['motivo_provavel'].value_counts(dropna=False).to_string())
        print("SUMMARY_JOIN_LT_MOTIVOS_END")

SUMMARY_LT_CADASTRO_START
product_status_classificado lead_time_status  linhas  produtos  fornecedores  criticos
                      ativo               ok     552       146            84         0
                    inativo               ok     175        92            42         0
       status_indeterminado               ok       9         9             3         0
SUMMARY_LT_CADASTRO_END
TOTAL_LINHAS_AUDITADAS=736
TOTAL_PRODUTOS_AUDITADOS=247
TOTAL_CHAVES_LT_CADASTRO_VALIDAS=736
TOTAL_ISSUES_CRITICAS_ATIVAS=0
TOTAL_LINHAS_LT_CADASTRO_INVALIDAS=0
TOTAL_LINHAS_DF_CAPACITY_PRESERVADO=241
TOTAL_OPS_AUDITORIA_JOIN_LT=0


## Governança — OPs com SLA de 90 dias

Cohort de OPs com antecedência planejada inferior a 100 dias. Esta seção usa o histórico completo para incluir OPs pendentes; respeita apenas os filtros dimensionais de fornecedor e produto.

In [45]:
# Governança SLA-90 — base do cohort, regras de negócio e controles de qualidade.
# Esta seção consulta o histórico completo para preservar OPs ainda pendentes.

from pathlib import Path
from IPython.display import FileLink, display

ANTECEDENCIA_THRESHOLD = 100
SLA_ALVO = 90
FLUXO_FILTRO = "Ambos"  # "PA", "Tri" ou "Ambos"
META_PCT_SLA = 0.80

sla90_sql = """
WITH base AS (
  SELECT
    h.op_code,
    ANY_VALUE(COALESCE(h.is_finished_product_order, false)) AS is_finished_product_order,
    STRING_AGG(DISTINCT h.product_name)                     AS product_names,
    ANY_VALUE(h.supplier_name)                              AS supplier_name,
    ANY_VALUE(h.cycle_name)                                 AS cycle_name,
    ANY_VALUE(h.production_order_type)                      AS production_order_type,
    ANY_VALUE(h.supplier_relationship_status)               AS supplier_relationship_status,
    STRING_AGG(DISTINCT h.status_sku)                       AS sku_status,
    ANY_VALUE(h.current_production_stage)                   AS current_production_stage,
    MAX(h.planned_quantity_op)                              AS planned_quantity_op,
    MAX(h.received_quantity_op)                             AS received_quantity_op,
    MIN(h.dt_planned_entry_warehouse)                       AS dt_first_planned_entry,
    MAX(h.dt_largest_entry_warehouse)                       AS dt_largest_entry_warehouse,
    MIN(CAST(h.ingestion_date AS TIMESTAMP))                AS stamp_created_production_order,
    MIN(CASE WHEN h.current_production_stage = 'waiting_fabric_arrival'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)       AS stamp_stage_waiting_fabric_arrival
  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history` h
  GROUP BY h.op_code
)
SELECT *
FROM base
WHERE production_order_type = 'committed'
  AND (supplier_relationship_status IS NULL
       OR supplier_relationship_status NOT IN ('terminated', 'discontinued'))
  AND NOT REGEXP_CONTAINS(LOWER(COALESCE(cycle_name, '')), r'b2b|epa')
  AND NOT REGEXP_CONTAINS(LOWER(COALESCE(sku_status, '')), r'desativado')
"""

try:
    _dntk
except NameError:
    from deepnote import _dntk

df_sla90_base = _dntk.execute_sql(
    sla90_sql,
    "SQL_5088268A_9640_458A_83D1_46B07B955FAE",
).copy()

if "df_postponement" not in globals():
    raise NameError("df_postponement não encontrado. Execute as células anteriores antes da seção SLA-90.")
if not {"op_code", "dt_planned_original"}.issubset(df_postponement.columns):
    raise ValueError("df_postponement precisa conter op_code e dt_planned_original.")

# O cohort respeita fornecedor/produto, mas deliberadamente não herda o filtro de datas.
df_sla90_dim = df_sla90_base.copy()
supplier_filtro_sla90 = globals().get("supplier_filtro", "(Todos)")
product_filtro_sla90 = globals().get("product_filtro", "(Todos)")

if supplier_filtro_sla90 != "(Todos)":
    df_sla90_dim = df_sla90_dim[df_sla90_dim["supplier_name"] == supplier_filtro_sla90].copy()
if product_filtro_sla90 != "(Todos)":
    df_sla90_dim = df_sla90_dim[
        df_sla90_dim["product_names"].fillna("").str.contains(
            str(product_filtro_sla90), case=False, regex=False
        )
    ].copy()

if FLUXO_FILTRO == "PA":
    df_sla90_dim = df_sla90_dim[df_sla90_dim["is_finished_product_order"]].copy()
elif FLUXO_FILTRO == "Tri":
    df_sla90_dim = df_sla90_dim[~df_sla90_dim["is_finished_product_order"]].copy()
elif FLUXO_FILTRO != "Ambos":
    raise ValueError("FLUXO_FILTRO deve ser 'PA', 'Tri' ou 'Ambos'.")

total_ops_committed_elegiveis = df_sla90_dim["op_code"].nunique()

postponement_ref = (
    df_postponement[["op_code", "dt_planned_original"]]
    .drop_duplicates(subset="op_code", keep="first")
    .copy()
)
df = df_sla90_dim.merge(postponement_ref, on="op_code", how="left", validate="one_to_one")

datetime_cols = [
    "stamp_created_production_order",
    "stamp_stage_waiting_fabric_arrival",
    "dt_largest_entry_warehouse",
    "dt_first_planned_entry",
    "dt_planned_original",
]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df["dt_planned_ref"] = df["dt_planned_original"].fillna(df["dt_first_planned_entry"])
df["antecedencia_planejada"] = (
    df["dt_planned_ref"] - df["stamp_created_production_order"]
).dt.days

df = df[
    df["antecedencia_planejada"].gt(0)
    & df["antecedencia_planejada"].lt(ANTECEDENCIA_THRESHOLD)
].copy()

df["alocado_pecas"] = pd.to_numeric(df["planned_quantity_op"], errors="coerce")
df["realizado_pecas"] = pd.to_numeric(df["received_quantity_op"], errors="coerce")
hoje_sla90 = pd.Timestamp.now(tz="UTC").normalize()
df["entregue"] = (
    df["dt_largest_entry_warehouse"].notna()
    & df["dt_largest_entry_warehouse"].le(hoje_sla90)
)

inicio_lt_sla90 = df["stamp_stage_waiting_fabric_arrival"].where(
    df["is_finished_product_order"],
    other=df["stamp_created_production_order"],
)
df["lead_time_realizado"] = np.where(
    df["entregue"],
    (df["dt_largest_entry_warehouse"] - inicio_lt_sla90).dt.days,
    np.nan,
)
df["entregue_no_sla"] = df["entregue"] & df["lead_time_realizado"].le(SLA_ALVO)

df["status_sla"] = np.select(
    [
        ~df["entregue"],
        df["entregue"] & df["lead_time_realizado"].le(SLA_ALVO),
        df["entregue"] & df["lead_time_realizado"].gt(SLA_ALVO),
    ],
    ["Pendente", "Entregue no SLA", "Entregue fora do SLA"],
    default="Pendente",
)

cycle_parts = df["cycle_name"].fillna("").str.extract(r"^C(\d{2})(\d{4})$")
df["cycle_order"] = (
    pd.to_numeric(cycle_parts[1], errors="coerce") * 100
    + pd.to_numeric(cycle_parts[0], errors="coerce")
).fillna(np.inf)

df_governanca_sla90 = df.sort_values(["cycle_order", "cycle_name", "op_code"]).reset_index(drop=True)

if not df_governanca_sla90["op_code"].is_unique:
    raise AssertionError("A seção SLA-90 gerou OPs duplicadas após o merge de postergação.")

def _sum_min(series):
    return series.sum(min_count=1)

def _safe_pct(numerator, denominator):
    return (numerator / denominator * 100) if pd.notna(denominator) and denominator > 0 else np.nan

governanca_por_ciclo = (
    df_governanca_sla90
    .groupby(["cycle_name", "cycle_order"], dropna=False, as_index=False)
    .agg(
        n_ops_alocadas=("op_code", "nunique"),
        pecas_alocadas=("alocado_pecas", _sum_min),
        pecas_realizadas=("realizado_pecas", _sum_min),
        antecedencia_mediana=("antecedencia_planejada", "median"),
        n_entregues=("entregue", "sum"),
        n_entregues_no_sla=("entregue_no_sla", "sum"),
        n_pendentes=("entregue", lambda x: (~x).sum()),
    )
    .sort_values(["cycle_order", "cycle_name"])
    .reset_index(drop=True)
)
governanca_por_ciclo["n_entregues_fora_sla"] = (
    governanca_por_ciclo["n_entregues"] - governanca_por_ciclo["n_entregues_no_sla"]
)
governanca_por_ciclo["pct_atendimento"] = [
    _safe_pct(realizado, alocado)
    for realizado, alocado in zip(
        governanca_por_ciclo["pecas_realizadas"], governanca_por_ciclo["pecas_alocadas"]
    )
]
governanca_por_ciclo["pct_no_sla"] = [
    _safe_pct(no_sla, entregues)
    for no_sla, entregues in zip(
        governanca_por_ciclo["n_entregues_no_sla"], governanca_por_ciclo["n_entregues"]
    )
]

carteira_quality_summary = pd.DataFrame(
    {
        "metrica": [
            "OPs committed elegíveis antes do cohort",
            "OPs no cohort SLA-90",
            "OPs únicas no cohort",
            "Cobertura dt_planned_ref",
            "Nulos em peças alocadas",
            "Nulos em peças realizadas",
        ],
        "valor": [
            total_ops_committed_elegiveis,
            len(df_governanca_sla90),
            df_governanca_sla90["op_code"].nunique(),
            df_governanca_sla90["dt_planned_ref"].notna().mean(),
            df_governanca_sla90["alocado_pecas"].isna().sum(),
            df_governanca_sla90["realizado_pecas"].isna().sum(),
        ],
    }
)

print("Governança SLA-90 preparada")
print(f"  OPs elegíveis antes do cohort: {total_ops_committed_elegiveis:,}")
print(f"  OPs no cohort: {len(df_governanca_sla90):,}")
display(carteira_quality_summary)

Governança SLA-90 preparada
  OPs elegíveis antes do cohort: 7,421
  OPs no cohort: 1,265


,metrica,valor
0,OPs committed elegíveis antes do cohort,7421.0
1,OPs no cohort SLA-90,1265.0
2,OPs únicas no cohort,1265.0
3,Cobertura dt_planned_ref,1.0
4,Nulos em peças alocadas,0.0
5,Nulos em peças realizadas,490.0


### Bloco 0 — KPIs do cohort SLA-90

In [46]:
# KPIs executivos: tamanho do cohort, cobertura de peças, atendimento e antecedência.
if df_governanca_sla90.empty:
    print("Sem OPs no cohort SLA-90 para os filtros dimensionais selecionados.")
else:
    total_alocado = _sum_min(df_governanca_sla90["alocado_pecas"])
    total_realizado = _sum_min(df_governanca_sla90["realizado_pecas"])
    pct_cohort = _safe_pct(len(df_governanca_sla90), total_ops_committed_elegiveis)
    pct_atendimento_global = _safe_pct(total_realizado, total_alocado)
    antecedencia_mediana_global = df_governanca_sla90["antecedencia_planejada"].median()

    kpi_specs = [
        ("OPs no cohort", len(df_governanca_sla90), ".0f", ""),
        ("% do committed elegível", pct_cohort, ".1f", "%"),
        ("Peças alocadas", total_alocado, ",.0f", ""),
        ("Peças realizadas", total_realizado, ",.0f", ""),
        ("% atendimento global", pct_atendimento_global, ".1f", "%"),
        ("Antecedência mediana", antecedencia_mediana_global, ".0f", " dias"),
    ]

    fig = go.Figure()
    for index, (title, value, number_format, suffix) in enumerate(kpi_specs):
        fig.add_trace(
            go.Indicator(
                mode="number",
                value=value if pd.notna(value) else 0,
                number={"valueformat": number_format, "suffix": suffix},
                title={"text": title},
                domain={"row": index // 3, "column": index % 3},
            )
        )
    fig.update_layout(
        title="KPIs — Governança de OPs com SLA de 90 dias",
        template=TEMPLATE,
        grid={"rows": 2, "columns": 3, "pattern": "independent"},
        height=310,
        margin={"l": 30, "r": 30, "t": 70, "b": 20},
    )
    fig.show()

### Bloco 2 — Evolução de peças alocadas versus realizadas por ciclo

In [47]:
# Evolução principal: volumes de peças e taxa de atendimento por ciclo.
if governanca_por_ciclo.empty:
    print("Sem ciclos no cohort SLA-90.")
else:
    ciclo = governanca_por_ciclo.copy()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for column, label, color in [
        ("pecas_alocadas", "Peças alocadas", COLORS["fluxo_pa"]),
        ("pecas_realizadas", "Peças realizadas", COLORS["status_ok"]),
    ]:
        values = ciclo[column]
        fig.add_trace(
            go.Bar(
                x=ciclo["cycle_name"],
                y=values,
                name=label,
                marker_color=color,
                text=[f"{value:,.0f}" if pd.notna(value) else "—" for value in values],
                textposition="outside",
            ),
            secondary_y=False,
        )
    fig.add_trace(
        go.Scatter(
            x=ciclo["cycle_name"],
            y=ciclo["pct_atendimento"],
            name="% atendimento",
            mode="lines+markers+text",
            line={"color": COLORS["status_atencao"], "width": 3},
            text=[f"{value:.1f}%" if pd.notna(value) else "—" for value in ciclo["pct_atendimento"]],
            textposition="top center",
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title="Alocado versus realizado por ciclo — Cohort SLA-90",
        template=TEMPLATE,
        barmode="group",
        height=500,
    )
    fig.update_yaxes(title_text="Peças", secondary_y=False)
    fig.update_yaxes(title_text="% atendimento", ticksuffix="%", rangemode="tozero", secondary_y=True)
    fig.show()

### Bloco 2 — Status das OPs por ciclo

In [48]:
# Status por ciclo: Pendente significa OP ainda não entregue, não OP entregue fora do SLA.
status_order = ["Entregue no SLA", "Entregue fora do SLA", "Pendente"]
status_colors = {
    "Entregue no SLA": COLORS["status_ok"],
    "Entregue fora do SLA": COLORS["status_atencao"],
    "Pendente": COLORS["neutro_claro"],
}

if df_governanca_sla90.empty:
    print("Sem OPs no cohort SLA-90.")
else:
    status_ciclo = (
        df_governanca_sla90
        .groupby(["cycle_name", "cycle_order", "status_sla"], dropna=False)["op_code"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(columns=status_order, fill_value=0)
        .reset_index()
        .sort_values(["cycle_order", "cycle_name"])
    )
    fig = go.Figure()
    for status in status_order:
        fig.add_trace(
            go.Bar(
                x=status_ciclo["cycle_name"],
                y=status_ciclo[status],
                name=status,
                marker_color=status_colors[status],
                text=status_ciclo[status],
                textposition="inside",
            )
        )
    fig.update_layout(
        title="Status das OPs por ciclo — Pendente ≠ Entregue fora do SLA",
        template=TEMPLATE,
        barmode="stack",
        yaxis_title="Número de OPs",
        height=500,
        legend_title="Status SLA",
    )
    fig.show()

### Bloco 3 — Taxa de cumprimento do SLA-90

In [49]:
# Taxa por ciclo: denominador inclui somente OPs já entregues no respectivo ciclo.
if governanca_por_ciclo.empty:
    print("Sem ciclos no cohort SLA-90.")
else:
    sla_ciclo = governanca_por_ciclo[governanca_por_ciclo["n_entregues"] > 0].copy()
    if sla_ciclo.empty:
        print("Ainda não há OPs entregues no cohort para calcular o cumprimento do SLA-90.")
    else:
        fig = go.Figure(
            go.Scatter(
                x=sla_ciclo["cycle_name"],
                y=sla_ciclo["pct_no_sla"],
                mode="lines+markers+text",
                name="% entregue no SLA",
                line={"color": COLORS["status_ok"], "width": 3},
                marker={"size": 9},
                text=[f"{value:.1f}%" for value in sla_ciclo["pct_no_sla"]],
                textposition="top center",
                hovertemplate="<b>%{x}</b><br>% no SLA: %{y:.1f}%<extra></extra>",
            )
        )
        fig.add_hline(
            y=META_PCT_SLA * 100,
            line_dash="dash",
            line_color=COLORS["target_line"],
            annotation_text=f"Meta: {META_PCT_SLA:.0%}",
            annotation_position="bottom right",
        )
        fig.update_layout(
            title="Taxa de cumprimento do SLA-90 por ciclo (somente OPs entregues)",
            template=TEMPLATE,
            yaxis={"title": "% de OPs entregues no SLA", "ticksuffix": "%", "range": [0, 100]},
            height=460,
        )
        fig.show()

### Bloco 6 — Tabela de governança por ciclo e exportação

In [52]:
# Tabela auditável por ciclo e exportação do resumo em CSV.
export_dir_sla90 = Path("exports")
export_dir_sla90.mkdir(exist_ok=True)

governanca_export = governanca_por_ciclo[
    [
        "cycle_name",
        "n_ops_alocadas",
        "pecas_alocadas",
        "pecas_realizadas",
        "pct_atendimento",
        "antecedencia_mediana",
        "n_entregues_no_sla",
        "n_entregues_fora_sla",
        "n_pendentes",
        "pct_no_sla",
    ]
].rename(columns={"cycle_name": "ciclo"}).copy()

governanca_export[["pct_atendimento", "pct_no_sla"]] = (
    governanca_export[["pct_atendimento", "pct_no_sla"]].round(1)
)
governanca_export["antecedencia_mediana"] = governanca_export["antecedencia_mediana"].round(1)

governanca_csv_path = export_dir_sla90 / "governanca_sla90_por_ciclo.csv"
governanca_export.to_csv(governanca_csv_path, index=False, encoding="utf-8-sig")

display(governanca_export)
display(FileLink(str(governanca_csv_path), result_html_prefix="Baixar governança por ciclo: "))

,ciclo,n_ops_alocadas,pecas_alocadas,pecas_realizadas,pct_atendimento,antecedencia_mediana,n_entregues_no_sla,n_entregues_fora_sla,n_pendentes,pct_no_sla
0,C092025,147,312972.0,253106.0,80.9,26.0,42,98,7,30.0
1,C102025,150,354043.0,282797.0,79.9,56.0,34,103,13,24.8
2,C112025,179,348334.0,281936.0,80.9,84.0,50,85,44,37.0
3,C122025,9,9053.0,7463.0,82.4,90.0,0,6,3,0.0
4,C012026,90,132105.0,NaN,NaN,63.0,0,0,90,NaN
...,...,...,...,...,...,...,...,...,...,...
56,RIBANA19-02,1,685.0,667.0,97.4,28.0,1,0,0,100.0
57,TESTE0509,2,1599.0,NaN,NaN,27.0,0,0,2,NaN
58,THE PERFECT TOP - VARIAÇÕES,28,32083.0,27518.0,85.8,49.0,5,21,2,19.2
59,UW FEMININA - NOVO DROP,9,15834.0,NaN,NaN,34.0,0,0,9,NaN


/datasets/_deepnote_work/exports/governanca_sla90_por_ciclo.csv

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=62b1671d-dc82-4747-9bbb-b134a69a3491' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>